# Multiview clustering

In this notebook we use a combination of techniques in order to cluster our data in a multiview approach. 

In [ ]:
# Resolve shared multiclust imports when this notebook is run from notebooks/<project>/.
from pathlib import Path
import os
import sys

_candidates = []
if os.environ.get("MULTICLUST_ROOT"):
    _candidates.append(Path(os.environ["MULTICLUST_ROOT"]).expanduser())
_candidates.extend([Path.cwd(), *Path.cwd().parents])
for _candidate in _candidates:
    if (_candidate / "Utils.py").exists() and (_candidate / "full_pipeline.py").exists():
        MULTICLUST_ROOT = _candidate.resolve()
        break
else:
    raise RuntimeError("Could not find multiclust root. Set MULTICLUST_ROOT to the Code/multiclust folder.")
if str(MULTICLUST_ROOT) not in sys.path:
    sys.path.insert(0, str(MULTICLUST_ROOT))
print(f"Using multiclust root: {MULTICLUST_ROOT}")


In [ ]:
# Import theme for plots
import theme
theme.apply_all()


# Data Preparation

### Load all requirements

In [ ]:
from Utils import *
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import pickle
import dill
from SVM import *

#Import own functions
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, leaves_list
import scipy.cluster.hierarchy as hierarchy

import seaborn as sns
from scipy.stats import f_oneway, kruskal, shapiro, levene, chi2_contingency, fisher_exact

from parea_classes import KMeansPyrea, ModelBasedClusteringPyrea, HierarchicalClusteringPyrea, EnsembleClusteringPyrea




In [ ]:
# If need to reload after change
#importlib.reload(Utils)

### Load in data and metatable

In [ ]:
data = pd.read_csv("path/to/multiclust_data/merged_data.csv")
meta = pd.read_csv("path/to/multiclust_data/merged_meta.csv")



Change colums to character type when categorical.

In [ ]:
# Load the dictionary from an Excel file
dictionary_DIR = "path/to/project/Feature selection/Complete_dictionary.xlsx"
dict_df = pd.read_excel(dictionary_DIR)
ignore_keys = {-900, -300}

# Infer variables with explicit value-label mappings in the dictionary.
# Some continuous variables have numeric notes too, so this list is only a candidate list.
def _extract_dictionary_codes(note):
    if pd.isna(note) or note == "":
        return []
    codes = []
    for raw in re.findall(r"(-?\d+(?:\.\d+)?)\s*=\s*[^,;]+", str(note)):
        try:
            code = float(raw)
        except ValueError:
            continue
        if code not in ignore_keys:
            codes.append(code)
    return sorted(set(codes))

dict_df["dictionary_codes"] = dict_df["Notes"].apply(_extract_dictionary_codes)
dict_df["InferredDataType"] = pd.Series(pd.NA, index=dict_df.index, dtype="object")
dict_df.loc[dict_df["dictionary_codes"].apply(len) >= 2, "InferredDataType"] = "Categorical"

cat_vars = dict_df.loc[dict_df["InferredDataType"] == "Categorical", "ElementName"].tolist()

# Only coerce truly discrete categorical variables to object/string.
# High-cardinality numeric variables, such as continuous cognition scores, must stay numeric;
# otherwise preprocessing will one-hot encode every observed value and create columns like cnb_*_nan.
def _should_convert_to_categorical(series, max_unique_numeric=20):
    non_missing = series.dropna()
    if non_missing.empty:
        return False
    if not pd.api.types.is_numeric_dtype(non_missing):
        return True

    numeric = pd.to_numeric(non_missing, errors="coerce")
    if numeric.isna().any():
        return False
    n_unique = numeric.nunique(dropna=True)
    integer_like = np.all(np.isclose(numeric, np.round(numeric)))
    return n_unique <= max_unique_numeric and integer_like

cols_to_convert = [
    col for col in cat_vars
    if col in data.columns and _should_convert_to_categorical(data[col])
]
cols_left_numeric = [col for col in cat_vars if col in data.columns and col not in cols_to_convert]

# Use object dtype so real NaN values remain missing rather than becoming the literal string "nan".
data[cols_to_convert] = data[cols_to_convert].astype("object")

print(f"Converted {len(cols_to_convert)} dictionary-categorical columns to object for encoding.")
print(f"Kept {len(cols_left_numeric)} dictionary-flagged columns numeric because they look continuous/high-cardinality.")
print("Examples kept numeric:", cols_left_numeric[:20])


### Remove community controls

In [ ]:

# Remove all rows where the 'phenotype' column has the value 'HC'
data_CHR = data[data['phenotype'] != 'HC'].reset_index(drop=True)
data_CC = data[data['phenotype'] == 'HC'].reset_index(drop=True)

data_all = data.copy()


### Split into discovery and test

In [ ]:
# 1. Load the Prescient ID list once:
prescient = pd.read_csv(
    "path/to/"
    "project/"
    "Data/Prescient_Client List_ DPACC ID with RA Name (ALL sites)_20_03_2025.txt",
    sep="\t"
)
prescient.columns = ["fkLocationID", "pkclientid", "src_subject_id", "RAname"]
prescient_ids = set(prescient["src_subject_id"])

# 2. Suppose your dict of modalities is called `data_dict`:
#    e.g. data_dict = {'fmri': fmri_df, 'eeg': eeg_df, ...}

# 3. Split it:
discovery_data, test_data = split_by_network(data_CHR, prescient_ids)
discovery_data_CC, test_data_CC = split_by_network(data_CC, prescient_ids)
discovery_data_all, test_data_all = split_by_network(data_all, prescient_ids)

# Now `discovery_dict['fmri']` has only Prescient rows for fMRI, 
# and `test_dict['fmri']` only the Pronet rows, etc.


In [ ]:
test_data_all

### Data cleaning

Handle missing values. 
First remove columns with more than 50% missing and then removing rows that have more than 50% missing. #TODO! Check later how much % you actually want to use. 
Right now we're using a simple model to impute the data Important to look at this and choose an appropriate way to deal with missing data. TODO!

In [ ]:
cleaned_discovery, cleaned_test = remove_high_missing_data_split(
    discovery_data, test_data,
    subject_id_column='src_subject_id',
    col_threshold=0.5,
    row_threshold=0.5
)

display(cleaned_discovery)

cleaned_discovery_CC, cleaned_test_CC = remove_high_missing_data_split(
    discovery_data_CC, test_data_CC,
    subject_id_column='src_subject_id',
    col_threshold=0.5,
    row_threshold=0.5
)

cleaned_discovery_all, cleaned_test_all = remove_high_missing_data_split(
    discovery_data_all, test_data_all,
    subject_id_column='src_subject_id',
    col_threshold=0.5,
    row_threshold=0.5
)


In [ ]:
# Only keep relevant modalities
modalities_keep = ["Psychoticism", "Detachment", "Functioning", "Internalising", "Cognition"]

vars_to_keep = meta["ElementName"][meta["Modality"].isin(modalities_keep)].tolist()
# Filter the data if the columns are present in cleaned_discovery. Also include src_subject_id
vars_to_keep.append("src_subject_id")
vars_to_keep.append("phenotype")
cleaned_discovery = cleaned_discovery[cleaned_discovery.columns.intersection(vars_to_keep)]

cleaned_discovery_CC = cleaned_discovery_CC[cleaned_discovery_CC.columns.intersection(vars_to_keep)]
cleaned_discovery_all = cleaned_discovery_all[cleaned_discovery_all.columns.intersection(vars_to_keep)]


In [ ]:
cleaned_discovery_CC

### Collinearity 


This diagnostic first applies the same pipeline-style preprocessing used for clustering: dummy/ordinal coding for categorical variables, imputation, and scaling within each modality. It then checks collinearity on the resulting numeric modality matrices, so categorical predictors are included through their encoded features. It does not remove variables from the analysis.


#### check within clinical domains

In [ ]:
def _pairwise_correlation_level(abs_r):
    if pd.isna(abs_r):
        return np.nan
    if abs_r >= 0.90:
        return "severe"
    if abs_r >= 0.70:
        return "moderate_high"
    if abs_r >= 0.50:
        return "moderate"
    return "low"


def _vif_level(vif):
    if pd.isna(vif):
        return np.nan
    if np.isinf(vif) or vif >= 10:
        return "critical"
    if vif >= 5:
        return "high"
    if vif >= 2.5:
        return "moderate"
    return "low"


def _condition_index_level(condition_index):
    if pd.isna(condition_index):
        return np.nan
    if np.isinf(condition_index) or condition_index >= 30:
        return "critical"
    if condition_index >= 10:
        return "moderate_strong"
    return "low"


def _prepare_numeric_domain_matrix(data, variables, min_pairwise_n=3):
    X = data[variables].apply(pd.to_numeric, errors="coerce")
    valid_vars = [
        col for col in X.columns
        if X[col].notna().sum() >= min_pairwise_n and X[col].nunique(dropna=True) > 1
    ]
    X = X[valid_vars]
    if X.empty:
        return X

    X = X.dropna(axis=0, how="all")
    X = X.fillna(X.median(numeric_only=True))
    valid_vars = [col for col in X.columns if X[col].nunique(dropna=True) > 1]
    return X[valid_vars]


def _compute_vif_table(X, domain_name):
    rows = []
    if X.shape[1] < 2:
        return rows

    X_values = X.to_numpy(dtype=float)
    X_values = (X_values - np.nanmean(X_values, axis=0)) / np.nanstd(X_values, axis=0, ddof=0)
    X_values = np.nan_to_num(X_values, nan=0.0, posinf=0.0, neginf=0.0)

    for idx, variable in enumerate(X.columns):
        y = X_values[:, idx]
        others = np.delete(X_values, idx, axis=1)
        design = np.column_stack([np.ones(others.shape[0]), others])
        try:
            coef, *_ = np.linalg.lstsq(design, y, rcond=None)
            y_hat = design @ coef
            ss_res = float(np.sum((y - y_hat) ** 2))
            ss_tot = float(np.sum((y - y.mean()) ** 2))
            r_squared = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
            r_squared = min(max(r_squared, 0.0), 1.0) if np.isfinite(r_squared) else np.nan
            tolerance = 1.0 - r_squared if np.isfinite(r_squared) else np.nan
            vif = 1.0 / tolerance if np.isfinite(tolerance) and tolerance > 0 else np.inf
        except np.linalg.LinAlgError:
            r_squared = np.nan
            tolerance = 0.0
            vif = np.inf

        rows.append({
            "domain": domain_name,
            "variable": variable,
            "vif": vif,
            "tolerance": tolerance,
            "r_squared_with_other_variables": r_squared,
            "vif_level": _vif_level(vif),
        })
    return rows


def _compute_condition_index(X):
    if X.shape[1] < 2:
        return np.nan, np.nan, np.nan

    X_values = X.to_numpy(dtype=float)
    X_values = (X_values - np.nanmean(X_values, axis=0)) / np.nanstd(X_values, axis=0, ddof=0)
    X_values = np.nan_to_num(X_values, nan=0.0, posinf=0.0, neginf=0.0)
    corr = np.corrcoef(X_values, rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    corr = (corr + corr.T) / 2.0
    np.fill_diagonal(corr, 1.0)

    eigenvalues = np.linalg.eigvalsh(corr)
    eigenvalues = np.clip(eigenvalues, 0.0, None)
    max_eigenvalue = float(np.max(eigenvalues)) if eigenvalues.size else np.nan
    positive_eigenvalues = eigenvalues[eigenvalues > 1e-12]
    min_positive_eigenvalue = float(np.min(positive_eigenvalues)) if positive_eigenvalues.size else np.nan

    if not np.isfinite(max_eigenvalue) or not np.isfinite(min_positive_eigenvalue):
        condition_index = np.nan
    elif min_positive_eigenvalue <= 0:
        condition_index = np.inf
    else:
        condition_index = float(np.sqrt(max_eigenvalue / min_positive_eigenvalue))
    return condition_index, max_eigenvalue, min_positive_eigenvalue


def assess_domain_collinearity(
    data,
    meta=None,
    preprocessed_modalities=None,
    domain_col="Modality",
    variable_col="ElementName",
    id_cols=("src_subject_id", "phenotype"),
    correlation_methods=("pearson", "spearman"),
    high_corr_threshold=0.90,
    moderate_corr_threshold=0.70,
    min_pairwise_n=3,
):
    """Summarise within-domain collinearity for raw data or preprocessed modality matrices."""
    id_cols = set(id_cols)
    if preprocessed_modalities is not None:
        domain_items = [
            (domain_name, [c for c in df_mod.columns if c not in id_cols])
            for domain_name, df_mod in preprocessed_modalities.items()
        ]
    else:
        if meta is None:
            raise ValueError("meta is required when preprocessed_modalities is not provided.")
        required_cols = {domain_col, variable_col}
        missing_meta_cols = required_cols.difference(meta.columns)
        if missing_meta_cols:
            raise KeyError(f"Missing metadata column(s): {sorted(missing_meta_cols)}")
        domain_map = (
            meta.loc[meta[variable_col].isin(data.columns), [variable_col, domain_col]]
            .dropna(subset=[domain_col])
            .drop_duplicates()
            .rename(columns={variable_col: "variable", domain_col: "domain"})
        )
        domain_items = [
            (domain_name, [v for v in domain_df["variable"].tolist() if v not in id_cols])
            for domain_name, domain_df in domain_map.groupby("domain", sort=True)
        ]
    pair_rows = []
    vif_rows = []
    condition_rows = []
    summary_rows = []
    skipped_rows = []

    for domain_name, variables in domain_items:
        source_df = preprocessed_modalities[domain_name] if preprocessed_modalities is not None else data
        X = _prepare_numeric_domain_matrix(source_df, variables, min_pairwise_n=min_pairwise_n)
        numeric_vars = X.columns.tolist()

        for variable in sorted(set(variables).difference(numeric_vars)):
            skipped_rows.append({
                "domain": domain_name,
                "variable": variable,
                "reason": "not numeric/coercible or insufficient variation",
            })

        corr_counts = {method: 0 for method in correlation_methods}
        corr_max = {method: np.nan for method in correlation_methods}
        corr_mean = {method: np.nan for method in correlation_methods}

        if len(numeric_vars) >= 2:
            for method in correlation_methods:
                corr = source_df[numeric_vars].apply(pd.to_numeric, errors="coerce").corr(
                    method=method,
                    min_periods=min_pairwise_n,
                )
                abs_corr = corr.abs()
                upper_mask = np.triu(np.ones(abs_corr.shape, dtype=bool), k=1)
                upper_values = abs_corr.where(upper_mask).stack().dropna()
                flagged_pairs = upper_values[upper_values >= moderate_corr_threshold].sort_values(ascending=False)

                corr_counts[method] = int((upper_values >= high_corr_threshold).sum())
                corr_max[method] = upper_values.max() if len(upper_values) else np.nan
                corr_mean[method] = upper_values.mean() if len(upper_values) else np.nan

                for (variable_1, variable_2), abs_value in flagged_pairs.items():
                    pair_rows.append({
                        "domain": domain_name,
                        "method": method,
                        "variable_1": variable_1,
                        "variable_2": variable_2,
                        "correlation": corr.loc[variable_1, variable_2],
                        "abs_correlation": abs_value,
                        "correlation_level": _pairwise_correlation_level(abs_value),
                    })

            vif_rows.extend(_compute_vif_table(X, domain_name))
            condition_index, max_eigenvalue, min_eigenvalue = _compute_condition_index(X)
        else:
            condition_index, max_eigenvalue, min_eigenvalue = np.nan, np.nan, np.nan

        condition_rows.append({
            "domain": domain_name,
            "condition_index": condition_index,
            "condition_index_level": _condition_index_level(condition_index),
            "max_eigenvalue": max_eigenvalue,
            "min_positive_eigenvalue": min_eigenvalue,
        })

        domain_vifs = [row["vif"] for row in vif_rows if row["domain"] == domain_name and np.isfinite(row["vif"])]
        domain_tolerances = [row["tolerance"] for row in vif_rows if row["domain"] == domain_name and np.isfinite(row["tolerance"])]
        summary_rows.append({
            "domain": domain_name,
            "n_variables_total": len(variables),
            "n_variables_numeric": len(numeric_vars),
            "n_severe_pearson_pairs_abs_r_ge_0_90": corr_counts.get("pearson", 0),
            "n_severe_spearman_pairs_abs_r_ge_0_90": corr_counts.get("spearman", 0),
            "max_abs_pearson": corr_max.get("pearson", np.nan),
            "max_abs_spearman": corr_max.get("spearman", np.nan),
            "mean_abs_pearson": corr_mean.get("pearson", np.nan),
            "mean_abs_spearman": corr_mean.get("spearman", np.nan),
            "max_vif": max(domain_vifs) if domain_vifs else np.nan,
            "min_tolerance": min(domain_tolerances) if domain_tolerances else np.nan,
            "n_variables_vif_ge_2_5": sum(v >= 2.5 for v in domain_vifs),
            "n_variables_vif_ge_5": sum(v >= 5 for v in domain_vifs),
            "n_variables_vif_ge_10": sum(v >= 10 for v in domain_vifs),
            "condition_index": condition_index,
            "condition_index_level": _condition_index_level(condition_index),
        })

    summary = (
        pd.DataFrame(summary_rows)
        .sort_values(
            ["n_severe_spearman_pairs_abs_r_ge_0_90", "max_vif", "condition_index", "domain"],
            ascending=[False, False, False, True],
        )
        .reset_index(drop=True)
    )
    correlation_pairs = (
        pd.DataFrame(pair_rows)
        .sort_values(["abs_correlation", "domain", "method", "variable_1", "variable_2"], ascending=[False, True, True, True, True])
        .reset_index(drop=True)
        if pair_rows else
        pd.DataFrame(columns=["domain", "method", "variable_1", "variable_2", "correlation", "abs_correlation", "correlation_level"])
    )
    vif_table = (
        pd.DataFrame(vif_rows)
        .sort_values(["vif", "domain", "variable"], ascending=[False, True, True])
        .reset_index(drop=True)
        if vif_rows else
        pd.DataFrame(columns=["domain", "variable", "vif", "tolerance", "r_squared_with_other_variables", "vif_level"])
    )
    condition_index_table = (
        pd.DataFrame(condition_rows)
        .sort_values(["condition_index", "domain"], ascending=[False, True])
        .reset_index(drop=True)
        if condition_rows else
        pd.DataFrame(columns=["domain", "condition_index", "condition_index_level", "max_eigenvalue", "min_positive_eigenvalue"])
    )
    skipped_variables = (
        pd.DataFrame(skipped_rows)
        .sort_values(["domain", "variable"])
        .reset_index(drop=True)
        if skipped_rows else
        pd.DataFrame(columns=["domain", "variable", "reason"])
    )

    return {
        "summary": summary,
        "correlation_pairs": correlation_pairs,
        "vif": vif_table,
        "condition_index": condition_index_table,
        "skipped_variables": skipped_variables,
    }


_, collinearity_subject_ids, collinearity_processed_modalities = preprocessing(
    cleaned_discovery,
    meta,
    subject_id_column="src_subject_id",
    col_threshold=0.5,
    row_threshold=0.5,
    skew_threshold=0.75,
    scaler_type="robust",
    modalities=modalities_keep,
    dummy_code_modalities=modalities_keep,
    mixed_categorical_modalities=[],
)

collinearity_results = assess_domain_collinearity(
    cleaned_discovery,
    meta=meta,
    preprocessed_modalities=collinearity_processed_modalities,
    domain_col="Modality",
    variable_col="ElementName",
    correlation_methods=("pearson", "spearman"),
    high_corr_threshold=0.90,
    moderate_corr_threshold=0.70,
)

collinearity_summary = collinearity_results["summary"]
domain_collinearity_correlation_pairs = collinearity_results["correlation_pairs"]
domain_collinearity_vif = collinearity_results["vif"]
domain_collinearity_condition_index = collinearity_results["condition_index"]
domain_collinearity_skipped_variables = collinearity_results["skipped_variables"]

collinearity_summary_path = "path/to/multiclust_data/domain_collinearity_summary.csv"
domain_collinearity_correlation_pairs_path = "path/to/multiclust_data/domain_collinearity_correlation_pairs.csv"
domain_collinearity_vif_path = "path/to/multiclust_data/domain_collinearity_vif_tolerance.csv"
domain_collinearity_condition_index_path = "path/to/multiclust_data/domain_collinearity_condition_index.csv"
domain_collinearity_skipped_variables_path = "path/to/multiclust_data/domain_collinearity_skipped_variables.csv"

collinearity_summary.to_csv(collinearity_summary_path, index=False)
domain_collinearity_correlation_pairs.to_csv(domain_collinearity_correlation_pairs_path, index=False)
domain_collinearity_vif.to_csv(domain_collinearity_vif_path, index=False)
domain_collinearity_condition_index.to_csv(domain_collinearity_condition_index_path, index=False)
domain_collinearity_skipped_variables.to_csv(domain_collinearity_skipped_variables_path, index=False)

print(f"Saved collinearity summary to: {collinearity_summary_path}")
print(f"Saved correlation-pair diagnostics to: {domain_collinearity_correlation_pairs_path}")
print(f"Saved VIF/tolerance diagnostics to: {domain_collinearity_vif_path}")
print(f"Saved condition-index diagnostics to: {domain_collinearity_condition_index_path}")
print(f"Saved skipped-variable log to: {domain_collinearity_skipped_variables_path}")

print("Table: domain-level collinearity summary")
display(collinearity_summary)

print("Table: pairwise correlation diagnostics, Pearson and Spearman, showing pairs with |r| >= 0.70")
display(domain_collinearity_correlation_pairs)

print("Table: variable-level VIF and tolerance diagnostics within each domain")
display(domain_collinearity_vif)

print("Table: domain-level condition index diagnostics")
display(domain_collinearity_condition_index)


#### Collinearity decision tables

Use these two tables in sequence. First review pairwise redundancy (`|r| >= 0.70`, especially `|r| >= 0.90`) to remove direct duplicates or total/subscale overlap. Then review VIF/domain redundancy to catch variables that are not duplicated by one single partner but are predictable from the rest of the domain.


In [ ]:
def _source_variable_from_feature(feature, raw_columns):
    if pd.isna(feature):
        return np.nan
    feature = str(feature)
    if feature in raw_columns:
        return feature
    matches = [col for col in raw_columns if feature.startswith(f"{col}_")]
    if not matches:
        return feature
    return max(matches, key=len)


def _feature_role(name):
    if pd.isna(name):
        return np.nan
    text = str(name).lower()
    if text.endswith("_tot") or "total" in text:
        return "total/composite"
    if "dimension" in text:
        return "dimension"
    if "domain" in text:
        return "domain/subscale"
    if "high" in text or "low" in text or "curr" in text or "current" in text:
        return "status/threshold score"
    if "sips" in text or "caarms" in text:
        return "item/scale score"
    return "variable"


def _max_numeric(values):
    vals = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    return vals.max() if len(vals) else np.nan


def build_pairwise_collinearity_decision_table(
    correlation_pairs,
    vif_table,
    condition_index_table,
    raw_columns,
    severe_r_threshold=0.90,
    review_r_threshold=0.70,
):
    raw_columns = list(raw_columns)
    rows = []
    if correlation_pairs is None or len(correlation_pairs) == 0:
        return pd.DataFrame(columns=[
            "evidence_type", "domain", "variable_1", "variable_2", "raw_variable_1", "raw_variable_2",
            "role_1", "role_2", "max_abs_pearson", "max_abs_spearman", "max_abs_r",
            "vif_1", "vif_2", "condition_index", "suggested_action", "raw_remove_candidate",
            "decision", "decision_notes"
        ])

    vif_lookup = vif_table.set_index(["domain", "variable"])["vif"].to_dict() if vif_table is not None and len(vif_table) else {}
    ci_lookup = condition_index_table.set_index("domain")["condition_index"].to_dict() if condition_index_table is not None and len(condition_index_table) else {}

    pairs = correlation_pairs.copy()
    pairs["pair_key"] = pairs.apply(
        lambda r: tuple(sorted([str(r["variable_1"]), str(r["variable_2"])])), axis=1
    )
    for (domain, pair_key), sub in pairs.groupby(["domain", "pair_key"], sort=False):
        variable_1, variable_2 = pair_key
        pearson = _max_numeric(sub.loc[sub["method"] == "pearson", "abs_correlation"])
        spearman = _max_numeric(sub.loc[sub["method"] == "spearman", "abs_correlation"])
        max_abs_r = _max_numeric([pearson, spearman])
        if pd.isna(max_abs_r) or max_abs_r < review_r_threshold:
            continue
        rows.append({
            "evidence_type": "severe_pairwise_r" if max_abs_r >= severe_r_threshold else "moderate_high_pairwise_r",
            "domain": domain,
            "variable_1": variable_1,
            "variable_2": variable_2,
            "raw_variable_1": _source_variable_from_feature(variable_1, raw_columns),
            "raw_variable_2": _source_variable_from_feature(variable_2, raw_columns),
            "role_1": _feature_role(variable_1),
            "role_2": _feature_role(variable_2),
            "max_abs_pearson": pearson,
            "max_abs_spearman": spearman,
            "max_abs_r": max_abs_r,
            "vif_1": vif_lookup.get((domain, variable_1), np.nan),
            "vif_2": vif_lookup.get((domain, variable_2), np.nan),
            "condition_index": ci_lookup.get(domain, np.nan),
            "suggested_action": "choose one variable to keep; remove the more derivative/duplicated construct" if max_abs_r >= severe_r_threshold else "review; keep both only if clinically distinct",
            "raw_remove_candidate": "",
            "decision": "",
            "decision_notes": "",
        })

    table = pd.DataFrame(rows)
    if len(table) == 0:
        return table
    table["max_vif_in_pair"] = pd.to_numeric(table[["vif_1", "vif_2"]].max(axis=1), errors="coerce")
    table = table.sort_values(
        ["domain", "evidence_type", "max_abs_r", "max_vif_in_pair"],
        ascending=[True, False, False, False],
    ).drop(columns=["max_vif_in_pair"]).reset_index(drop=True)
    return table


def build_vif_collinearity_decision_table(
    vif_table,
    condition_index_table,
    raw_columns,
    high_vif_threshold=10.0,
):
    raw_columns = list(raw_columns)
    if vif_table is None or len(vif_table) == 0:
        return pd.DataFrame(columns=[
            "evidence_type", "domain", "variable", "raw_variable", "role", "vif", "tolerance",
            "r_squared_with_other_variables", "condition_index", "suggested_action", "raw_remove_candidate",
            "decision", "decision_notes"
        ])

    ci_lookup = condition_index_table.set_index("domain")["condition_index"].to_dict() if condition_index_table is not None and len(condition_index_table) else {}
    high_vif = vif_table[pd.to_numeric(vif_table["vif"], errors="coerce") >= high_vif_threshold].copy()
    rows = []
    for _, row in high_vif.iterrows():
        rows.append({
            "evidence_type": "high_vif",
            "domain": row["domain"],
            "variable": row["variable"],
            "raw_variable": _source_variable_from_feature(row["variable"], raw_columns),
            "role": _feature_role(row["variable"]),
            "vif": row["vif"],
            "tolerance": row.get("tolerance", np.nan),
            "r_squared_with_other_variables": row.get("r_squared_with_other_variables", np.nan),
            "condition_index": ci_lookup.get(row["domain"], np.nan),
            "suggested_action": "review redundant set; remove only if construct is duplicated by other variables in the domain",
            "raw_remove_candidate": "",
            "decision": "",
            "decision_notes": "",
        })
    table = pd.DataFrame(rows)
    if len(table) == 0:
        return table
    return table.sort_values(["domain", "vif"], ascending=[True, False]).reset_index(drop=True)


raw_clustering_columns = cleaned_discovery.columns.tolist()
pairwise_collinearity_decision_table = build_pairwise_collinearity_decision_table(
    domain_collinearity_correlation_pairs,
    domain_collinearity_vif,
    domain_collinearity_condition_index,
    raw_columns=raw_clustering_columns,
)
vif_collinearity_decision_table = build_vif_collinearity_decision_table(
    domain_collinearity_vif,
    domain_collinearity_condition_index,
    raw_columns=raw_clustering_columns,
)

# Backward-compatible combined table for export only.
collinearity_decision_table = pd.concat(
    [
        pairwise_collinearity_decision_table.assign(decision_table="pairwise"),
        vif_collinearity_decision_table.assign(decision_table="vif"),
    ],
    ignore_index=True,
    sort=False,
)

base_path = "path/to/multiclust_data"
pairwise_decision_table_path = f"{base_path}/domain_collinearity_pairwise_decision_table.csv"
vif_decision_table_path = f"{base_path}/domain_collinearity_vif_decision_table.csv"
collinearity_decision_table_path = f"{base_path}/domain_collinearity_decision_table.csv"

pairwise_collinearity_decision_table.to_csv(pairwise_decision_table_path, index=False)
vif_collinearity_decision_table.to_csv(vif_decision_table_path, index=False)
collinearity_decision_table.to_csv(collinearity_decision_table_path, index=False)

print(f"Saved pairwise decision table to: {pairwise_decision_table_path}")
print(f"Saved VIF decision table to: {vif_decision_table_path}")
print(f"Saved combined decision table to: {collinearity_decision_table_path}")

print("Table: pairwise collinearity decisions")
display(pairwise_collinearity_decision_table)

print("Table: VIF/domain-level collinearity decisions")
display(vif_collinearity_decision_table)


#### Select variables to remove and rerun collinearity

Edit `variables_to_remove` below after reviewing both decision tables. Start with the pairwise table to remove direct duplicates, then use the VIF table to decide whether any remaining domain-level redundancy still needs pruning. Use raw variable names from `raw_variable_1`, `raw_variable_2`, `raw_variable`, or `raw_remove_candidate`, not encoded dummy-column names unless the encoded feature is also a raw column. This creates pruned dataframes and reruns the diagnostics without overwriting the original cleaned CSVs.


In [ ]:
# Edit this list after reviewing the pairwise and VIF decision tables.
# Example:
# variables_to_remove = [
#     "chrnsipr_diminished_expression_dimension",
#     "sips_pos_tot",
# ]
variables_to_remove = ["chrnsipr_diminished_expression_dimension", "chrnsipr_motivation_and_pleasure_dimension", "chrgfs_gf_social_high", "chrgfr_gf_role_low", "chrgfr_gf_role_high", "chrsofas_lowscore", ]

variables_to_remove = list(dict.fromkeys([str(v).strip() for v in variables_to_remove if str(v).strip()]))
unknown_remove_vars = sorted(set(variables_to_remove).difference(cleaned_discovery.columns))
if unknown_remove_vars:
    raise ValueError(f"Variables requested for removal are not in cleaned_discovery: {unknown_remove_vars}")

cleaned_discovery_pruned = cleaned_discovery.drop(columns=variables_to_remove, errors="ignore")
cleaned_discovery_CC_pruned = cleaned_discovery_CC.drop(columns=variables_to_remove, errors="ignore")
cleaned_discovery_all_pruned = cleaned_discovery_all.drop(columns=variables_to_remove, errors="ignore")
cleaned_test_pruned = cleaned_test.drop(columns=variables_to_remove, errors="ignore")
meta_pruned = meta.loc[~meta["ElementName"].isin(variables_to_remove)].copy()

print(f"Removed {len(variables_to_remove)} variable(s): {variables_to_remove}")
print(f"Discovery data shape before pruning: {cleaned_discovery.shape}")
print(f"Discovery data shape after pruning:  {cleaned_discovery_pruned.shape}")

_, collinearity_subject_ids_pruned, collinearity_processed_modalities_pruned = preprocessing(
    cleaned_discovery_pruned,
    meta_pruned,
    subject_id_column="src_subject_id",
    col_threshold=0.5,
    row_threshold=0.5,
    skew_threshold=0.75,
    scaler_type="robust",
    modalities=modalities_keep,
    dummy_code_modalities=modalities_keep,
    mixed_categorical_modalities=[],
)

collinearity_results_pruned = assess_domain_collinearity(
    cleaned_discovery_pruned,
    meta=meta_pruned,
    preprocessed_modalities=collinearity_processed_modalities_pruned,
    domain_col="Modality",
    variable_col="ElementName",
    correlation_methods=("pearson", "spearman"),
    high_corr_threshold=0.90,
    moderate_corr_threshold=0.70,
)

collinearity_summary_pruned = collinearity_results_pruned["summary"]
domain_collinearity_correlation_pairs_pruned = collinearity_results_pruned["correlation_pairs"]
domain_collinearity_vif_pruned = collinearity_results_pruned["vif"]
domain_collinearity_condition_index_pruned = collinearity_results_pruned["condition_index"]
domain_collinearity_skipped_variables_pruned = collinearity_results_pruned["skipped_variables"]

pruned_base = "path/to/multiclust_data"
collinearity_summary_pruned.to_csv(f"{pruned_base}/domain_collinearity_summary_pruned.csv", index=False)
domain_collinearity_correlation_pairs_pruned.to_csv(f"{pruned_base}/domain_collinearity_correlation_pairs_pruned.csv", index=False)
domain_collinearity_vif_pruned.to_csv(f"{pruned_base}/domain_collinearity_vif_tolerance_pruned.csv", index=False)
domain_collinearity_condition_index_pruned.to_csv(f"{pruned_base}/domain_collinearity_condition_index_pruned.csv", index=False)
domain_collinearity_skipped_variables_pruned.to_csv(f"{pruned_base}/domain_collinearity_skipped_variables_pruned.csv", index=False)

print("Table: pruned domain-level collinearity summary")
display(collinearity_summary_pruned)

print("Table: pruned pairwise correlation diagnostics, Pearson and Spearman, showing pairs with |r| >= 0.70")
display(domain_collinearity_correlation_pairs_pruned)

print("Table: pruned variable-level VIF and tolerance diagnostics within each domain")
display(domain_collinearity_vif_pruned)

print("Table: pruned domain-level condition index diagnostics")
display(domain_collinearity_condition_index_pruned)


In [ ]:
cleaned_discovery_CC


In [ ]:
# Save the cleaned dataframes to a CSV file
cleaned_discovery.to_csv(
    "path/to/"
    "multiclust_data/cleaned_discovery_data.csv",
    index=False
)

cleaned_test.to_csv(
    "path/to/multiclust_data/cleaned_test_data.csv",
    index=False
)

cleaned_discovery_all.to_csv(
    "path/to/multiclust_data/cleaned_discovery_data_all.csv",
    index=False
)


# Full pipeline results

## Load in results

The actual clustering pipeline is run on Spartan using the full pipeline scripts.

In [ ]:

# Load in the metrics.pkl from each training fold 
# Results_noDim_4opt_5fold_FIXED_highmut_onlyagree_min50_SVM_ARI
base = 'path/to/results/study_1/release/FinalRun_PCA99_extended100-100'
# Define results path
results_dir = os.path.join(base, "results/")
#intermediates_dir = os.path.join(base, "intermediates/")

# Make and define directory for plots etc.
plots_dir = os.path.join(base, "plots/")
os.makedirs(plots_dir, exist_ok=True)
# Find all fold directories that contain a metrics.pkl
metrics_files = sorted(glob.glob(os.path.join(results_dir, 'fold*/metrics.pkl')))

# Load metrics dynamically
metrics = {}

for metrics_file in metrics_files:
    fold_name = os.path.basename(os.path.dirname(metrics_file))  # e.g., 'fold0'
    with open(metrics_file, 'rb') as f:
        metrics[fold_name] = pickle.load(f)

# Access your metrics like this:
# metrics['fold0'], metrics['fold1'], etc.
print(f"Loaded metrics for folds: {list(metrics.keys())}")



In [ ]:

# Print best_params for each fold
for fold_name, data in metrics.items():
    print(f"{fold_name}: {data['best_params']}")


In [ ]:

# Collect parameters from folds
param_list = [data["best_params"] for fold_name, data in metrics.items()]
param_df = pd.DataFrame(param_list)

reconstructed = {}

for col in param_df.columns:
    s = param_df[col]

    # Column contains lists/tuples → per-position mode
    if s.apply(lambda x: isinstance(x, (list, tuple))).any():
        # turn each list into a row of a small DF
        tmp = pd.DataFrame(s.tolist())
        # mode per column, take first mode; convert back to list
        reconstructed[col] = tmp.mode().iloc[0].tolist()
    else:
        # scalar column → simple mode
        m = s.mode()
        reconstructed[col] = m.iloc[0] if not m.empty else None

print(reconstructed)


## Check quality of clusters

### Stability of best individual in each fold

In [ ]:
# Print best_fitness for each fold
for fold_name, data in metrics.items():
    print(f"Best individual fitness in {fold_name}: {data['best_fitness']}")


# Final on all data results

## Load in results and check quality

In [ ]:
# Load in the metrics.pkl from each training fold 
# Find all fold directories that contain a metrics.pkl
final_metric_file = sorted(glob.glob(os.path.join(results_dir, 'final/final_metrics.pkl')))

with open(final_metric_file[0], 'rb') as f:
    final_metrics = pickle.load(f)

print(f"Loaded metrics from: {final_metric_file[0]}")


In [ ]:
# Print final params

print("Final parameters used in the model:", final_metrics['final_params'])


In [ ]:
final_metrics.keys()


In [ ]:
# Print the final metrics

print("Final quality per view:", final_metrics['view_scores_per_view'])
print("Final mean quality across views:", final_metrics['view_quality_mean'])
print("Final quality of integrated cluster:", final_metrics['final_quality'])

print("Final cluster stability per view:", final_metrics['per_view_stabilities'])
print("Final mean cluster stability across views:", final_metrics['mean_view_stability'])
print("Final integrated cluster stability:", final_metrics['final_stability'])

cluster_pvalues = final_metrics.get('cluster_pvalues', {})
quality_pvalues = cluster_pvalues.get('pvalues_raw', {})
quality_pvalues_fdr = cluster_pvalues.get('pvalues_fdr', {})
ari_pvalues = cluster_pvalues.get('ari_stability', {}).get('pvalues_raw', {})
ari_pvalues_fdr = cluster_pvalues.get('ari_stability', {}).get('pvalues_fdr', {})

print("\nQuality p-values (raw):")
print("  Per modality:", quality_pvalues.get('modalities'))
print("  Final integrated:", quality_pvalues.get('final'))
print("Quality p-values (FDR):")
print("  Per modality:", quality_pvalues_fdr.get('modalities'))
print("  Final integrated:", quality_pvalues_fdr.get('with_final'))

print("\nARI stability p-values (raw):")
print("  Per modality:", ari_pvalues.get('modalities'))
print("  Final integrated:", ari_pvalues.get('final'))
print("ARI stability p-values (FDR):")
print("  Per modality:", ari_pvalues_fdr.get('modalities'))
print("  Final integrated:", ari_pvalues_fdr.get('with_final'))


In [ ]:
final_metrics['cluster_pvalues']


In [ ]:
print("Final mean cluster stability across views (MAT_CCC):", final_metrics['mean_view_stability_MAT_CCC'])
print("Final mean cluster stability across views (MAT_PAC):", final_metrics['mean_view_stability_MAT_PAC'])

print("Final integrated cluster stability (CCC):", final_metrics['final_stability_SUM_MAT_full']['CCC'])
print("Final integrated cluster stability (PAC):", final_metrics['final_stability_SUM_MAT_full']['PAC'])


In [ ]:
for i in range(0, len(final_metrics['per_view_stabilities_SUM_MAT_full'])):
    modality = final_metrics['per_view_stabilities_SUM_MAT_full'][i]
    print(f"Final cluster stability for modality {i} (CCC):", modality['CCC'])
    print(f"Final cluster stability for modality {i} (PAC):", modality['PAC'])


In [ ]:
final_metrics['final_reporting']


## Cluster validation sensitivity analyses

This section reports the optional no-cluster / continuum sensitivity analyses when they were enabled for the run profile and the exported sensitivity files are present.

In [ ]:
# Optional cluster-validation / continuum sensitivity analyses
from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd


def _truthy_profile_value(value):
    return str(value).strip().strip('"\'').upper() in {"TRUE", "1", "YES", "Y"}


def _parse_profile_exports(profile_path):
    exports = {}
    if profile_path is None or not profile_path.exists():
        return exports
    for line in profile_path.read_text().splitlines():
        line = line.strip()
        if not line.startswith("export ") or "=" not in line:
            continue
        key, value = line[len("export "):].split("=", 1)
        value = value.strip()
        # Keep literal defaults such as ${VAR:-0} as-is; simple quoted values are enough for the flag check.
        exports[key.strip()] = value.strip('"\'')
    return exports


def _infer_notebook_profile():
    notebook_dir = Path.cwd().name
    profile_candidates = []
    if notebook_dir == "clinical_paper":
        profile_candidates.append("clinical_paper")
    elif notebook_dir == "multiclust_extended":
        profile_candidates.append("multiclust_extended")
    elif notebook_dir == "prospect":
        profile_candidates.append("prospect")
    elif notebook_dir == "schizbull_legacy":
        profile_candidates.append("schizbull_legacy")
    profile_candidates.append(os.environ.get("RUN_PROFILE", ""))
    for candidate in profile_candidates:
        if candidate:
            return candidate
    return None


def _find_repo_root(start):
    start = Path(start).resolve()
    for parent in [start] + list(start.parents):
        if (parent / "run_profiles").is_dir() and (parent / "full_pipeline.py").exists():
            return parent
    return None


def _profile_enabled_for_sensitivity(repo_root, profile_name):
    if not profile_name or repo_root is None:
        return None, None
    profile_path = repo_root / "run_profiles" / f"{profile_name}.sh"
    exports = _parse_profile_exports(profile_path)
    value = exports.get("DO_CLUSTER_VALIDATION_SENSITIVITY", "FALSE")
    return _truthy_profile_value(value), profile_path


def _display_if_available(obj):
    if "display" in globals():
        display(obj)
    else:
        print(obj)


def _get_nested(dct, path, default=np.nan):
    cur = dct
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def _flatten_sensitivity_results(results):
    rows = []
    for solution, payload in results.get("solutions", {}).items():
        rows.append({
            "solution": solution,
            "kind": payload.get("kind"),
            "observed_k": _get_nested(payload, ["observed_quality", "k"]),
            "observed_n": _get_nested(payload, ["observed_quality", "n"]),
            "observed_n_features": _get_nested(payload, ["observed_quality", "n_features"]),
            "observed_composite": _get_nested(payload, ["observed_quality", "composite"]),
            "observed_silhouette": _get_nested(payload, ["observed_quality", "silhouette"]),
            "observed_calinski_harabasz": _get_nested(payload, ["observed_quality", "calinski_harabasz"]),
            "observed_davies_bouldin": _get_nested(payload, ["observed_quality", "davies_bouldin"]),
            "k1_composite": _get_nested(payload, ["uni_cluster_baseline", "quality", "composite"]),
            "pc1_variance_explained": _get_nested(payload, ["pc1_median_split", "pc1_variance_explained"]),
            "pc1_median_composite": _get_nested(payload, ["pc1_median_split", "quality", "composite"]),
            "pc1_median_ari_with_observed": _get_nested(payload, ["pc1_median_split", "ari_with_observed_labels"]),
            "pc1_median_stability_ari": _get_nested(payload, ["pc1_median_split", "bootstrap_stability", "mean_ari"]),
            "pc1_median_stability_sd_ari": _get_nested(payload, ["pc1_median_split", "bootstrap_stability", "sd_ari"]),
            "dip_available": _get_nested(payload, ["dip_test_pc1", "available"], default=False),
            "dip_statistic_pc1": _get_nested(payload, ["dip_test_pc1", "dip"]),
            "dip_p_value_pc1": _get_nested(payload, ["dip_test_pc1", "p_value"]),
            "gap_selected_k_tibshirani": _get_nested(payload, ["gap_statistic", "selected_k_tibshirani_rule"]),
            "gap_selected_k_max_gap": _get_nested(payload, ["gap_statistic", "selected_k_max_gap"]),
            "sigclust_cluster_index": _get_nested(payload, ["sigclust_approx", "observed_cluster_index"]),
            "sigclust_p_value": _get_nested(payload, ["sigclust_approx", "p_value"]),
            "sigclust_null_mean_cluster_index": _get_nested(payload, ["sigclust_approx", "null_mean_cluster_index"]),
            "gaussian_null_p_quality": _get_nested(payload, ["covariance_matched_gaussian_null", "p_quality_ge_observed_recluster"]),
            "gaussian_null_p_stability": _get_nested(payload, ["covariance_matched_gaussian_null", "p_stability_ge_observed_recluster"]),
            "observed_recluster_stability_ari": _get_nested(payload, ["covariance_matched_gaussian_null", "observed_recluster_stability", "mean_ari"]),
            "gaussian_null_mean_stability_ari": _get_nested(payload, ["covariance_matched_gaussian_null", "null_stability_mean_ari"]),
            "gaussian_null_mean_quality": _get_nested(payload, ["covariance_matched_gaussian_null", "null_quality_mean"]),
        })
    return pd.DataFrame(rows)


_repo_root = _find_repo_root(Path.cwd())
_profile_name = "clinical_paper"
_sensitivity_enabled, _profile_path = _profile_enabled_for_sensitivity(_repo_root, _profile_name)

if _sensitivity_enabled is False:
    print(
        "Cluster validation sensitivity analyses were not enabled for "
        f"profile '{_profile_name}' ({_profile_path})."
    )
else:
    _base_results_dir = Path(results_dir) if "results_dir" in globals() else None
    _sensitivity_dir = _base_results_dir / "final" / "cluster_validation_sensitivity" if _base_results_dir else None
    _summary_csv = _sensitivity_dir / "cluster_validation_sensitivity_summary.csv" if _sensitivity_dir else None
    _results_json = _sensitivity_dir / "cluster_validation_sensitivity_results.json" if _sensitivity_dir else None

    if _sensitivity_enabled is None:
        print("Could not infer whether the run profile enabled cluster validation sensitivity analyses.")
    elif _sensitivity_enabled:
        print(f"Cluster validation sensitivity analyses were enabled for profile '{_profile_name}'.")

    if not _summary_csv or not _summary_csv.exists():
        print(
            "No cluster validation sensitivity summary was found. "
            "Expected file:",
            _summary_csv,
        )
    else:
        cluster_validation_sensitivity_summary = pd.read_csv(_summary_csv)
        print("Cluster validation sensitivity summary:", _summary_csv)
        _display_if_available(cluster_validation_sensitivity_summary)

        if _results_json and _results_json.exists():
            with open(_results_json, "r") as f:
                cluster_validation_sensitivity_results = json.load(f)

            print("Included sensitivity tests:")
            for _test in cluster_validation_sensitivity_results.get("included_tests", []):
                print("-", _test)

            cluster_validation_sensitivity_details = _flatten_sensitivity_results(
                cluster_validation_sensitivity_results
            )
            print("Detailed sensitivity metrics:")
            _display_if_available(cluster_validation_sensitivity_details)
        else:
            print("Detailed sensitivity JSON not found:", _results_json)

        _gap_tables = sorted((_sensitivity_dir / "tables").glob("*/*_gap_statistic.csv"))
        cluster_validation_gap_tables = {}
        if _gap_tables:
            print("Gap-statistic tables:")
            for _gap_table in _gap_tables:
                _gap_name = _gap_table.parent.name
                cluster_validation_gap_tables[_gap_name] = pd.read_csv(_gap_table)
                print("-", _gap_table)
                _display_if_available(cluster_validation_gap_tables[_gap_name])

        _plot_files = sorted((_sensitivity_dir / "plots").glob("*.pdf")) + sorted((_sensitivity_dir / "plots").glob("*.png"))
        if _plot_files:
            print("Sensitivity plot files:")
            for _plot_file in _plot_files:
                print("-", _plot_file)

## Matrix plots

#### Matrix plots final labels

In [ ]:
# Grab consensus
M = final_metrics['final_stability_SUM_MAT_full']['consensus']
M = np.asarray(M, dtype=float)

# Safety checks
assert M.ndim == 2 and M.shape[0] == M.shape[1], "Consensus must be square."
M = (M + M.T) / 2.0           # enforce symmetry (just in case)
np.fill_diagonal(M, 1.0)      # conventional

# Distance from consensus
D = 1.0 - M
dvec = squareform(D, checks=False)

# Hierarchical clustering (match your CCC linkage_method if you want)
Z = linkage(dvec, method="average")

# Leaf order and reordered matrix
order = leaves_list(Z)
M_ord = M[np.ix_(order, order)]

# Plot
plt.figure()
plt.imshow(M_ord, aspect='auto')
#plt.title("Consensus matrix (reordered by hierarchical clustering)")
plt.xlabel("Subjects")
plt.ylabel("Subjects")
plt.colorbar(label="Consensus")
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

diag = final_metrics['final_stability_SUM_MAT_full']
M = np.asarray(diag['consensus'], dtype=float)
M = (M + M.T) / 2.0
np.fill_diagonal(M, 1.0)

union_ids = np.asarray(diag['union_ids'])

# --- you need the IDs that match final_labels order ---
final_labels = np.asarray(final_metrics['final_labels'])

# Use the first retained modality; all final-metric modality frames share subject order.
reference_modality = next(iter(final_metrics['data']))
final_ids = np.asarray(final_metrics['data'][reference_modality]['src_subject_id'])

# Map id -> label
id2lab = dict(zip(final_ids, final_labels))

# Align labels to consensus matrix order
labels_aligned = np.array([id2lab[sid] for sid in union_ids])

# Now sort consensus by aligned labels
order = np.lexsort((np.arange(len(labels_aligned)), labels_aligned))
M_ord = M[np.ix_(order, order)]
labels_ord = labels_aligned[order]

plt.figure()
plt.imshow(M_ord, aspect='auto')
plt.title("Consensus matrix (sorted by final labels, aligned to union_ids)")
plt.colorbar(label="Consensus")
plt.show()


In [ ]:
M = final_metrics['final_stability_SUM_MAT_full']['consensus']
M = np.asarray(M, float)
iu = np.triu_indices_from(M, k=1)
vals = M[iu]

print("fraction <0.1:", np.mean(vals < 0.1))
print("fraction >0.9:", np.mean(vals > 0.9))
print("PAC (0.1..0.9):", np.mean((vals > 0.1) & (vals < 0.9)))
print("mean:", np.mean(vals), "median:", np.median(vals))


#### Matrix plots per view

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, leaves_list, cut_tree
from scipy.cluster import hierarchy


indiv_labels_TEST = []
for i in range(0, len(final_metrics['per_view_stabilities_SUM_MAT_full'])):
    modality = final_metrics['per_view_stabilities_SUM_MAT_full'][i]
    mod_name = list(final_metrics['data'].keys())[i]
    print(f"Processing modality {mod_name} with stability metrics: CCC={modality['CCC']}, PAC={modality['PAC']}")
    M = np.asarray(modality['consensus'], dtype=float)

    # Safety checks
    assert M.ndim == 2 and M.shape[0] == M.shape[1], "Consensus must be square."
    M = (M + M.T) / 2.0           # enforce symmetry (just in case)
    np.fill_diagonal(M, 1.0)      # conventional

    # Distance from consensus
    D = 1.0 - M
    dvec = squareform(D, checks=False)

    # Hierarchical clustering (match your CCC linkage_method if you want)
    Z = linkage(dvec, method="average")

    indiv_labels_TEST.append(mod_name)


    # Leaf order and reordered matrix
    order = leaves_list(Z)
    M_ord = M[np.ix_(order, order)]

    # Plot
    plt.figure()
    plt.imshow(M_ord, aspect='auto')
    #plt.title(f"Consensus matrix for {mod_name} (reordered by hierarchical clustering)")
    plt.xlabel("Subjects")
    plt.ylabel("Subjects")
    #plt.colorbar(label="Consensus")
    plt.show()


In [ ]:
for i in range(0, len(final_metrics['per_view_stabilities_SUM_MAT_full'])):
    modality = final_metrics['per_view_stabilities_SUM_MAT_full'][i]
    mod_name = list(final_metrics['data'].keys())[i]
    M = np.asarray(modality['consensus'], dtype=float)
    M = (M + M.T) / 2.0
    np.fill_diagonal(M, 1.0)

    labels = final_metrics['individual_labels'][i]

    # Align these labels to this modality's consensus-matrix ID order.
    final_ids = np.asarray(final_metrics['data'][mod_name]['src_subject_id'])

    # Map id -> label
    id2lab = dict(zip(final_ids, labels))

    # Align labels to this modality's consensus matrix order.
    union_ids_view = np.asarray(modality['union_ids'])
    labels_aligned = np.array([id2lab[sid] for sid in union_ids_view])

    # Now sort consensus by aligned labels
    order = np.lexsort((np.arange(len(labels_aligned)), labels_aligned))
    M_ord = M[np.ix_(order, order)]
    labels_ord = labels_aligned[order]


    plt.figure()
    plt.imshow(M_ord, aspect='auto')
    plt.title(f"Consensus matrix for {mod_name} (reordered by modality  cluster labels)")
    plt.xlabel("Subjects (sorted by label)")
    plt.ylabel("Subjects (sorted by label)")
    plt.colorbar(label="Consensus")
    plt.show()


## tSNE plots

### Individual labels

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import numpy as np
import umap  # pip install umap-learn

#Create directory for final latent space plots
os.makedirs(os.path.join(plots_dir, "merged_tsne/"), exist_ok=True)

data = final_metrics['ae_res']                     # dict: modality -> DataFrame
labels_list = final_metrics['individual_labels']  # list of label vectors

modality_names = list(data.keys())

if len(labels_list) != len(modality_names):
    raise ValueError(
        f"labels_list has {len(labels_list)} items but there are "
        f"{len(modality_names)} modalities for {fold_name}."
    )

for i, modality in enumerate(modality_names):
    print(f"  Modality: {modality}")
    df = data[modality]
    X = np.asarray(df['final_latent'])
    y = np.asarray(labels_list[i])

    if len(y) != len(X):
        raise ValueError(
            f"Label vector length ({len(y)}) does not match samples ({len(X)}) "
            f"for {fold_name}, modality {modality}."
        )

    # ----- color mapping -----
    classes = np.unique(y)
    palette = sns.color_palette(n_colors=len(classes))  # <-- uses current theme palette
    color_map = {cls: palette[j] for j, cls in enumerate(classes)}
    colors = [color_map[cls] for cls in y]

    # ----- dimensionality reductions -----
    pca = PCA(n_components=2)
    pca_proj = pca.fit_transform(X)

    tsne_2d = TSNE(n_components=2, perplexity=30, random_state=42)
    tsne_proj_2d = tsne_2d.fit_transform(X)

    umap_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    umap_proj_2d = umap_2d.fit_transform(X)

    # ----- plots -----
    fig = plt.figure(figsize=(22, 5))

    ax1 = fig.add_subplot(1, 4, 1)
    ax1.scatter(pca_proj[:, 0], pca_proj[:, 1], c=colors, alpha=0.6, edgecolors="none")
    ax1.set_title(f'PCA (2D) — {modality}')

    ax2 = fig.add_subplot(1, 4, 2)
    ax2.scatter(tsne_proj_2d[:, 0], tsne_proj_2d[:, 1], c=colors, alpha=0.6, edgecolors="none")
    ax2.set_title(f't-SNE (2D) — {modality}')

    ax4 = fig.add_subplot(1, 4, 3)
    ax4.scatter(umap_proj_2d[:, 0], umap_proj_2d[:, 1], c=colors, alpha=0.6, edgecolors="none")
    ax4.set_title(f'UMAP (2D) — {modality}')

    # unified legend
    handles = [Line2D([0], [0], marker='o', linestyle='', color=color_map[cls], label=str(cls))
                for cls in classes]
    for ax in [ax1, ax2, ax4]:
        ax.legend(handles=handles, title='Label', loc='best', frameon=True)

    plt.tight_layout()
    fig.savefig(os.path.join(plots_dir, "merged_tsne/", f"{fold_name}_{modality}_merged_individual_tsne.png"))
    plt.show()


### Final labels

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import numpy as np



data = final_metrics['ae_res']                       # dict: modality -> DataFrame
y = np.asarray(final_metrics['final_labels'])  # one label vector for the whole fold

# Build a stable color map for this fold (consistent colors across all modalities)
classes = np.unique(y)
palette = sns.color_palette(n_colors=len(classes))  # <-- uses current theme palette
color_map = {cls: palette[j] for j, cls in enumerate(classes)}
colors = [color_map[cls] for cls in y]

modality_names = list(data.keys())

for modality in modality_names:
    print(f"  Modality: {modality}")
    df = data[modality]
    X = np.asarray(df['final_latent'])

    if len(y) != len(X):
        raise ValueError(
            f"Label vector length ({len(y)}) does not match samples ({len(X)}) "
            f"for {fold_name}, modality {modality}. Ensure correct alignment."
        )

    colors = [color_map[cls] for cls in y]

    # --- Dimensionality reductions ---
    pca = PCA(n_components=2)
    pca_proj = pca.fit_transform(X)

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    tsne_proj = tsne.fit_transform(X)

    # --- Plot side-by-side ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(pca_proj[:, 0], pca_proj[:, 1], c=colors, alpha=0.6, edgecolors="none")
    axes[0].set_title(f'PCA projection — {modality} ({fold_name})')

    axes[1].scatter(tsne_proj[:, 0], tsne_proj[:, 1], c=colors, alpha=0.6, edgecolors="none")
    axes[1].set_title(f't-SNE projection — {modality} ({fold_name})')

    # Legend
    handles = [
        Line2D([0], [0], marker='o', linestyle='', color=color_map[cls], label=str(cls))
        for cls in classes
    ]
    for ax in axes:
        ax.legend(handles=handles, title='Label', loc='best', frameon=True)

    plt.tight_layout()
    fig.savefig(os.path.join(plots_dir, "merged_tsne/", f"{modality}_merged_final_tsne.png"))
    plt.show()


### Plot final labels across modalities

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import numpy as np


data = final_metrics['ae_res']                       # modality -> dict with 'final_latent'
y = np.asarray(final_metrics['final_labels'])  # labels aligned to subjects

modality_names = list(data.keys())
latent_blocks = []

for modality in modality_names:
    X_mod = np.asarray(data[modality]['final_latent'])
    if X_mod.shape[0] != len(y):
        raise ValueError(
            f"Label vector length ({len(y)}) != samples ({X_mod.shape[0]}) "
            f"for {fold_name}, modality {modality}"
        )
    latent_blocks.append(X_mod)

# concatenate [n_samples, sum(latent_dims)]
X_all = np.hstack(latent_blocks)

classes = np.unique(y)
palette = sns.color_palette(n_colors=len(classes))  # <-- uses current theme palette
color_map = {cls: palette[j] for j, cls in enumerate(classes)}
colors = [color_map[cls] for cls in y]

# --- PCA projection over all modalities ---
pca = PCA(n_components=2)
pca_proj = pca.fit_transform(X_all)

# --- t-SNE projection over all modalities ---
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
tsne_proj = tsne.fit_transform(X_all)

# --- UMAP projection over all modalities ---
umap_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
umap_proj = umap_2d.fit_transform(X_all)

# --- Plot side-by-side ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(pca_proj[:, 0], pca_proj[:, 1], c=colors, alpha=0.6, edgecolors="none")
axes[0].set_title(f"PCA — all modalities ({fold_name})")

axes[1].scatter(tsne_proj[:, 0], tsne_proj[:, 1], c=colors, alpha=0.6, edgecolors="none")
axes[1].set_title(f"t-SNE — all modalities ({fold_name})")

axes[2].scatter(umap_proj[:, 0], umap_proj[:, 1], c=colors, alpha=0.6, edgecolors="none")
axes[2].set_title(f"UMAP — all modalities ({fold_name})")

handles = [
    Line2D([0], [0], marker='o', linestyle='', color=color_map[cls], label=str(cls))
    for cls in classes
]
for ax in axes:
    ax.legend(handles=handles, title='Label', loc='best', frameon=True)

plt.tight_layout()
fig.savefig(os.path.join(plots_dir, "merged_tsne/", f"merged_final_alldata_tsne.png"))
plt.show()




### Show the final quality and stability metrics

In [ ]:
print('Mean quality across modalities',final_metrics['view_quality_mean'])
print('Final integrated cluster quality',final_metrics['final_quality'])
print('Mean stability across modalities',final_metrics['mean_view_stability'])
print('Final integrated cluster stability',final_metrics['final_stability'])


## Differences features

### Differences in original features based on modality labels

In [ ]:
from sklearn.feature_selection import f_classif
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Create directory for differences in original features plots
os.makedirs(os.path.join(plots_dir, "merged_feature_differences/"), exist_ok=True)

mod_num=0
for modality, df in final_metrics['data'].items():
    print(f"\n=== Modality: {modality} ===")

    # Extract data and labels
    X = df.drop(columns=['src_subject_id']).values
    clusters = final_metrics['individual_labels'][mod_num]
    feature_names = df.drop(columns=['src_subject_id']).columns

    # --- Feature Ranking (ANOVA F-test) ---
    f_vals, _ = f_classif(X, clusters)
    f_df = (
        pd.DataFrame({'feature': feature_names, 'f_value': f_vals})
        .sort_values('f_value', ascending=False)
    )

    # --- Barplot of Top Features ---
    top_k = 30
    plt.figure(figsize=(10, 6))
    sns.barplot(x='f_value', y='feature', data=f_df.head(top_k), palette=sns.color_palette(n_colors=top_k))
    plt.title(f"{modality} — Top {top_k} Discriminative Features (ANOVA F-value)")
    plt.xlabel("F-value")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()


    # --- Boxplots + Scatter Points of Top Features + CC ---
    top_features = f_df['feature'].head(top_k).tolist()

    max_cols = 3
    n_feats = len(top_features)
    n_rows = int(np.ceil(n_feats / max_cols))

    fig, axes = plt.subplots(n_rows, max_cols, figsize=(max_cols * 6, n_rows * 5))
    axes = axes.flatten()

    cluster_order = [str(x) for x in np.unique(clusters)]
    df_cc = None #dict_final_cc.get(modality)
    cc_available = df_cc is not None
    if cc_available:
        df_cc = df_cc.copy().reindex(columns=df.columns, fill_value=np.nan)
    group_order = cluster_order + (["CC"] if cc_available else [])
    pal = sns.color_palette(n_colors=len(group_order))

    for i, feat in enumerate(top_features):
        ax = axes[i]
        plot_df = pd.DataFrame({
            'group': pd.Series(clusters).astype(str),
            'value': df[feat].values,
        }).dropna()
        if cc_available and feat in df_cc.columns:
            cc_plot_df = pd.DataFrame({
                'group': 'CC',
                'value': df_cc[feat].values,
            }).dropna()
            plot_df = pd.concat([plot_df, cc_plot_df], ignore_index=True)
        sns.boxplot(
            data=plot_df, x='group', y='value', order=group_order, palette=pal, ax=ax
        )
        sns.stripplot(
            data=plot_df, x='group', y='value', order=group_order,
            color='black', size=4, jitter=True, alpha=0.5, ax=ax
        )

        # Larger text sizes
        ax.set_title(feat, fontsize=14)
        ax.set_xlabel("", fontsize=22)
        ax.set_ylabel("", fontsize=22)
        ax.tick_params(axis='both', labelsize=14)

    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f"{modality} — Distribution of Top Features by Cluster and CC\n(with individual data points)",
        y=1.02, fontsize=18
    )

    plt.tight_layout()
    fig.savefig(os.path.join(plots_dir, "merged_feature_differences/", f"{modality}_top_features_with_cc.png"))
    plt.show()

    mod_num=mod_num+1


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# =========================
# PCA aggregated plots per modality + variance explained (first 5 PCs)
# - Respects your existing global theme (NO sns.set_theme here)
# - PC1 by cluster: violin (quartiles) + jitter + median marker + n labels
# - Variance plot: explained variance ratio for PC1..PC5
# - Optional: PC1 loadings (top +/- contributors)
# =========================

out_dir = os.path.join(plots_dir, "merged_feature_pca/")
os.makedirs(out_dir, exist_ok=True)

loadings_dir = os.path.join(out_dir, "pc1_loadings/")
os.makedirs(loadings_dir, exist_ok=True)

variance_dir = os.path.join(out_dir, "variance/")
os.makedirs(variance_dir, exist_ok=True)

mod_num = 0
for modality, df in final_metrics["data"].items():
    print(f"\n=== PCA Aggregated Plots — Modality: {modality} ===")

    # --- Extract X and cluster labels ---
    feature_df = df.drop(columns=["src_subject_id"])
    feature_names = feature_df.columns.to_list()

    X = feature_df.values
    clusters = np.asarray(final_metrics["individual_labels"][mod_num])

    # Stable cluster order (customize if you want a specific ordering)
    cluster_order = np.sort(pd.unique(clusters))

    # --- Standardize ---
    Xz = StandardScaler().fit_transform(X)

    # --- PCA for variance (first 5 components) ---
    n_pcs = min(5, Xz.shape[1])  # cannot exceed number of features
    pca_var = PCA(n_components=n_pcs, random_state=0)
    pca_var.fit(Xz)

    evr = pca_var.explained_variance_ratio_
    cum_evr = np.cumsum(evr)

    # --- Also compute PC scores (at least PC1) ---
    pc_scores = pca_var.transform(Xz)  # shape: (n_samples, n_pcs)
    pc1 = pc_scores[:, 0]
    evr1 = float(evr[0])
    print(f"PC1 EVR: {evr1:.2%}")

    plot_df = pd.DataFrame({"cluster": clusters, "PC1": pc1})

    # -------------------------
    # 1) PC1 by cluster (publication-ready, respects global theme)
    # -------------------------
    fig, ax = plt.subplots(figsize=(10.5, 6.5))


    sns.violinplot(
        data=plot_df,
        x="cluster",
        y="PC1",
        order=cluster_order,
        palette=sns.color_palette(n_colors=len(cluster_order)),
        inner="quartile",
        cut=0,
        bw_adjust=1.0,
        linewidth=1,
        ax=ax
    )

    sns.stripplot(
        data=plot_df,
        x="cluster",
        y="PC1",
        order=cluster_order,
        color="black",
        size=3,
        jitter=0.22,
        alpha=0.35,
        ax=ax
    )

    # Median marker per cluster
    medians = plot_df.groupby("cluster")["PC1"].median()
    x_positions = np.arange(len(cluster_order))
    ax.scatter(
        x=x_positions,
        y=[medians.loc[c] for c in cluster_order],
        s=180,
        marker="_",
        linewidths=3
    )

    # Annotate n per cluster near the bottom
    counts = plot_df["cluster"].value_counts()
    ymin, ymax = ax.get_ylim()
    y_annot = ymin + 0.04 * (ymax - ymin)
    for i, c in enumerate(cluster_order):
        ax.text(i, y_annot, f"n={int(counts.loc[c])}", ha="center", va="top", fontsize=12)

    ax.set_xlabel("Cluster", fontsize=16)
    ax.set_ylabel("PCA Component 1 score", fontsize=16)

    # Light grid for readability; remove if your global theme already handles grids
    ax.grid(axis="y", alpha=0.15)
    sns.despine(ax=ax)

    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"{modality}_PC1_violin_pubready.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # 2) Variance explained (PC1..PC5): bar + cumulative line
    # -------------------------
    figv, axv = plt.subplots(figsize=(10, 5.5))

    pcs_idx = np.arange(1, n_pcs + 1)
    axv.bar(pcs_idx, evr)  # uses your global matplotlib color cycle
    axv.plot(pcs_idx, cum_evr, marker="o")

    axv.set_xticks(pcs_idx)
    axv.set_xlabel("Principal Component")
    axv.set_ylabel("Explained variance ratio")
    axv.set_title(f"{modality} — PCA Variance Explained (Top {n_pcs} PCs)", pad=12)

    axv.set_ylim(0, max(0.25, evr.max() * 1.2))  # keeps plot readable if EVR is small/large
    axv.grid(axis="y", alpha=0.15)
    sns.despine(ax=axv)

    figv.tight_layout()
    figv.savefig(os.path.join(variance_dir, f"{modality}_variance_top{n_pcs}.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # 3) Optional: PC1 feature loadings (top +/- contributors)
    # -------------------------
    loadings = pca_var.components_[0]  # PC1 loadings
    load_df = pd.DataFrame({"feature": feature_names, "loading": loadings})

    top_n = min(15, len(feature_names) // 2) if len(feature_names) >= 2 else 1
    top_pos = load_df.sort_values("loading", ascending=False).head(top_n)
    top_neg = load_df.sort_values("loading", ascending=True).head(top_n)
    load_plot_df = pd.concat([top_neg, top_pos], axis=0)

    fig2, ax2 = plt.subplots(figsize=(10.5, 7.5))
    sns.barplot(data=load_plot_df, x="loading", y="feature", ax=ax2)
    ax2.axvline(0, linewidth=1)

    ax2.set_title(f"{modality} — PC1 Feature Loadings (Top ±{top_n})", pad=12)
    ax2.set_xlabel("PC1 loading")
    ax2.set_ylabel("")

    ax2.grid(axis="x", alpha=0.15)
    sns.despine(ax=ax2)

    fig2.tight_layout()
    fig2.savefig(os.path.join(loadings_dir, f"{modality}_PC1_loadings_top_pm{top_n}.png"), dpi=300, bbox_inches="tight")
    plt.show()

    mod_num += 1


#### Plot with CC

In [ ]:
# Apply the learned CHR preprocessing to BOTH discovery CC and test CC.
# Discovery CC should use discovery CHR preprocessing, while test CC must
# use preprocessing learned from the test CHR sample.
preproc_discovery = final_metrics['preprocessing_details']

cc_df_discovery = cleaned_discovery_CC.copy()
ae_data_cc_discovery, subject_id_list_cc_discovery, dict_final_cc = apply_preprocessing_to_new_data(
    cc_df_discovery,
    meta,
    preproc_discovery,
    subject_id_column="src_subject_id"
)


print("Discovery CC preprocessing complete:", {k: v.shape for k, v in dict_final_cc.items()})


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# =========================
# PCA aggregated plots per modality: CHR clusters + CC
# - Each modality can have its own manual colors
# - Fits scaler+PCA on CHR only, projects CC into same space
# - Black jittered points always in front of violins
# =========================

# Required inputs:
# final_metrics  -> CHR final metrics dict
# dict_final_cc  -> output of apply_preprocessing_to_new_data(...): discovery CC per modality
# plots_dir      -> base directory for plots

out_dir = os.path.join(plots_dir, "merged_feature_pca_chr_vs_cc")
os.makedirs(out_dir, exist_ok=True)

# --------------------------------------------------
# MANUAL COLORS PER MODALITY
# Edit these however you want
# Keys must match your modality names exactly
# Inner keys must match your group labels exactly: "0", "1", "CC", etc.
# --------------------------------------------------
modality_palettes = {
    "Internalising": {
        "0": "#2F7F73",   # deep teal
        "1": "#8EDFD1",   # soft aqua
        "CC": "#065982"
    },
    "Functioning": {
        "0": "#0B7C25",   # muted coral
        "1": "#C5FDC7",   # soft blush-peach
        "CC": "#065982"
    },
    "Detachment": {
        "0": "#7C68C0",   # soft violet
        "1": "#C7B8EA",   # pale lavender
        "CC": "#065982"
    },
    "Psychoticism": {
        "0": "#D15B94",   # muted rose-magenta
        "1": "#E8B4CC",   # light pink-mauve
        "CC": "#065982"
    },
    "Cognition": {
        "0": "#5196E0",   # calm medium blue
        "1": "#B7D4F0",   # light sky blue
        "CC": "#065982"
    }
}



# Optional fallback if a modality is not listed above
default_palette = {
    "0": "#327D6D",
    "1": "#7FE3CD",
    "CC": "#065982"
}


mod_num = 0
for modality, df_chr in final_metrics["data"].items():
    print(f"\n=== PCA Aggregated Plots — Modality: {modality} ===")

    # CHR features + labels
    X_chr_df = df_chr.drop(columns=["src_subject_id"]).copy()
    clusters_chr = np.asarray(final_metrics["individual_labels"][mod_num])

    # CC features
    df_cc = dict_final_cc[modality]
    X_cc_df = df_cc.drop(columns=["src_subject_id"]).copy()

    # Strict feature alignment to CHR
    X_cc_df = X_cc_df.reindex(columns=X_chr_df.columns)

    # Optional diagnostics
    print("CHR shape:", X_chr_df.shape, "CC shape:", X_cc_df.shape)
    print("NaNs in CC feature matrix:", int(X_cc_df.isna().sum().sum()))

    # Convert to arrays
    X_chr = X_chr_df.values
    X_cc = X_cc_df.values

    # Fit scaler + PCA on CHR only
    scaler = StandardScaler()
    X_chr_z = scaler.fit_transform(X_chr)
    X_cc_z = scaler.transform(X_cc)

    n_pcs = min(5, X_chr_z.shape[1])
    pca = PCA(n_components=n_pcs, random_state=0)
    PC_chr = pca.fit_transform(X_chr_z)
    PC_cc = pca.transform(X_cc_z)

    pc1_chr = PC_chr[:, 0]
    pc1_cc = PC_cc[:, 0]

    # Plot dataframe
    plot_chr = pd.DataFrame({
        "group": clusters_chr.astype(str),
        "PC1": pc1_chr,
        "cohort": "CHR"
    })
    plot_cc = pd.DataFrame({
        "group": "CC",
        "PC1": pc1_cc,
        "cohort": "CC"
    })
    plot_df = pd.concat([plot_chr, plot_cc], ignore_index=True)

    cluster_order = sorted(
        plot_chr["group"].unique(),
        key=lambda x: int(x) if str(x).isdigit() else str(x)
    )
    group_order = cluster_order + ["CC"]

    # ---------------------------------------------
    # Pick palette for this modality
    # ---------------------------------------------
    group_palette = modality_palettes.get(modality, default_palette).copy()

    # Safety check: ensure every group has a color
    missing_groups = [g for g in group_order if g not in group_palette]
    if missing_groups:
        raise ValueError(
            f"Missing colors for modality '{modality}' and groups: {missing_groups}"
        )

    fig, ax = plt.subplots(figsize=(10.5, 6.5))

    # Violin by group color
    sns.violinplot(
        data=plot_df,
        x="group",
        y="PC1",
        hue="group",
        order=group_order,
        hue_order=group_order,
        palette=group_palette,
        dodge=False,
        inner="quartile",
        cut=0,
        bw_adjust=1.0,
        linewidth=1,
        legend=False,
        ax=ax
    )

    # Black points on top
    sns.stripplot(
        data=plot_df,
        x="group",
        y="PC1",
        order=group_order,
        color="black",
        size=3,
        jitter=0.22,
        alpha=0.35,
        ax=ax
    )

    # Bring points forward
    for c in ax.collections:
        c.set_zorder(2)
    for l in ax.lines:
        l.set_zorder(3)

    # Median marker
    medians = plot_df.groupby("group")["PC1"].median()
    x_positions = np.arange(len(group_order))
    med_vals = [medians.loc[g] for g in group_order]
    ax.scatter(
        x_positions, med_vals,
        marker="_", s=180, linewidths=3,
        color="black", zorder=4
    )

    # n labels
    counts = plot_df["group"].value_counts()
    ymin, ymax = ax.get_ylim()
    y_annot = ymin + 0.04 * (ymax - ymin)
    for i, g in enumerate(group_order):
        ax.text(i, y_annot, f"n={int(counts.get(g, 0))}",
                ha="center", va="top", fontsize=14)

    ax.set_xlabel("CHR clusters + CC", fontsize=16)
    ax.set_ylabel("PCA Component 1 score", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    ax.set_title(f"{modality}: CHR cluster PC1 vs CC", fontsize=16)
    ax.grid(axis="y", alpha=0.15)
    sns.despine(ax=ax)

    fig.tight_layout()
    fig.savefig(
        os.path.join(out_dir, f"{modality}_PC1_CHR_vs_CC.svg"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    mod_num += 1


### Differences in original features based on final labels

In [ ]:
from sklearn.feature_selection import f_classif
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Create directory for differences in original features plots
os.makedirs(os.path.join(plots_dir, "merged_feature_differences/"), exist_ok=True)

for modality, df in final_metrics['data'].items():
    print(f"\n=== Modality: {modality} ===")

    # Extract data and labels
    X = df.drop(columns=['src_subject_id']).values
    clusters = final_metrics['final_labels']
    feature_names = df.drop(columns=['src_subject_id']).columns

    # --- Feature Ranking (ANOVA F-test) ---
    f_vals, _ = f_classif(X, clusters)
    f_df = (
        pd.DataFrame({'feature': feature_names, 'f_value': f_vals})
        .sort_values('f_value', ascending=False)
    )

    # --- Barplot of Top Features ---
    top_k = 10
    plt.figure(figsize=(10, 6))
    sns.barplot(x='f_value', y='feature', data=f_df.head(top_k), palette=sns.color_palette(n_colors=top_k))
    plt.title(f"{modality} — Top {top_k} Discriminative Features (ANOVA F-value)")
    plt.xlabel("F-value")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()


    # --- Boxplots + Scatter Points of Top Features + CC ---
    top_features = f_df['feature'].head(top_k).tolist()

    max_cols = 3
    n_feats = len(top_features)
    n_rows = int(np.ceil(n_feats / max_cols))

    fig, axes = plt.subplots(n_rows, max_cols, figsize=(max_cols * 6, n_rows * 5))
    axes = axes.flatten()

    cluster_order = [str(x) for x in np.unique(clusters)]
    df_cc = dict_final_cc.get(modality)
    cc_available = df_cc is not None
    if cc_available:
        df_cc = df_cc.copy().reindex(columns=df.columns, fill_value=np.nan)
    group_order = cluster_order + (["CC"] if cc_available else [])
    pal = sns.color_palette(n_colors=len(group_order))

    for i, feat in enumerate(top_features):
        ax = axes[i]
        plot_df = pd.DataFrame({
            'group': pd.Series(clusters).astype(str),
            'value': df[feat].values,
        }).dropna()
        if cc_available and feat in df_cc.columns:
            cc_plot_df = pd.DataFrame({
                'group': 'CC',
                'value': df_cc[feat].values,
            }).dropna()
            plot_df = pd.concat([plot_df, cc_plot_df], ignore_index=True)
        sns.boxplot(
            data=plot_df, x='group', y='value', order=group_order, palette=pal, ax=ax
        )
        sns.stripplot(
            data=plot_df, x='group', y='value', order=group_order,
            color='black', size=4, jitter=True, alpha=0.5, ax=ax
        )

        # Larger text sizes
        ax.set_title(feat, fontsize=14)
        ax.set_xlabel("", fontsize=22)
        ax.set_ylabel("", fontsize=22)
        ax.tick_params(axis='both', labelsize=14)

    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f"{modality} — Distribution of Top Features by Cluster and CC\n(with individual data points)",
        y=1.02, fontsize=18
    )

    plt.tight_layout()
    fig.savefig(os.path.join(plots_dir, "merged_feature_differences/", f"{modality}_top_features_with_cc.png"))
    plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import f_classif

# -----------------------------
# Settings
# -----------------------------
top_k = 15                    # change as you like
max_cols = 3                  # grid columns for feature distributions
out_dir = os.path.join(plots_dir, "merged_feature_differences")
os.makedirs(out_dir, exist_ok=True)

# -----------------------------
# 1) Build ONE merged feature table across ALL modalities
#    aligned by src_subject_id, with modality-prefixed feature names
# -----------------------------
# Use first modality as "reference" ordering (assumes labels align to this order)
modalities = list(final_metrics["data"].keys())
ref_mod = modalities[0]
ref_df = final_metrics["data"][ref_mod].copy()

# Keep reference subject ordering
ref_ids = ref_df["src_subject_id"].astype(str)
clusters = np.asarray(final_metrics["final_labels"])
if len(clusters) != len(ref_df):
    raise ValueError(
        f"final_labels length ({len(clusters)}) != number of subjects in reference modality "
        f"{ref_mod} ({len(ref_df)}). You likely need to align labels by src_subject_id."
    )

# Start merged dataframe with IDs
merged = pd.DataFrame({"src_subject_id": ref_ids.values})

# Add each modality's features, aligned by src_subject_id, modality-prefixed
for modality, df in final_metrics["data"].items():
    tmp = df.copy()
    tmp["src_subject_id"] = tmp["src_subject_id"].astype(str)

    # Drop duplicates and set index for alignment
    tmp = tmp.drop_duplicates("src_subject_id").set_index("src_subject_id")

    # Align to reference IDs (inner join behavior -> missing subjects become NaN)
    tmp = tmp.reindex(ref_ids.values)

    # Prefix feature names to avoid collisions across modalities
    feat_cols = tmp.columns
    tmp = tmp.rename(columns={c: f"{modality}::{c}" for c in feat_cols})

    merged = pd.concat([merged, tmp.reset_index(drop=True)], axis=1)

# Option A: drop any subjects with missing modality data (strict intersection)
merged_clean = merged.dropna(axis=0).reset_index(drop=True)

# Make sure labels match rows after dropping NaNs (keep only subjects that remain)
keep_mask = merged.notna().all(axis=1).values
clusters_clean = clusters[keep_mask]

# Feature matrix
X = merged_clean.drop(columns=["src_subject_id"]).values
feature_names = merged_clean.drop(columns=["src_subject_id"]).columns

# -----------------------------
# 2) Global feature ranking (ANOVA F-test) using integrated labels
# -----------------------------
f_vals, _ = f_classif(X, clusters_clean)
f_df = (
    pd.DataFrame({"feature": feature_names, "f_value": f_vals})
      .sort_values("f_value", ascending=False)
)

# -----------------------------
# 3) Plot: Top-K contributing features ACROSS ALL modalities
# -----------------------------
# --- Barplot of Top Features ---
plt.figure(figsize=(12, 7))
sns.barplot(
    x="f_value", y="feature",
    data=f_df.head(top_k),
    palette=sns.color_palette(n_colors=top_k)
)
plt.title(f"All modalities — Top {top_k} Discriminative Features (ANOVA F-value)")
plt.xlabel("F-value")
plt.ylabel("Feature (modality::feature)")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "ALL_modalities_top_features_bar.png"), dpi=200)
plt.show()

# --- Boxplots + Scatter Points of Top Features ---
top_features = f_df["feature"].head(top_k).tolist()

n_feats = len(top_features)
n_rows = int(np.ceil(n_feats / max_cols))
fig, axes = plt.subplots(n_rows, max_cols, figsize=(max_cols * 6, n_rows * 5))
axes = np.array(axes).reshape(-1)

cluster_order = np.unique(clusters_clean)
pal = sns.color_palette(n_colors=len(cluster_order))

# Build a plotting df where each feature is a column
plot_df = merged_clean.copy()
plot_df["cluster"] = clusters_clean

for i, feat in enumerate(top_features):
    ax = axes[i]
    sns.boxplot(x="cluster", y=feat, data=plot_df, palette=pal, ax=ax)
    sns.stripplot(
        x="cluster", y=feat, data=plot_df,
        color="black", size=4, jitter=True, alpha=0.5, ax=ax
    )
    ax.set_title(feat, fontsize=12)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="both", labelsize=12)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    f"All modalities — Distribution of Top {top_k} Features by Integrated Cluster\n(with individual data points)",
    y=1.02, fontsize=16
)
plt.tight_layout()
fig.savefig(os.path.join(out_dir, "ALL_modalities_top_features_box.png"), dpi=200, bbox_inches="tight")
plt.show()

print("Saved:")
print(" -", os.path.join(out_dir, "ALL_modalities_top_features_bar.png"))
print(" -", os.path.join(out_dir, "ALL_modalities_top_features_box.png"))


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ==========================================================
# 1) SINGLE PLOT: PC1 (per-modality PCA) distributions by modality (hue=integrated cluster)
# 2) ADDITIONAL "GLOBAL" DIFFERENCE: one shared PC1 computed from ALL features across ALL modalities
#    -> a separate global violin plot + optional printout of EVR
#
# IMPORTANT: Test integrated labels are aligned to src_subject_id via
# test_final_labels_by_subject_id, created from the test predictions.
# ==========================================================

out_dir = os.path.join(plots_dir, "merged_feature_pca/")
os.makedirs(out_dir, exist_ok=True)

# --------------------------
# A) Per-modality PC1 plot (your current single plot)
# --------------------------
rows = []
for modality, df in final_metrics["data"].items():
    feature_df = df.drop(columns=["src_subject_id"])
    X = feature_df.values
    clusters = np.asarray(final_metrics["final_labels"])

    Xz = StandardScaler().fit_transform(X)
    pca = PCA(n_components=1, random_state=0).fit(Xz)
    pc1 = pca.transform(Xz)[:, 0]
    evr1 = float(pca.explained_variance_ratio_[0])

    tmp = pd.DataFrame({"modality": modality, "cluster": clusters, "PC1": pc1})
    tmp["PC1_EVR"] = evr1
    rows.append(tmp)

    print(f"{modality}: PC1 EVR={evr1:.2%}")

plot_df = pd.concat(rows, ignore_index=True)

modality_order = list(final_metrics["data"].keys())
cluster_order = np.sort(plot_df["cluster"].unique())

fig_w = max(18, 2.2 * len(modality_order))
fig_h = 9
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

sns.violinplot(
    data=plot_df,
    x="modality",
    y="PC1",
    hue="cluster",
    order=modality_order,
    hue_order=cluster_order,
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    dodge=True,
    width=0.65,
    ax=ax
)

sns.stripplot(
    data=plot_df,
    x="modality",
    y="PC1",
    hue="cluster",
    order=modality_order,
    hue_order=cluster_order,
    dodge=True,
    jitter=0.18,
    size=2.2,
    alpha=0.18,
    color="black",
    ax=ax
)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:len(cluster_order)], labels[:len(cluster_order)], title="Cluster", frameon=False, loc="upper right")

for x in np.arange(0.5, len(modality_order), 1.0):
    ax.axvline(x, linewidth=0.8, alpha=0.25)

ax.set_xlabel("Modality", fontsize=16)
ax.set_ylabel("PCA Component 1 score", fontsize=16)
ax.set_title("PC1 Distributions by Modality (Integrated Cluster Differences)", fontsize=18, pad=14)
ax.grid(axis="y", alpha=0.15)
sns.despine(ax=ax)

plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=13)
plt.setp(ax.get_yticklabels(), fontsize=13)

fig.tight_layout()
fig.savefig(os.path.join(out_dir, "ALL_modalities_PC1_singleplot_violin_big.png"), dpi=300, bbox_inches="tight")
plt.show()


# --------------------------
# B) GLOBAL PCA PC1 across ALL modalities/features (shared PC axis)
# --------------------------

# 1) Merge all modalities into one wide table keyed by src_subject_id
dfs = []
for modality, df in final_metrics["data"].items():
    tmp = df.copy()
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})  # avoid name collisions
    dfs.append(tmp)

merged = dfs[0]
for d in dfs[1:]:
    merged = merged.merge(d, on="src_subject_id", how="inner")  # subjects present in ALL modalities

print(f"\n[GLOBAL PCA] Subjects after inner-join: {merged.shape[0]}")
print(f"[GLOBAL PCA] Total merged features: {merged.shape[1] - 1}")

# 2) Align integrated labels to merged subject IDs
label_map = final_metrics.get("final_labels_by_subject_id", None)

if label_map is not None:
    merged_clusters = merged["src_subject_id"].map(label_map).to_numpy()
    if np.any(pd.isna(merged_clusters)):
        missing = merged.loc[pd.isna(merged_clusters), "src_subject_id"].head(5).tolist()
        raise ValueError(
            "Some merged subjects are missing labels in final_labels_by_subject_id. "
            f"Examples: {missing}"
        )
else:
    # Fallback assumption: final_labels already correspond exactly to rows in the merged table.
    # This is ONLY valid if your data are already aligned and the merge didn't drop/reorder subjects.
    if len(final_metrics["final_labels"]) != merged.shape[0]:
        raise ValueError(
            f"final_labels length ({len(final_metrics['final_labels'])}) != merged subjects ({merged.shape[0]}).\n"
            "To do this safely, provide:\n"
            "  final_metrics['final_labels_by_subject_id'] = {src_subject_id: label, ...}\n"
            "so labels can be aligned after the merge."
        )
    merged_clusters = np.asarray(final_metrics["final_labels"])

# 3) Global PCA (PC1)
X_global = merged.drop(columns=["src_subject_id"]).values
Xg_z = StandardScaler().fit_transform(X_global)

pca_global = PCA(n_components=1, random_state=0).fit(Xg_z)
pc1_global = pca_global.transform(Xg_z)[:, 0]
evr1_global = float(pca_global.explained_variance_ratio_[0])
print(f"[GLOBAL PCA] PC1 EVR: {evr1_global:.2%}")

global_df = pd.DataFrame({
    "cluster": merged_clusters,
    "PC1_global": pc1_global
})

global_cluster_order = np.sort(global_df["cluster"].unique())

# 4) Plot global PC1 difference by integrated cluster
fig2, ax2 = plt.subplots(figsize=(10.5, 6.5))

sns.violinplot(
    data=global_df,
    x="cluster",
    y="PC1_global",
    order=global_cluster_order,
    palette=sns.color_palette(n_colors=len(global_cluster_order)),
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    ax=ax2
)

sns.stripplot(
    data=global_df,
    x="cluster",
    y="PC1_global",
    order=global_cluster_order,
    color="black",
    size=3,
    jitter=0.22,
    alpha=0.30,
    ax=ax2
)

# Median markers
med = global_df.groupby("cluster")["PC1_global"].median()
xpos = np.arange(len(global_cluster_order))
ax2.scatter(
    xpos,
    [med.loc[c] for c in global_cluster_order],
    s=180,
    marker="_",
    linewidths=3
)

# n labels
counts = global_df["cluster"].value_counts()
ymin, ymax = ax2.get_ylim()
y_annot = ymin + 0.04 * (ymax - ymin)
#for i, c in enumerate(global_cluster_order):
#    ax2.text(i, y_annot, f"n={int(counts.loc[c])}", ha="center", va="top", fontsize=16)

ax2.set_xlabel("Integrated cluster", fontsize=20)
ax2.set_ylabel("Global PC1 score (all modalities/features)", fontsize=20)
ax2.set_title(f"Global PC1 Across All Features and Modalities", fontsize=20, pad=14)
ax2.grid(axis="y", alpha=0.15)
ax2.set_xticklabels(ax2.get_xticklabels(), fontsize=16)
ax2.set_yticklabels(ax2.get_yticklabels(), fontsize=16)
sns.despine(ax=ax2)

fig2.tight_layout()
fig2.savefig(os.path.join(out_dir, "GLOBAL_all_modalities_PC1_violin.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ==========================================================
# GLOBAL PCA (ALL modalities merged) -> PC1 by final clusters
# - Merges modalities on src_subject_id (inner-join by default)
# - Standardizes all features
# - PCA -> PC1
# - Publication-ready violin (quartiles) + jitter + median + n
# - Also saves a variance plot (top 5 PCs) for the merged space
# ==========================================================

global_out_dir = os.path.join(plots_dir, "global_pca_all_modalities/")
os.makedirs(global_out_dir, exist_ok=True)

# ---- 1) Merge all modalities into one wide dataframe ----
dfs = []
for modality, df in final_metrics["data"].items():
    tmp = df.copy()

    # Prefix feature names with modality to avoid collisions
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})

    dfs.append(tmp)

# Inner join across modalities by subject id (keeps subjects present in ALL modalities)
merged = dfs[0]
for d in dfs[1:]:
    merged = merged.merge(d, on="src_subject_id", how="inner")

print(f"[GLOBAL PCA] Subjects after inner-join across modalities: {merged.shape[0]}")
print(f"[GLOBAL PCA] Total merged features: {merged.shape[1] - 1}")

# ---- 2) Align final labels to the merged subject IDs ----
clusters = np.asarray(final_metrics["final_labels"])

# If your final_labels are already in the same row-order as each modality df,
# then this should match the *intersection* order only if you kept the same subjects.
# Safest is to align by subject ID if you have a mapping.
#
# Try to auto-detect a mapping dict if provided:
label_map = final_metrics.get("final_labels_by_subject_id", None)

if label_map is not None:
    # label_map should be {src_subject_id: label}
    merged_clusters = merged["src_subject_id"].map(label_map).to_numpy()
    if np.any(pd.isna(merged_clusters)):
        raise ValueError("Some merged subjects are missing labels in final_labels_by_subject_id.")
else:
    # Fallback: assume labels are already aligned to the rows of EACH modality df.
    # This is ONLY safe if all modality dfs share the same subject order and
    # inner-joining didn't change it. If you see mismatches, create label_map above.
    if len(clusters) != merged.shape[0]:
        raise ValueError(
            f"final_labels length ({len(clusters)}) != merged subjects ({merged.shape[0]}). "
            "Provide final_metrics['final_labels_by_subject_id'] to align labels safely."
        )
    merged_clusters = clusters

# ---- 3) PCA on all features ----
X = merged.drop(columns=["src_subject_id"]).values

# Standardize before PCA
Xz = StandardScaler().fit_transform(X)

# Fit PCA (enough components to report variance for first 5, but at least 2 if possible)
n_pcs = min(5, Xz.shape[1])
pca = PCA(n_components=n_pcs, random_state=0)
scores = pca.fit_transform(Xz)

pc1 = scores[:, 0]
evr = pca.explained_variance_ratio_
cum_evr = np.cumsum(evr)

print(f"[GLOBAL PCA] PC1 EVR: {evr[0]:.2%}")

# ---- 4) Plot PC1 by cluster (global / all modalities) ----
plot_df = pd.DataFrame({"cluster": merged_clusters, "PC1": pc1})
cluster_order = np.sort(pd.unique(plot_df["cluster"]))

fig, ax = plt.subplots(figsize=(10.5, 6.5))

sns.violinplot(
    data=plot_df,
    x="cluster",
    y="PC1",
    order=cluster_order,
    palette=sns.color_palette(n_colors=len(cluster_order)),
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    ax=ax
)

sns.stripplot(
    data=plot_df,
    x="cluster",
    y="PC1",
    order=cluster_order,
    color="black",
    size=3,
    jitter=0.22,
    alpha=0.35,
    ax=ax
)

# Median marker per cluster
medians = plot_df.groupby("cluster")["PC1"].median()
x_positions = np.arange(len(cluster_order))
ax.scatter(
    x=x_positions,
    y=[medians.loc[c] for c in cluster_order],
    s=180,
    marker="_",
    linewidths=3
)

# Annotate n per cluster near bottom
counts = plot_df["cluster"].value_counts()
ymin, ymax = ax.get_ylim()
y_annot = ymin + 0.04 * (ymax - ymin)
for i, c in enumerate(cluster_order):
    ax.text(i, y_annot, f"n={int(counts.loc[c])}", ha="center", va="bottom", fontsize=12)

ax.set_xlabel("Cluster", fontsize=16)
ax.set_ylabel("PCA Component 1 score", fontsize=16)
ax.grid(axis="y", alpha=0.15)
sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(os.path.join(global_out_dir, "GLOBAL_all_modalities_PC1_violin.png"), dpi=300, bbox_inches="tight")
plt.show()

# ---- 5) Variance explained plot (Top PCs) ----
figv, axv = plt.subplots(figsize=(10, 5.5))
pcs_idx = np.arange(1, n_pcs + 1)

axv.bar(pcs_idx, evr)
axv.plot(pcs_idx, cum_evr, marker="o")

axv.set_xticks(pcs_idx)
axv.set_xlabel("Principal Component")
axv.set_ylabel("Explained variance ratio")
axv.set_title(f"GLOBAL (All Modalities) — PCA Variance Explained (Top {n_pcs} PCs)", pad=12)

axv.grid(axis="y", alpha=0.15)
sns.despine(ax=axv)

figv.tight_layout()
figv.savefig(os.path.join(global_out_dir, f"GLOBAL_all_modalities_variance_top{n_pcs}.png"),
             dpi=300, bbox_inches="tight")
plt.show()


### Add CC to PCA integrated results plots

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ==========================================================
# 1) SINGLE PLOT: PC1 (per-modality PCA) distributions by modality
#    - CHR integrated clusters + CC group
# 2) GLOBAL PCA across ALL modalities/features
#    - CHR integrated clusters + CC group
# ==========================================================

out_dir = os.path.join(plots_dir, "merged_feature_pca_chr_vs_cc/")
os.makedirs(out_dir, exist_ok=True)

# ---------- custom colors ----------
cluster_colors = ["#327D6D", "#7FE3CD", "#F2B134", "#B65FCF", "#E76F51", "#8DA0CB"]
CC_COLOR = "#065982ff"

# --------------------------
# A) Per-modality PC1 plot: CHR clusters + CC
# --------------------------
rows = []
final_labels_chr = np.asarray(final_metrics["final_labels"]).astype(str)

for modality, df_chr in final_metrics["data"].items():
    # CHR
    X_chr_df = df_chr.drop(columns=["src_subject_id"]).copy()
    if len(final_labels_chr) != len(X_chr_df):
        raise ValueError(f"{modality}: final_labels length != CHR rows. Use final_labels_by_subject_id alignment if needed.")
    grp_chr = final_labels_chr

    # CC (already transformed via apply_preprocessing_to_new_data)
    df_cc = dict_final_cc[modality]
    X_cc_df = df_cc.drop(columns=["src_subject_id"]).copy()
    X_cc_df = X_cc_df.reindex(columns=X_chr_df.columns)  # strict CHR feature order

    # Fit scaler+PCA on CHR only
    scaler = StandardScaler()
    X_chr_z = scaler.fit_transform(X_chr_df.values)
    X_cc_z = scaler.transform(X_cc_df.values)

    pca = PCA(n_components=1, random_state=0).fit(X_chr_z)
    pc1_chr = pca.transform(X_chr_z)[:, 0]
    pc1_cc = pca.transform(X_cc_z)[:, 0]
    evr1 = float(pca.explained_variance_ratio_[0])

    print(f"{modality}: CHR-fitted PC1 EVR={evr1:.2%}")

    tmp_chr = pd.DataFrame({
        "modality": modality,
        "group": grp_chr,
        "PC1": pc1_chr,
        "cohort": "CHR"
    })
    tmp_cc = pd.DataFrame({
        "modality": modality,
        "group": "CC",
        "PC1": pc1_cc,
        "cohort": "CC"
    })
    tmp = pd.concat([tmp_chr, tmp_cc], ignore_index=True)
    tmp["PC1_EVR_chrfit"] = evr1
    rows.append(tmp)

plot_df = pd.concat(rows, ignore_index=True)

modality_order = list(final_metrics["data"].keys())
chr_cluster_order = sorted(pd.unique(final_labels_chr), key=lambda x: int(x) if str(x).isdigit() else str(x))
group_order = chr_cluster_order + ["CC"]

group_palette = {g: cluster_colors[i % len(cluster_colors)] for i, g in enumerate(chr_cluster_order)}
group_palette["CC"] = CC_COLOR

fig_w = max(18, 2.2 * len(modality_order))
fig_h = 6
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

sns.violinplot(
    data=plot_df,
    x="modality",
    y="PC1",
    hue="group",
    order=modality_order,
    hue_order=group_order,
    palette=group_palette,
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    dodge=True,
    width=0.8,
    ax=ax
)

ax.set_xlim(-0.55, len(modality_order) - 0.45)

# --- manual black points centered inside each dodged violin ---
rng = np.random.default_rng(0)

n_hue = len(group_order)
violin_width = 0.8                      # must match sns.violinplot(width=0.65)
sub_width = violin_width / n_hue         # width allotted to each group within a modality
jitter_scale = sub_width * 0.28          # small jitter within each subgroup

for i, modality in enumerate(modality_order):
    for j, group in enumerate(group_order):
        vals = plot_df.loc[
            (plot_df["modality"] == modality) & (plot_df["group"] == group),
            "PC1"
        ].to_numpy()

        if len(vals) == 0:
            continue

        # exact center of this group's violin within this modality
        center = i - violin_width / 2 + (j + 0.5) * sub_width

        # small symmetric jitter around that center
        x = center + rng.uniform(-jitter_scale, jitter_scale, size=len(vals))

        ax.scatter(
            x,
            vals,
            color="black",
            s=10,
            alpha=0.22,
            zorder=3,
            linewidths=0
        )

# Legend cleanup (keep one)
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:len(group_order)],
    labels[:len(group_order)],
    title="Group",
    frameon=False,
    loc="upper right"
)

for x in np.arange(0.5, len(modality_order), 1.0):
    ax.axvline(x, linewidth=0.8, alpha=0.25)

ax.set_xlabel("Modality", fontsize=16)
ax.set_ylabel("PC1 score (CHR-fitted PCA)", fontsize=16)
ax.set_title("PC1 Distributions by Modality (CHR integrated clusters + CC)", fontsize=18, pad=14)
ax.tick_params(axis="both", labelsize=14)
ax.grid(axis="y", alpha=0.15)
sns.despine(ax=ax)

plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=13)
plt.setp(ax.get_yticklabels(), fontsize=13)

fig.tight_layout()
fig.savefig(os.path.join(out_dir, "ALL_modalities_PC1_singleplot_violin_CHR_vs_CC.svg"), dpi=300, bbox_inches="tight")
plt.show()

# --------------------------
# B) GLOBAL PCA PC1 across ALL modalities/features: CHR clusters + CC
# --------------------------

# CHR merged wide
dfs_chr = []
for modality, df in final_metrics["data"].items():
    tmp = df.copy()
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})
    dfs_chr.append(tmp)

merged_chr = dfs_chr[0]
for d in dfs_chr[1:]:
    merged_chr = merged_chr.merge(d, on="src_subject_id", how="inner")

print(f"\n[GLOBAL PCA] CHR subjects after inner-join: {merged_chr.shape[0]}")
print(f"[GLOBAL PCA] CHR merged features: {merged_chr.shape[1] - 1}")

# CC merged wide
dfs_cc = []
for modality, df in dict_final_cc.items():
    tmp = df.copy()
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})
    dfs_cc.append(tmp)

merged_cc = dfs_cc[0]
for d in dfs_cc[1:]:
    merged_cc = merged_cc.merge(d, on="src_subject_id", how="inner")

print(f"[GLOBAL PCA] CC subjects after inner-join: {merged_cc.shape[0]}")
print(f"[GLOBAL PCA] CC merged features: {merged_cc.shape[1] - 1}")

# Align CHR labels to merged CHR IDs
label_map = final_metrics.get("final_labels_by_subject_id", None)
if label_map is not None:
    chr_clusters = merged_chr["src_subject_id"].map(label_map).astype(str).to_numpy()
    if np.any(pd.isna(chr_clusters)):
        missing = merged_chr.loc[pd.isna(chr_clusters), "src_subject_id"].head(5).tolist()
        raise ValueError(f"Missing labels for merged CHR IDs. Examples: {missing}")
else:
    if len(final_labels_chr) != merged_chr.shape[0]:
        raise ValueError(
            f"final_labels length ({len(final_labels_chr)}) != merged CHR subjects ({merged_chr.shape[0]}). "
            "Provide final_labels_by_subject_id for safe alignment."
        )
    chr_clusters = final_labels_chr

# Strict CC feature alignment to CHR merged features
chr_feature_cols = [c for c in merged_chr.columns if c != "src_subject_id"]
cc_feature_cols = [c for c in merged_cc.columns if c != "src_subject_id"]
missing_in_cc = [c for c in chr_feature_cols if c not in cc_feature_cols]
if missing_in_cc:
    raise ValueError(f"CC missing {len(missing_in_cc)} global features. Example: {missing_in_cc[:10]}")

X_chr_global = merged_chr[chr_feature_cols].values
X_cc_global = merged_cc.reindex(columns=chr_feature_cols).values

# CHR-fitted global PCA
Xg_scaler = StandardScaler()
Xg_chr_z = Xg_scaler.fit_transform(X_chr_global)
Xg_cc_z = Xg_scaler.transform(X_cc_global)

pca_global = PCA(n_components=1, random_state=0).fit(Xg_chr_z)
pc1_chr_global = pca_global.transform(Xg_chr_z)[:, 0]
pc1_cc_global = pca_global.transform(Xg_cc_z)[:, 0]
evr1_global = float(pca_global.explained_variance_ratio_[0])
print(f"[GLOBAL PCA] CHR-fitted PC1 EVR: {evr1_global:.2%}")

global_df = pd.concat([
    pd.DataFrame({"group": chr_clusters, "PC1_global": pc1_chr_global, "cohort": "CHR"}),
    pd.DataFrame({"group": "CC", "PC1_global": pc1_cc_global, "cohort": "CC"})
], ignore_index=True)

global_group_order = chr_cluster_order + ["CC"]

fig2, ax2 = plt.subplots(figsize=(10.5, 6.5))

sns.violinplot(
    data=global_df,
    x="group",
    y="PC1_global",
    hue="group",
    order=global_group_order,
    hue_order=global_group_order,
    palette=group_palette,
    dodge=False,
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    legend=False,
    ax=ax2
)

sns.stripplot(
    data=global_df,
    x="group",
    y="PC1_global",
    order=global_group_order,
    color="black",
    size=3,
    jitter=0.22,
    alpha=0.30,
    ax=ax2
)

med = global_df.groupby("group")["PC1_global"].median()
xpos = np.arange(len(global_group_order))
ax2.scatter(xpos, [med.loc[g] for g in global_group_order], s=180, marker="_", linewidths=3, color="black", zorder=4)

counts = global_df["group"].value_counts()
ymin, ymax = ax2.get_ylim()
y_annot = ymin + 0.04 * (ymax - ymin)
for i, g in enumerate(global_group_order):
    ax2.text(i, y_annot, f"n={int(counts.get(g, 0))}", ha="center", va="top", fontsize=16)

ax2.set_xlabel("Group", fontsize=20)
ax2.set_ylabel("Global PC1 score (CHR-fitted)", fontsize=20)
ax2.set_title("Global PC1 Across All Features and Modalities (CHR clusters + CC)", fontsize=20, pad=14)
ax2.grid(axis="y", alpha=0.15)
ax2.tick_params(axis="x", labelsize=16)
ax2.tick_params(axis="y", labelsize=16)
sns.despine(ax=ax2)

fig2.tight_layout()
fig2.savefig(os.path.join(out_dir, "GLOBAL_all_modalities_PC1_violin_CHR_vs_CC.svg"), dpi=300, bbox_inches="tight")
plt.show()


### Differences categorical with individual labels

In [ ]:
def add_metadata_and_clusters(final_metrics, data_full, mod_num):
    """
    Merge cluster labels into full metadata DataFrame using src_subject_id
    from the actual training data stored in fold_metrics['data'].
    This ensures correct alignment even when not all train_ids were clustered.
    """
    clusters = pd.Series(final_metrics['individual_labels'][mod_num])

    # Get the first modality’s dataframe — all have the same subject order
    modality_dfs = final_metrics.get('data', {})
    if not modality_dfs:
        raise ValueError("final_metrics['data'] is empty; cannot extract subject IDs.")
    
    # Use the src_subject_id column from the first available modality
    first_modality = list(modality_dfs.keys())[0]
    subj_ids = modality_dfs[first_modality]['src_subject_id'].reset_index(drop=True)
    
    # Sanity check: should match cluster array length
    if len(subj_ids) != len(clusters):
        print(f"⚠️ Mismatch: {len(subj_ids)} subject IDs vs {len(clusters)} cluster labels.")
        min_len = min(len(subj_ids), len(clusters))
        subj_ids = subj_ids.iloc[:min_len]
        clusters = clusters.iloc[:min_len]
        print(f"Trimmed both to {min_len} entries to align.")

    # Build cluster mapping dataframe
    cluster_df = pd.DataFrame({
        'src_subject_id': subj_ids,
        'Cluster': clusters
    })

    # Merge back into full metadata
    merged = pd.merge(data_full, cluster_df, on='src_subject_id', how='left')

    print(f"✅ Merged clusters for {merged['Cluster'].notna().sum()} subjects (out of {len(merged)} total).")
    return merged

def chi_square_comparison(df, group_col, label_col, title_prefix):
    """
    Perform chi-square test and plot grouped bar chart.
    """
    # Drop missing values
    df = df.dropna(subset=[group_col, label_col])

    # Make a copy for plotting / stats so we can safely relabel
    df_plot = df.copy()

    # Rename 'HC' -> 'CC' only for phenotype (for plotting and stats)
    if label_col == 'phenotype':
        df_plot[label_col] = df_plot[label_col].replace({'HC': 'CC'})

    # 1. Summarize counts
    summary_df = (
        df_plot.groupby([group_col, label_col])
        .size()
        .reset_index(name='n')
    )

    # 2. Chi-square test
    tbl = pd.crosstab(df_plot[group_col], df_plot[label_col])
    chi2, pval, dof, expected = chi2_contingency(tbl)
    pval = round(pval, 3)

    # 3. Grouped bar chart
    plt.figure(figsize=(8, 6))
    sns.barplot(
        data=summary_df,
        x=group_col,
        y='n',
        hue=label_col,
        dodge=True
    )
    plt.title(f"{title_prefix}\n(Chi-square p = {pval})")
    plt.xlabel("Cluster")
    plt.ylabel("Count")
    sns.despine()
    plt.tight_layout()
    plt.show()

mod_num = 0
for modality in final_metrics['data'].keys():
    print(f"\n=== Analyzing categorical differences for modality: {modality} ===")

    # Merge cluster labels into full data
    df = add_metadata_and_clusters(final_metrics, discovery_data, mod_num)

    # Compare by site
    if 'Site' in df.columns:
        chi_square_comparison(
            df=df,
            group_col='Cluster',
            label_col='Site',
            title_prefix=f"Comparison of Site Distribution per Subgroup",
        )

    # Optional: extend for other categorical variables
    for col in ['sips_bips_scr_lifetime', 'sips_aps_scr_lifetime', 'sips_grd_scr_lifetime']:
        if col in df.columns:
            chi_square_comparison(
                df=df,
                group_col='Cluster',
                label_col=col,
                title_prefix=f"Comparison of {col} per Subgroup",
            )
    mod_num = mod_num + 1




### Differences categorical with final labels

In [ ]:
## Create directory for categorical differences plots
os.makedirs(os.path.join(plots_dir, "merged_cat_diff/"), exist_ok=True)

def add_metadata_and_clusters(final_metrics, data_full):
    """
    Merge cluster labels into full metadata DataFrame using src_subject_id
    from the actual training data stored in fold_metrics['data'].
    This ensures correct alignment even when not all train_ids were clustered.
    """
    clusters = pd.Series(final_metrics['final_labels'])

    # Get the first modality’s dataframe — all have the same subject order
    modality_dfs = final_metrics.get('data', {})
    if not modality_dfs:
        raise ValueError("final_metrics['data'] is empty; cannot extract subject IDs.")
    
    # Use the src_subject_id column from the first available modality
    first_modality = list(modality_dfs.keys())[0]
    subj_ids = modality_dfs[first_modality]['src_subject_id'].reset_index(drop=True)
    
    # Sanity check: should match cluster array length
    if len(subj_ids) != len(clusters):
        print(f"⚠️ Mismatch: {len(subj_ids)} subject IDs vs {len(clusters)} cluster labels.")
        min_len = min(len(subj_ids), len(clusters))
        subj_ids = subj_ids.iloc[:min_len]
        clusters = clusters.iloc[:min_len]
        print(f"Trimmed both to {min_len} entries to align.")

    # Build cluster mapping dataframe
    cluster_df = pd.DataFrame({
        'src_subject_id': subj_ids,
        'Cluster': clusters
    })

    # Merge back into full metadata
    merged = pd.merge(data_full, cluster_df, on='src_subject_id', how='left')

    print(f"✅ Merged clusters for {merged['Cluster'].notna().sum()} subjects (out of {len(merged)} total).")
    return merged


def chi_square_comparison(df, group_col, label_col, title_prefix, save_path):
    """
    Perform chi-square test and plot grouped bar chart.
    """
    # Drop missing values
    df = df.dropna(subset=[group_col, label_col])

    # Make a copy for plotting / stats so we can safely relabel
    df_plot = df.copy()

    # Rename 'HC' -> 'CC' only for phenotype (for plotting and stats)
    if label_col == 'phenotype':
        df_plot[label_col] = df_plot[label_col].replace({'HC': 'CC'})

    # 1. Summarize counts
    summary_df = (
        df_plot.groupby([group_col, label_col])
        .size()
        .reset_index(name='n')
    )

    # 2. Chi-square test
    tbl = pd.crosstab(df_plot[group_col], df_plot[label_col])
    chi2, pval, dof, expected = chi2_contingency(tbl)
    pval = round(pval, 3)

    # 3. Grouped bar chart
    plt.figure(figsize=(8, 6))
    sns.barplot(
        data=summary_df,
        x=group_col,
        y='n',
        hue=label_col,
        dodge=True
    )
    plt.title(f"{title_prefix}\n(Chi-square p = {pval})")
    plt.xlabel("Cluster")
    plt.ylabel("Count")
    sns.despine()
    plt.tight_layout()
    plt.savefig(save_path, dpi=1000)
    plt.show()

    # 4. Cluster summary (including CHR percentage if relevant)
    cluster_summary = (
        df_plot.groupby(group_col)
        .agg(
            Size=(group_col, 'size'),
            CHR_percentage=(
                label_col,
                lambda x: (x.eq('CHR').sum() / len(x)) * 100
                if 'CHR' in x.values else None
            )
        )
        .reset_index()
        .sort_values(group_col)
    )
    print(f"\nCluster summary for {title_prefix}")
    print(cluster_summary)
    print("\n" + "-"*60)


# Merge cluster labels into full data
df = add_metadata_and_clusters(final_metrics, discovery_data)


# Compare by site
if 'Site' in df.columns:
    chi_square_comparison(
        df=df,
        group_col='Cluster',
        label_col='Site',
        title_prefix=f"Comparison of Site Distribution per Subgroup",
        save_path=os.path.join(plots_dir,'merged_cat_diff/', f"merged_Site_by_subgroup.png")
    )

# Optional: extend for other categorical variables
for col in ['sips_bips_scr_lifetime', 'sips_aps_scr_lifetime', 'sips_grd_scr_lifetime']:
    if col in df.columns:
        chi_square_comparison(
            df=df,
            group_col='Cluster',
            label_col=col,
            title_prefix=f"Comparison of {col} per Subgroup",
            save_path=os.path.join(plots_dir,'merged_cat_diff/', f"merged_{col}_by_subgroup.png")
        )





## Overlap between modalities (in labels)

In [ ]:
# Rename labels for clarity
# The cluster with generally high scores should be called 'high' and the cluster with generally low scores should be called 'low' for all modalities

new_labels_list = []
modality_names = list(final_metrics["data"].keys())
for i, modality in enumerate(modality_names):
    print(f"Processing modality: {modality}")

    labels = final_metrics['individual_labels'][i]
    df = final_metrics['data'][modality]

    # Skip for modality if num_cluster<2
    if len(np.unique(labels)) < 2:
        continue
    
    # Calculate mean score per cluster
    cluster_means = {}
    for cluster in np.unique(labels):
        cluster_means[cluster] = df.drop(columns=['src_subject_id']).mean().mean()

    # Determine which cluster is 'high' and which is 'low'
    sorted_clusters = sorted(cluster_means, key=cluster_means.get, reverse=True)
    high_cluster = sorted_clusters[0]
    low_cluster = sorted_clusters[1]
    
    if modality == "Functioning" or modality == "Cognition":
        # For Functioning, reverse the assignment
        new_labels = ['low_severity' if lbl == high_cluster else 'high_severity' for lbl in labels]
    else:
        # Create new labels
        new_labels = ['high_severity' if lbl == high_cluster else 'low_severity' for lbl in labels]

    new_labels_list.append(new_labels)


In [ ]:
# pip install upsetplot pandas matplotlib
from collections import Counter
import pandas as pd
from upsetplot import UpSet

modality_names = list(final_metrics["data"].keys())

labels = new_labels_list
M = len(labels)
N = len(labels[0])
assert all(len(x) == N for x in labels)

# majority label per sample
majority = [Counter(labels[m][k] for m in range(M)).most_common(1)[0][0] for k in range(N)]

# boolean matrix: modality agrees with majority?
agree = pd.DataFrame({f"{modality_names[m]}"    : [labels[m][k] == majority[k] for k in range(N)] for m in range(M)})

# UpSet expects a MultiIndex of booleans
counts = agree.value_counts()
UpSet(counts, show_counts=True, sort_by="cardinality").plot()


In [ ]:
import pandas as pd

labels = new_labels_list
M = len(labels)
N = len(labels[0])

all_agree = sum(len({labels[m][k] for m in range(M)}) == 1 for k in range(N))
print("All-4 agree:", all_agree, "/", N, "=", all_agree / N)

# 1 means full agreement, 2 means split (high vs low), since only two labels exist
unique_per_sample = [len({labels[m][k] for m in range(M)}) for k in range(N)]
print("Unique-labels-per-sample distribution:")
print(pd.Series(unique_per_sample).value_counts().sort_index())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

labels = new_labels_list
modality_names = list(final_metrics["data"].keys())

names = modality_names
N = len(labels[0])

mat = np.zeros((len(labels), len(labels)), dtype=float)
for i in range(len(labels)):
    for j in range(len(labels)):
        mat[i, j] = sum(labels[i][k] == labels[j][k] for k in range(N)) / N

df = pd.DataFrame(mat, index=names, columns=names)
print(df)

plt.figure()
plt.imshow(df.values)
plt.xticks(range(len(names)), names)
plt.yticks(range(len(names)), names)
plt.title("Pairwise agreement rate")
plt.colorbar()
plt.show()


In [ ]:

final = final_metrics["final_labels"]
mods = new_labels_list  # list of 4 lists, length 665 each


assert all(len(m) == len(final) for m in mods)

for name, m in zip(names, mods):
    ct = pd.crosstab(
        pd.Series(m, name=f"{name}_label"),
        pd.Series(final, name="final_label")
    )
    print(f"\n=== {name}: counts (rows=mod label, cols=final label) ===")
    print(ct)

    # Row-normalized: P(final | modality_label)
    ct_row = ct.div(ct.sum(axis=1), axis=0).fillna(0)
    print(f"\n=== {name}: row-normalized (P(final | mod label)) ===")
    print(ct_row.round(3))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

final = pd.Series(final_metrics["final_labels"], name="final")
mods = new_labels_list

rows = []
for name, m in zip(names, mods):
    m = pd.Series(m, name="mod")
    ct = pd.crosstab(m, final, normalize="index").fillna(0)  # P(final | mod_label)
    # make rows like "mod0:0", "mod0:1"
    for mod_label in ct.index:
        rowname = f"{name}:{mod_label}"
        rows.append(pd.Series(ct.loc[mod_label], name=rowname))

mat = pd.DataFrame(rows)  # rows=modality:label, cols=final labels

plt.figure(figsize=(6, 6))
plt.imshow(mat.values)
plt.xticks(range(mat.shape[1]), mat.columns)
plt.yticks(range(mat.shape[0]), mat.index)
plt.title("P(final label | modality label)")
plt.xlabel("final label")
plt.colorbar()
plt.tight_layout()
plt.show()


In [ ]:

final = pd.Series(final_metrics["final_labels"], name="final")
mods = [pd.Series(m, name=f"mod{i}") for i, m in enumerate(new_labels_list)]
df = pd.concat([final] + mods, axis=1)

mod_names = names

for col in df.columns[1:]:
    # distribution of modality labels within each final label
    dist = pd.crosstab(df["final"], df[col], normalize="index")
    name = mod_names[df.columns.get_loc(col)-1]
    print(f"\n=== {name}: distribution of {col} within each final label (rows sum to 1) ===")
    print(dist.round(3))


In [ ]:
new_labels_by_modality = {}
modality_names = list(final_metrics["data"].keys())

for i, modality in enumerate(modality_names):
    labels = final_metrics['individual_labels'][i]
    df = final_metrics['data'][modality]

    if len(np.unique(labels)) < 2:
        continue

    cluster_means = {}
    for cluster in np.unique(labels):
        cluster_means[cluster] = df.drop(columns=['src_subject_id']).mean().mean()

    sorted_clusters = sorted(cluster_means, key=cluster_means.get, reverse=True)
    high_cluster = sorted_clusters[0]
    low_cluster = sorted_clusters[1]

    if modality in ("Functioning", "Cognition"):
        new_labels = ['low_severity' if lbl == high_cluster else 'high_severity' for lbl in labels]
    else:
        new_labels = ['high_severity' if lbl == high_cluster else 'low_severity' for lbl in labels]

    new_labels_by_modality[modality] = new_labels


In [ ]:
import pandas as pd

stage_order = ["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition"]

# Pick subject_id vector from one modality (must share row order with labels)
subject_ids = final_metrics["data"][stage_order[0]]["src_subject_id"].astype(str).reset_index(drop=True)

# Build the path dataframe
df_paths = pd.DataFrame({"src_subject_id": subject_ids})
for stage in stage_order:
    df_paths[stage] = pd.Series(new_labels_by_modality[stage]).astype(str).reset_index(drop=True)

df_paths["final"] = pd.Series(final_metrics["final_labels"]).astype(str).reset_index(drop=True)

# Sanity checks
N = len(df_paths)
assert all(len(new_labels_by_modality[s]) == N for s in stage_order), "Label lengths don't match subject_ids length"
assert len(final_metrics["final_labels"]) == N, "Final labels length doesn't match subject_ids length"

df_paths.head()


In [ ]:
def summarize_streams(df_paths, stage_order, top_k=20, sample_ids=10):
    group_cols = stage_order + ["final"]
    total = len(df_paths)

    g = (df_paths
         .groupby(group_cols, dropna=False)
         .agg(
             n=("src_subject_id", "size"),
             example_ids=("src_subject_id", lambda x: list(x.head(sample_ids))),
         )
         .reset_index()
        )

    g["pct"] = (g["n"] / total * 100).round(2)

    # A readable "stream label"
    def make_stream_label(row):
        parts = [f"{c}={row[c]}" for c in stage_order] + [f"final={row['final']}"]
        return " → ".join(parts)

    g["stream"] = g.apply(make_stream_label, axis=1)

    g = g.sort_values("n", ascending=False).head(top_k)

    # Reorder columns nicely
    cols = ["n", "pct", "stream", "example_ids"] + stage_order + ["final"]
    return g[cols]

stream_summary = summarize_streams(df_paths, stage_order, top_k=100, sample_ids=12)
stream_summary


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

stages = ["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition"]
df = df_paths.copy()

fig, axes = plt.subplots(1, len(stages), figsize=(4*len(stages), 4), sharey=True)
for ax, s in zip(axes, stages):
    ct = pd.crosstab(df[s], df["final"])
    ct.plot(kind="bar", ax=ax)
    ax.set_title(s)
    ax.set_xlabel("")
    ax.legend(title="final")
plt.tight_layout()
plt.show()


In [ ]:
# General k-safe parallel-categories helper.
# The implementation lives in Utils.py so the same behavior is used across notebooks and reports.
from Utils import domain_map

In [ ]:
stage_order = ["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition"]

domain_map(
    new_labels_by_modality=new_labels_by_modality,
    final_labels=final_metrics["final_labels"],
    stage_order=["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition"],
    final_name="final",
    top_token="high_severity",          # now HIGH is on top
    bottom_token="low_severity",
    invert_final=False,
    color_for_top_final="#005341",      # top-like final ribbons
    color_for_bottom_final="#B3D9D5",   # bottom-like final ribbons
    add_gap_in_final=True,
    gap_weight=20,
    plots_dir=plots_dir,
    save_file_name = "Parcats_by_final.pdf"
)


## SVM results

### Final labels

#### Accuracy

In [ ]:
for metric in final_metrics['svm_results']['mean_metrics']:
    print(f"SVM Mean {metric}: {final_metrics['svm_results']['mean_metrics'][metric]}")


#### Uncertainty

In [ ]:
final_metrics['svm_results']['oof_uncertainty']


In [ ]:
# Find most confident mistakes 
df = final_metrics['svm_results']['oof_uncertainty'] 

df_bad = df[(df["y_true"] != df["y_pred"]) & (df["confidence"] > 0.9)]
df_bad.sort_values("confidence", ascending=False).head(20)


In [ ]:
# Find most uncertain predictions
df_uncertain = df.sort_values(["confidence", "entropy"], ascending=[True, False])
df_uncertain.head(20)


In [ ]:
correct = df["y_true"] == df["y_pred"]

plt.figure()
plt.hist(df.loc[correct, "confidence"].dropna(), bins=30, alpha=0.7, label="correct")
plt.hist(df.loc[~correct, "confidence"].dropna(), bins=30, alpha=0.7, label="wrong")
plt.xlabel("Confidence (max predicted probability)")
plt.ylabel("Count")
plt.title("Confidence distribution: correct vs wrong")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
df2 = df.dropna(subset=["confidence"]).copy()
bins = np.linspace(0, 1, 11)

df2["bin"] = pd.cut(df2["confidence"], bins=bins, include_lowest=True)
grp = df2.groupby("bin", observed=True)

acc = grp.apply(lambda g: (g["y_true"] == g["y_pred"]).mean())
cnt = grp.size()

# Midpoints only for bins that exist
x = np.array([interval.mid for interval in acc.index])

plt.figure()
plt.plot(x, acc.values, marker="o")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Confidence bin midpoint")
plt.ylabel("Observed accuracy")
plt.title("Accuracy vs confidence (OOF)")
plt.tight_layout()
plt.show()

print(pd.DataFrame({"bin": acc.index.astype(str), "bin_mid": x, "accuracy": acc.values, "n": cnt.values}))


In [ ]:
df2 = df.dropna(subset=["confidence"]).copy()
df2["correct"] = (df2["y_true"] == df2["y_pred"]).astype(int)

thresholds = np.linspace(0, 1, 101)
coverage = []
error_rate = []

for t in thresholds:
    kept = df2[df2["confidence"] >= t]
    if len(kept) == 0:
        coverage.append(0.0)
        error_rate.append(np.nan)
        continue
    coverage.append(len(kept) / len(df2))
    error_rate.append(1 - kept["correct"].mean())

plt.figure()
plt.plot(coverage, error_rate)
plt.xlabel("Coverage (fraction kept)")
plt.ylabel("Error rate among kept samples")
plt.title("Reject option: trade coverage for accuracy")
plt.tight_layout()
plt.show()


#### Variable contribution

In [ ]:

meta_svm = final_metrics['svm_results']['feature_importance_meta']
print(f"Model information: {meta_svm}")

feat_imp_mean = final_metrics['svm_results']['feature_importance_mean']
feat_imp_std  = final_metrics['svm_results']['feature_importance_std']

# If loaded from packed results (dict), convert to Series
if isinstance(feat_imp_mean, dict):
    feat_imp_mean = pd.Series(feat_imp_mean, dtype=float)
if isinstance(feat_imp_std, dict):
    feat_imp_std = pd.Series(feat_imp_std, dtype=float)

# Build tidy table
df_imp = (
    pd.DataFrame({
        "feature": feat_imp_mean.index,
        "importance_mean": feat_imp_mean.values,
        "importance_std": feat_imp_std.reindex(feat_imp_mean.index).values if feat_imp_std is not None else None,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

df_imp.head(20)


In [ ]:
import matplotlib.pyplot as plt

top_n = 25
plot_df = df_imp.head(top_n).iloc[::-1]  # reverse for nice ordering

plt.figure()
plt.barh(plot_df["feature"], plot_df["importance_mean"],
         xerr=plot_df["importance_std"] if "importance_std" in plot_df else None)
plt.xlabel("Feature contribution (mean)")
plt.ylabel("Feature")
plt.title(f"Top {top_n} feature contributions ({meta_svm.get('method','')}, kernel={meta_svm.get('kernel','')})")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure()
plt.hist(df_imp["importance_mean"].values, bins=50)
plt.xlabel("Feature contribution (mean)")
plt.ylabel("Count")
plt.title("Distribution of feature contributions")
plt.tight_layout()
plt.show()


### Individual labels

#### Accuracy

In [ ]:
modality_names = list(final_metrics["data"].keys())
svm_results = final_metrics.get("svm_results_modalities", [])

for mod_name, res in zip(modality_names, svm_results):
    print(f"SVM results for modality: {mod_name}")

    mean_metrics = (res or {}).get("mean_metrics")
    if mean_metrics:
        for metric, value in mean_metrics.items():
            print(f"SVM Mean {metric}: {value}")
    else:
        print("SVM Mean metrics: None")

    print()  # empty line between modalities


#### Uncertainty

In [ ]:
for mod_name, res in zip(modality_names, svm_results):
    print(f"SVM results for modality: {mod_name}")

    oof_uncertainty_mod = (res or {}).get("oof_uncertainty")

    # Find most confident mistakes 
    df = oof_uncertainty_mod

    df_bad = df[(df["y_true"] != df["y_pred"]) & (df["confidence"] > 0.9)]
    df_bad.sort_values("confidence", ascending=False).head(20)
    if df_bad is None or df_bad.empty:
        print(f"No confident mistakes found for modality {mod_name}.")
    else:
        print(df_bad)

    # Find most uncertain predictions
    df_uncertain = df.sort_values(["confidence", "entropy"], ascending=[True, False])
    print(df_uncertain.head(10))

    correct = df["y_true"] == df["y_pred"]

    plt.figure()
    plt.hist(df.loc[correct, "confidence"].dropna(), bins=30, alpha=0.7, label="correct")
    plt.hist(df.loc[~correct, "confidence"].dropna(), bins=30, alpha=0.7, label="wrong")
    plt.xlabel("Confidence (max predicted probability)")
    plt.ylabel("Count")
    plt.title("Confidence distribution: correct vs wrong")
    plt.legend()
    plt.tight_layout()
    plt.show()

    df2 = df.dropna(subset=["confidence"]).copy()
    bins = np.linspace(0, 1, 11)

    df2["bin"] = pd.cut(df2["confidence"], bins=bins, include_lowest=True)
    grp = df2.groupby("bin", observed=True)

    acc = grp.apply(lambda g: (g["y_true"] == g["y_pred"]).mean())
    cnt = grp.size()

    # Midpoints only for bins that exist
    x = np.array([interval.mid for interval in acc.index])

    plt.figure()
    plt.plot(x, acc.values, marker="o")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("Confidence bin midpoint")
    plt.ylabel("Observed accuracy")
    plt.title("Accuracy vs confidence (OOF)")
    plt.tight_layout()
    plt.show()

    print(pd.DataFrame({"bin": acc.index.astype(str), "bin_mid": x, "accuracy": acc.values, "n": cnt.values}))

    df2 = df.dropna(subset=["confidence"]).copy()
    df2["correct"] = (df2["y_true"] == df2["y_pred"]).astype(int)

    thresholds = np.linspace(0, 1, 101)
    coverage = []
    error_rate = []

    for t in thresholds:
        kept = df2[df2["confidence"] >= t]
        if len(kept) == 0:
            coverage.append(0.0)
            error_rate.append(np.nan)
            continue
        coverage.append(len(kept) / len(df2))
        error_rate.append(1 - kept["correct"].mean())

    plt.figure()
    plt.plot(coverage, error_rate)
    plt.xlabel("Coverage (fraction kept)")
    plt.ylabel("Error rate among kept samples")
    plt.title("Reject option: trade coverage for accuracy")
    plt.tight_layout()
    plt.show()




#### Variable contribution

In [ ]:
for mod_name, res in zip(modality_names, svm_results):
    print(f"SVM results for modality: {mod_name}")

    oof_uncertainty_mod = (res or {}).get("oof_uncertainty")
    feat_imp_mean = (res or {}).get("feature_importance_mean")
    feat_imp_std  = (res or {}).get("feature_importance_std")
    meta_svm = (res or {}).get("feature_importance_meta")       

    print(f"Model information: {meta_svm}")


    # If loaded from packed results (dict), convert to Series
    if isinstance(feat_imp_mean, dict):
        feat_imp_mean = pd.Series(feat_imp_mean, dtype=float)
    if isinstance(feat_imp_std, dict):
        feat_imp_std = pd.Series(feat_imp_std, dtype=float)

    # Build tidy table
    df_imp = (
        pd.DataFrame({
            "feature": feat_imp_mean.index,
            "importance_mean": feat_imp_mean.values,
            "importance_std": feat_imp_std.reindex(feat_imp_mean.index).values if feat_imp_std is not None else None,
        })
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True)
    )

    top_n = 25
    plot_df = df_imp.head(top_n).iloc[::-1]  # reverse for nice ordering

    plt.figure()
    plt.barh(plot_df["feature"], plot_df["importance_mean"],
            xerr=plot_df["importance_std"] if "importance_std" in plot_df else None)
    plt.xlabel(f"Feature contribution (mean)")
    plt.ylabel("Feature")
    plt.title(f"Top feature contributions ({meta_svm.get('method','')}, kernel={meta_svm.get('kernel','')})")
    plt.tight_layout()
    plt.show()

    


# Apply to test set

## Prepare data

In [ ]:
# Only keep relevant modalities
modalities_keep = ["Psychoticism", "Detachment", "Functioning", "Internalising", "Cognition"]

vars_to_keep = meta["ElementName"][meta["Modality"].isin(modalities_keep)].tolist()
# Filter the data if the columns are present in cleaned_discovery. Also include src_subject_id
vars_to_keep.append("src_subject_id")
vars_to_keep.append("phenotype")
cleaned_test = cleaned_test[cleaned_test.columns.intersection(vars_to_keep)]


## Apply SVM models

Get the test data

In [ ]:
Test_df = pd.read_csv(
    "path/to/multiclust_data/cleaned_test_data.csv"
)


Apply preprocessing

In [ ]:
from Utils import *
#from full_pipeline import preprocessing

# Set parameters
subject_id_column = 'src_subject_id'
col_threshold = 0.5
row_threshold = 0.5
skew_threshold = 0.75
scaler_type = 'robust'  # Options: 'standard', 'minmax', 'robust'
modalities = list(final_metrics["data"].keys())


In [ ]:
import importlib
import Utils
Utils = importlib.reload(Utils)
from Utils import apply_dimensionality_reduction_to_new_data

_pipeline_preproc = final_metrics.get("preprocessing_details", final_metrics.get("preprocessing", {}))
_pipeline_preproc_params = _pipeline_preproc.get("preprocessing_parameters", {}) if isinstance(_pipeline_preproc, dict) else {}
_validation_dummy_code_modalities = _pipeline_preproc_params.get(
    "dummy_code_modalities",
    dummy_code_modalities if "dummy_code_modalities" in globals() else [],
)
_validation_mixed_categorical_modalities = _pipeline_preproc_params.get(
    "mixed_categorical_modalities",
    mixed_categorical_modalities if "mixed_categorical_modalities" in globals() else [],
)

print("Validation preprocessing choices from pipeline:")
print("  dummy_code_modalities:", _validation_dummy_code_modalities)
print("  mixed_categorical_modalities:", _validation_mixed_categorical_modalities)

ae_data, subject_id_list, dict_final, test_preprocessing_details = preprocessing(
    Test_df,
    meta,
    subject_id_column=subject_id_column,
    col_threshold=col_threshold,
    row_threshold=row_threshold,
    skew_threshold=skew_threshold,
    scaler_type=scaler_type,
    modalities=modalities,
    dummy_code_modalities=_validation_dummy_code_modalities,
    mixed_categorical_modalities=_validation_mixed_categorical_modalities,
    export_preprocessing_details=True,
)

# Assert identical subject order across modalities after preprocessing.
base_ids = dict_final[modalities[0]][subject_id_column].tolist()
for m in modalities[1:]:
    assert dict_final[m][subject_id_column].tolist() == base_ids, (
        f"Subject-ID order mismatch between {modalities[0]} and {m} after preprocessing"
    )

# Project the independently preprocessed validation sample into the exact SVM
# feature space learned in discovery. This reuses fitted discovery reducers
# such as PCA models, but not discovery transformed outputs.
ae_res_test, X_test, X_test_by_modality = apply_dimensionality_reduction_to_new_data(
    dict_final_new=dict_final,
    final_metrics=final_metrics,
    modalities=modalities,
    subject_id_column=subject_id_column,
)

ae_res = ae_res_test
_context = final_metrics.get("final_reporting", {}).get("compute_context", {}) if isinstance(final_metrics, dict) else {}
_dimred_by_modality = _context.get("dim_reduction_by_modality", {}) if isinstance(_context, dict) else {}
_dimred_default = _context.get("dim_reduction", "none") if isinstance(_context, dict) else "none"
_validation_dimred_methods = {
    mod: str(_dimred_by_modality.get(mod, _dimred_default)).strip().lower()
    for mod in modalities
}
print("Validation dimensionality reduction methods:", _validation_dimred_methods)
print("Validation dimensionality reduction complete:", {k: v["final_latent"].shape for k, v in ae_res_test.items()})
print("Integrated SVM validation matrix:", X_test.shape)

# Feature-space diagnostics for SVM prediction.
svm_feat = final_metrics.get("svm_feature_names")
if svm_feat is not None:
    missing = [c for c in svm_feat if c not in X_test.columns]
    extra = [c for c in X_test.columns if c not in svm_feat]
    print(f"Integrated SVM feature check: expected={len(svm_feat)}, got={X_test.shape[1]}, missing={len(missing)}, extra={len(extra)}")
    if missing:
        print("Missing integrated feature examples:", missing[:10])
    if extra:
        print("Extra integrated feature examples:", extra[:10])

feat_mods = final_metrics.get("svm_feature_names_modalities", [None] * len(modalities))
for i, mod in enumerate(modalities):
    feat_i = feat_mods[i] if feat_mods is not None and i < len(feat_mods) else None
    cols_i = list(X_test_by_modality[mod].columns)
    uses_latent = bool(cols_i) and all(str(c).startswith(f"{mod}__latent_") for c in cols_i)
    print(f"{mod}: SVM input shape={X_test_by_modality[mod].shape}, latent_features={uses_latent}")
    if feat_i is not None:
        missing_i = [c for c in feat_i if c not in cols_i]
        extra_i = [c for c in cols_i if c not in feat_i]
        print(f"  modality SVM feature check: expected={len(feat_i)}, got={len(cols_i)}, missing={len(missing_i)}, extra={len(extra_i)}")
        if missing_i:
            print("  missing examples:", missing_i[:10])
        if extra_i:
            print("  extra examples:", extra_i[:10])


In [ ]:
## Apply preprocessing to CC in test sample 

# Apply the learned CHR preprocessing to test CC.
# use preprocessing learned from the test CHR sample.
preproc_test = test_preprocessing_details

cc_df_test = cleaned_test_CC.copy()
ae_data_cc_test, subject_id_list_cc_test, dict_final_cc_test = apply_preprocessing_to_new_data(
    cc_df_test,
    meta,
    preproc_test,
    subject_id_column="src_subject_id"
)

print("Test CC preprocessing complete:", {k: v.shape for k, v in dict_final_cc_test.items()})


Run model

In [ ]:
svm_final = final_metrics['svm_final_model']
svm_modalities = final_metrics['svm_final_models_modalities']


Integrated clusters

In [ ]:
import pandas as pd
import numpy as np

svm_final = final_metrics["svm_final_model"]

# Align columns to the training feature order and fail loudly if the validation
# representation is not the same SVM feature space.
feat = final_metrics.get("svm_feature_names", None)
if feat is not None and isinstance(X_test, pd.DataFrame):
    missing = [c for c in feat if c not in X_test.columns]
    extra = [c for c in X_test.columns if c not in feat]
    if missing:
        raise ValueError(f"Validation SVM matrix is missing {len(missing)} trained features; examples: {missing[:10]}")
    if extra:
        print(f"Dropping {len(extra)} validation features not used by the integrated SVM; examples: {extra[:10]}")
    X_test_aligned = X_test.loc[:, feat]
else:
    X_test_aligned = X_test

y_pred, proba, confidence, entropy, margin = svm_predict_with_uncertainty(svm_final, X_test_aligned)

pred_final = pd.DataFrame({
    "y_pred": y_pred,
    "confidence": confidence,
    "entropy": entropy,
    "margin": margin,
})

# Optional: add per-class probabilities for debugging.
if proba is not None:
    proba_df = pd.DataFrame(proba, columns=[f"p_{c}" for c in svm_final.classes_])
    pred_final = pd.concat([pred_final, proba_df], axis=1)

test_subject_ids = dict_final[modalities[0]]["src_subject_id"].astype(str).reset_index(drop=True)
if len(test_subject_ids) != len(pred_final):
    raise ValueError(
        f"Test subject ID count ({len(test_subject_ids)}) does not match integrated predictions ({len(pred_final)})."
    )

pred_final.insert(0, "src_subject_id", test_subject_ids)
labels_test_final = pred_final["y_pred"].to_numpy()
test_final_labels_by_subject_id = dict(zip(test_subject_ids, pred_final["y_pred"]))

final_counts = pred_final["y_pred"].astype(str).value_counts().sort_index()
print("Integrated SVM predicted classes:", final_counts.to_dict())
prob_cols = [c for c in pred_final.columns if str(c).startswith("p_")]
if prob_cols:
    prob_summary = pred_final[prob_cols].describe(percentiles=[0.05, 0.5, 0.95]).loc[["mean", "5%", "50%", "95%"]]
    print("Integrated SVM probability summary:")
    display(prob_summary)
if len(final_counts) < 2:
    print("WARNING: Integrated SVM predicted a single class. Check feature-space diagnostics above before interpreting subgroup plots.")

pred_final.head(10)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# pred_final: DataFrame with columns
# ['y_pred', 'confidence', 'entropy', 'margin', 'p_0', 'p_1']

df = pred_final.copy()

# --- (Optional) sanity / recompute checks ---
# If you want to ensure these are consistent with p_0/p_1, uncomment:
# df["confidence_chk"] = np.maximum(df["p_0"], df["p_1"])
# df["margin_chk"] = np.abs(df["p_1"] - df["p_0"])
# eps = 1e-12
# df["entropy_chk"] = -(df["p_0"] * np.log(df["p_0"] + eps) + df["p_1"] * np.log(df["p_1"] + eps))

# --- 1) Histograms ---
plt.figure()
plt.hist(df["p_1"].astype(float), bins=30)
plt.xlabel("Predicted probability p(class=1)")
plt.ylabel("Count")
plt.title("p_1 distribution (unlabeled hold-out)")
plt.show()

plt.figure()
plt.hist(df["confidence"].astype(float), bins=30)
plt.xlabel("Confidence = max(p_0, p_1)")
plt.ylabel("Count")
plt.title("Confidence distribution (unlabeled hold-out)")
plt.show()

plt.figure()
plt.hist(df["entropy"].astype(float), bins=30)
plt.xlabel("Entropy (higher = more uncertain)")
plt.ylabel("Count")
plt.title("Entropy distribution (unlabeled hold-out)")
plt.show()

plt.figure()
plt.hist(df["margin"].astype(float), bins=30)
plt.xlabel("Margin |p_1 - p_0| (lower = more uncertain)")
plt.ylabel("Count")
plt.title("Margin distribution (unlabeled hold-out)")
plt.show()

# --- 2) Scatter plots ---
plt.figure()
plt.scatter(df["confidence"].astype(float), df["entropy"].astype(float), s=10)
plt.xlabel("Confidence")
plt.ylabel("Entropy")
plt.title("Confidence vs entropy")
plt.show()

plt.figure()
plt.scatter(df["confidence"].astype(float), df["margin"].astype(float), s=10)
plt.xlabel("Confidence")
plt.ylabel("Margin")
plt.title("Confidence vs margin")
plt.show()

# --- 3) Quick numeric summaries (handy for write-up) ---
conf = df["confidence"].astype(float).to_numpy()
print("\n=== Summary (unlabeled) ===")
print(f"N = {len(df)}")
print(f"Confidence mean={conf.mean():.3f}, median={np.median(conf):.3f}, min={conf.min():.3f}, max={conf.max():.3f}")
for thr in [0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
    print(f"Fraction with confidence ≥ {thr}: {(conf >= thr).mean():.3f}")

# --- 4) Show most-uncertain rows for manual inspection ---
# (lowest confidence, highest entropy, lowest margin)
print("\nLowest confidence (top 10):")
display(df.sort_values("confidence", ascending=True).head(10))

print("\nHighest entropy (top 10):")
display(df.sort_values("entropy", ascending=False).head(10))

print("\nLowest margin (top 10):")
display(df.sort_values("margin", ascending=True).head(10))


Individual modalities

In [ ]:
svm_modalities = final_metrics["svm_final_models_modalities"]
subject_id_column = 'src_subject_id'

pred_modalities = {}
labels_test_modalities = []
labels_test_by_modality = {}

feat_mods = final_metrics.get("svm_feature_names_modalities", [None] * len(modalities))

for i, mod in enumerate(modalities):
    model_mod = svm_modalities[i]
    if model_mod is None:
        print(f"Skipping SVM prediction for modality {mod} due to lack of trained model.")
        pred_modalities[mod] = None
        labels_test_by_modality[mod] = None
        labels_test_modalities.append(None)
        continue

    X_mod = X_test_by_modality[mod]

    # Align to training feature order for this modality model and fail loudly if
    # the validation representation is not the same SVM feature space.
    feat_i = feat_mods[i] if feat_mods is not None and i < len(feat_mods) else None
    if feat_i is not None and isinstance(X_mod, pd.DataFrame):
        missing_i = [c for c in feat_i if c not in X_mod.columns]
        extra_i = [c for c in X_mod.columns if c not in feat_i]
        if missing_i:
            raise ValueError(f"{mod} validation SVM matrix is missing {len(missing_i)} trained features; examples: {missing_i[:10]}")
        if extra_i:
            print(f"{mod}: dropping {len(extra_i)} validation features not used by this SVM; examples: {extra_i[:10]}")
        X_mod_aligned = X_mod.loc[:, feat_i]
    else:
        X_mod_aligned = X_mod

    y_pred, proba, confidence, entropy, margin = svm_predict_with_uncertainty(model_mod, X_mod_aligned)

    out = pd.DataFrame({
        "y_pred": y_pred,
        "confidence": confidence,
        "entropy": entropy,
        "margin": margin,
    })

    if proba is not None:
        proba_df = pd.DataFrame(proba, columns=[f"p_{c}" for c in model_mod.classes_])
        out = pd.concat([out, proba_df], axis=1)

    pred_modalities[mod] = out
    labels_test_by_modality[mod] = y_pred
    labels_test_modalities.append(y_pred)

    counts = pd.Series(y_pred).astype(str).value_counts().sort_index()
    print(f"{mod}: modality SVM predicted classes:", counts.to_dict())
    prob_cols = [c for c in out.columns if str(c).startswith("p_")]
    if prob_cols:
        prob_summary = out[prob_cols].describe(percentiles=[0.05, 0.5, 0.95]).loc[["mean", "5%", "50%", "95%"]]
        print(f"{mod}: probability summary:")
        display(prob_summary)
    if len(counts) < 2:
        print(f"WARNING: {mod} modality SVM predicted a single class. Domain-specific subgroup plots will be skipped for this modality.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_pred_modality(df, name):
    """
    Plots confidence/uncertainty diagnostics for a single modality DataFrame.
    Expected columns (any subset is ok): p_0, p_1, confidence, entropy, margin
    """
    cols = set(df.columns)

    # Compute missing fields if probabilities exist
    if {"p_0", "p_1"}.issubset(cols):
        p0 = df["p_0"].astype(float).to_numpy()
        p1 = df["p_1"].astype(float).to_numpy()

        if "confidence" not in cols:
            df = df.copy()
            df["confidence"] = np.maximum(p0, p1)

        if "margin" not in cols:
            df = df.copy()
            df["margin"] = np.abs(p1 - p0)

        if "entropy" not in cols:
            df = df.copy()
            eps = 1e-12
            df["entropy"] = -(p0 * np.log(p0 + eps) + p1 * np.log(p1 + eps))

    # Helper to plot a histogram if column exists
    def hist_if_exists(col, bins=30, xlabel=None):
        if col in df.columns:
            plt.figure()
            plt.hist(df[col].dropna().astype(float), bins=bins)
            plt.xlabel(xlabel or col)
            plt.ylabel("Count")
            plt.title(f"{name}: {col} distribution (unlabeled hold-out)")
            plt.show()

    # 1) Histograms
    hist_if_exists("p_1", xlabel="Predicted probability p(class=1)")
    hist_if_exists("confidence", xlabel="Confidence = max(p_0, p_1)")
    hist_if_exists("entropy", xlabel="Entropy (higher = more uncertain)")
    hist_if_exists("margin", xlabel="Margin = |p_1 - p_0| (lower = more uncertain)")

    # 2) Scatter plots that are often informative
    if ("confidence" in df.columns) and ("entropy" in df.columns):
        plt.figure()
        plt.scatter(df["confidence"].astype(float), df["entropy"].astype(float), s=10)
        plt.xlabel("Confidence")
        plt.ylabel("Entropy")
        plt.title(f"{name}: confidence vs entropy")
        plt.show()

    if ("confidence" in df.columns) and ("margin" in df.columns):
        plt.figure()
        plt.scatter(df["confidence"].astype(float), df["margin"].astype(float), s=10)
        plt.xlabel("Confidence")
        plt.ylabel("Margin")
        plt.title(f"{name}: confidence vs margin")
        plt.show()

    # 3) Print “most uncertain” cases (lowest confidence / lowest margin / highest entropy)
    # (useful for manual review)
    print(f"\n=== {name}: most uncertain examples ===")

    if "confidence" in df.columns:
        print("\nLowest confidence:")
        display(df.sort_values("confidence", ascending=True).head(10))

    if "margin" in df.columns:
        print("\nLowest margin:")
        display(df.sort_values("margin", ascending=True).head(10))

    if "entropy" in df.columns:
        print("\nHighest entropy:")
        display(df.sort_values("entropy", ascending=False).head(10))


# ---- Run for your pred_modalities dict ----
for modality_name, modality_df in pred_modalities.items():
    print(f"\n\n=== Diagnostics for modality: {modality_name} ===")
    plot_pred_modality(modality_df, modality_name)


## Visualise test labels

### Differences in original variables - individual labels

In [ ]:
from sklearn.feature_selection import f_classif
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Create directory for differences in original features plots
os.makedirs(os.path.join(plots_dir, "merged_feature_differences/"), exist_ok=True)

mod_num=0
for modality, df in dict_final.items():

    print(f"\n=== Modality: {modality} ===")

    # Extract data and labels
    X = df.drop(columns=['src_subject_id']).values
    clusters = labels_test_modalities[mod_num]
    feature_names = df.drop(columns=['src_subject_id']).columns

    # --- Feature Ranking (ANOVA F-test) ---
    f_vals, _ = f_classif(X, clusters)
    f_df = (
        pd.DataFrame({'feature': feature_names, 'f_value': f_vals})
        .sort_values('f_value', ascending=False)
    )

    # --- Barplot of Top Features ---
    top_k = 10
    plt.figure(figsize=(10, 6))
    sns.barplot(x='f_value', y='feature', data=f_df.head(top_k), palette=sns.color_palette(n_colors=top_k))
    plt.title(f"{modality} — Top {top_k} Discriminative Features (ANOVA F-value)")
    plt.xlabel("F-value")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()


    # --- Boxplots + Scatter Points of Top Features + CC ---
    top_features = f_df['feature'].head(top_k).tolist()

    max_cols = 3
    n_feats = len(top_features)
    n_rows = int(np.ceil(n_feats / max_cols))

    fig, axes = plt.subplots(n_rows, max_cols, figsize=(max_cols * 6, n_rows * 5))
    axes = axes.flatten()

    cluster_order = [str(x) for x in np.unique(clusters)]
    df_cc = dict_final_cc_test.get(modality)
    cc_available = df_cc is not None
    if cc_available:
        df_cc = df_cc.copy().reindex(columns=df.columns, fill_value=np.nan)
    group_order = cluster_order + (["CC"] if cc_available else [])
    pal = sns.color_palette(n_colors=len(group_order))

    for i, feat in enumerate(top_features):
        ax = axes[i]
        plot_df = pd.DataFrame({
            'group': pd.Series(clusters).astype(str),
            'value': df[feat].values,
        }).dropna()
        if cc_available and feat in df_cc.columns:
            cc_plot_df = pd.DataFrame({
                'group': 'CC',
                'value': df_cc[feat].values,
            }).dropna()
            plot_df = pd.concat([plot_df, cc_plot_df], ignore_index=True)
        sns.boxplot(
            data=plot_df, x='group', y='value', order=group_order, palette=pal, ax=ax
        )
        sns.stripplot(
            data=plot_df, x='group', y='value', order=group_order,
            color='black', size=4, jitter=True, alpha=0.5, ax=ax
        )

        # Larger text sizes
        ax.set_title(feat, fontsize=14)
        ax.set_xlabel("", fontsize=22)
        ax.set_ylabel("", fontsize=22)
        ax.tick_params(axis='both', labelsize=14)

    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f"{modality} — Distribution of Top Features by Cluster and CC\n(with individual data points)",
        y=1.02, fontsize=18
    )

    plt.tight_layout()
    fig.savefig(os.path.join(plots_dir, "merged_feature_differences/", f"{modality}_top_features_with_cc.png"))
    plt.show()

    mod_num=mod_num+1


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# =========================
# PCA aggregated plots per modality + variance explained (first 5 PCs)
# - Respects your existing global theme (NO sns.set_theme here)
# - PC1 by cluster: violin (quartiles) + jitter + median marker + n labels
# - Variance plot: explained variance ratio for PC1..PC5
# - Optional: PC1 loadings (top +/- contributors)
# =========================

out_dir = os.path.join(plots_dir, "merged_feature_pca/")
os.makedirs(out_dir, exist_ok=True)

loadings_dir = os.path.join(out_dir, "pc1_loadings/")
os.makedirs(loadings_dir, exist_ok=True)

variance_dir = os.path.join(out_dir, "variance/")
os.makedirs(variance_dir, exist_ok=True)

mod_num = 0
for modality, df in dict_final.items():
    print(f"\n=== PCA Aggregated Plots test sample — Modality: {modality} ===")

    # Extract data and labels
    X = df.drop(columns=['src_subject_id']).values
    clusters = labels_test_modalities[mod_num]
    feature_names = df.drop(columns=['src_subject_id']).columns

    # Stable cluster order (customize if you want a specific ordering)
    cluster_order = np.sort(pd.unique(clusters))

    # --- Standardize ---
    Xz = StandardScaler().fit_transform(X)

    # --- PCA for variance (first 5 components) ---
    n_pcs = min(5, Xz.shape[1])  # cannot exceed number of features
    pca_var = PCA(n_components=n_pcs, random_state=0)
    pca_var.fit(Xz)

    evr = pca_var.explained_variance_ratio_
    cum_evr = np.cumsum(evr)

    # --- Also compute PC scores (at least PC1) ---
    pc_scores = pca_var.transform(Xz)  # shape: (n_samples, n_pcs)
    pc1 = pc_scores[:, 0]
    evr1 = float(evr[0])
    print(f"PC1 EVR: {evr1:.2%}")

    plot_df = pd.DataFrame({"cluster": clusters, "PC1": pc1})

    # -------------------------
    # 1) PC1 by cluster (publication-ready, respects global theme)
    # -------------------------
    fig, ax = plt.subplots(figsize=(10.5, 6.5))


    sns.violinplot(
        data=plot_df,
        x="cluster",
        y="PC1",
        order=cluster_order,
        palette=sns.color_palette(n_colors=len(cluster_order)),
        inner="quartile",
        cut=0,
        bw_adjust=1.0,
        linewidth=1,
        ax=ax
    )

    sns.stripplot(
        data=plot_df,
        x="cluster",
        y="PC1",
        order=cluster_order,
        color="black",
        size=3,
        jitter=0.22,
        alpha=0.35,
        ax=ax
    )

    # Median marker per cluster
    medians = plot_df.groupby("cluster")["PC1"].median()
    x_positions = np.arange(len(cluster_order))
    ax.scatter(
        x=x_positions,
        y=[medians.loc[c] for c in cluster_order],
        s=180,
        marker="_",
        linewidths=3
    )

    # Annotate n per cluster near the bottom
    counts = plot_df["cluster"].value_counts()
    ymin, ymax = ax.get_ylim()
    y_annot = ymin + 0.04 * (ymax - ymin)
    for i, c in enumerate(cluster_order):
        ax.text(i, y_annot, f"n={int(counts.loc[c])}", ha="center", va="bottom", fontsize=12)

    ax.set_xlabel("Cluster", fontsize=16)
    ax.set_ylabel("PCA Component 1 score", fontsize=16)

    # Light grid for readability; remove if your global theme already handles grids
    ax.grid(axis="y", alpha=0.15)
    sns.despine(ax=ax)

    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"test_{modality}_PC1_violin_pubready.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # 2) Variance explained (PC1..PC5): bar + cumulative line
    # -------------------------
    figv, axv = plt.subplots(figsize=(10, 5.5))

    pcs_idx = np.arange(1, n_pcs + 1)
    axv.bar(pcs_idx, evr)  # uses your global matplotlib color cycle
    axv.plot(pcs_idx, cum_evr, marker="o")

    axv.set_xticks(pcs_idx)
    axv.set_xlabel("Principal Component")
    axv.set_ylabel("Explained variance ratio")
    axv.set_title(f"{modality} — PCA Variance Explained (Top {n_pcs} PCs)", pad=12)

    axv.set_ylim(0, max(0.25, evr.max() * 1.2))  # keeps plot readable if EVR is small/large
    axv.grid(axis="y", alpha=0.15)
    sns.despine(ax=axv)

    figv.tight_layout()
    figv.savefig(os.path.join(variance_dir, f"test_{modality}_variance_top{n_pcs}.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # 3) Optional: PC1 feature loadings (top +/- contributors)
    # -------------------------
    loadings = pca_var.components_[0]  # PC1 loadings
    load_df = pd.DataFrame({"feature": feature_names, "loading": loadings})

    top_n = min(15, len(feature_names) // 2) if len(feature_names) >= 2 else 1
    top_pos = load_df.sort_values("loading", ascending=False).head(top_n)
    top_neg = load_df.sort_values("loading", ascending=True).head(top_n)
    load_plot_df = pd.concat([top_neg, top_pos], axis=0)

    fig2, ax2 = plt.subplots(figsize=(10.5, 7.5))
    sns.barplot(data=load_plot_df, x="loading", y="feature", ax=ax2)
    ax2.axvline(0, linewidth=1)

    ax2.set_title(f"{modality} — PC1 Feature Loadings (Top ±{top_n})", pad=12)
    ax2.set_xlabel("PC1 loading")
    ax2.set_ylabel("")

    ax2.grid(axis="x", alpha=0.15)
    sns.despine(ax=ax2)

    fig2.tight_layout()
    fig2.savefig(os.path.join(loadings_dir, f"test_{modality}_PC1_loadings_top_pm{top_n}.png"), dpi=300, bbox_inches="tight")
    plt.show()

    mod_num += 1


Different colours

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# =========================
# PCA aggregated plots per modality: CHR clusters + CC
# - Each modality can have its own manual colors
# - Fits scaler+PCA on CHR only, projects CC into same space
# - Black jittered points always in front of violins
# =========================

# Required inputs:
# final_metrics  -> CHR final metrics dict
# dict_final_cc_test -> output of apply_preprocessing_to_new_data(...): test CC per modality
# plots_dir      -> base directory for plots

out_dir = os.path.join(plots_dir, "merged_feature_pca_chr_vs_cc")
os.makedirs(out_dir, exist_ok=True)

# --------------------------------------------------
# MANUAL COLORS PER MODALITY
# Edit these however you want
# Keys must match your modality names exactly
# Inner keys must match your group labels exactly: "0", "1", "CC", etc.
# --------------------------------------------------
modality_palettes = {
    "Internalising": {
        "0": "#2F7F73",   # deep teal
        "1": "#8EDFD1",   # soft aqua
        "CC": "#065982"
    },
    "Functioning": {
        "0": "#0B7C25",   # muted coral
        "1": "#C5FDC7",   # soft blush-peach
        "CC": "#065982"
    },
    "Detachment": {
        "0": "#7C68C0",   # soft violet
        "1": "#C7B8EA",   # pale lavender
        "CC": "#065982"
    },
    "Psychoticism": {
        "0": "#D15B94",   # muted rose-magenta
        "1": "#E8B4CC",   # light pink-mauve
        "CC": "#065982"
    },
    "Cognition": {
        "0": "#5196E0",   # calm medium blue
        "1": "#B7D4F0",   # light sky blue
        "CC": "#065982"
    }
}



# Optional fallback if a modality is not listed above
default_palette = {
    "0": "#327D6D",
    "1": "#7FE3CD",
    "CC": "#065982"
}

mod_num = 0
for modality, df_chr in dict_final.items():
    print(f"\n=== PCA Aggregated Plots test sample — Modality: {modality} ===")

    # CHR features + labels
    X_chr_df = df_chr.drop(columns=["src_subject_id"]).copy()
    clusters_chr = np.asarray(labels_test_modalities[mod_num])

    # CC features
    df_cc = dict_final_cc_test[modality]
    X_cc_df = df_cc.drop(columns=["src_subject_id"]).copy()

    # Strict feature alignment to CHR
    X_cc_df = X_cc_df.reindex(columns=X_chr_df.columns)

    # Optional diagnostics
    print("CHR shape:", X_chr_df.shape, "CC shape:", X_cc_df.shape)
    print("NaNs in CC feature matrix:", int(X_cc_df.isna().sum().sum()))

    # Convert to arrays
    X_chr = X_chr_df.values
    X_cc = X_cc_df.values

    # Fit scaler + PCA on CHR only
    scaler = StandardScaler()
    X_chr_z = scaler.fit_transform(X_chr)
    X_cc_z = scaler.transform(X_cc)

    n_pcs = min(5, X_chr_z.shape[1])
    pca = PCA(n_components=n_pcs, random_state=0)
    PC_chr = pca.fit_transform(X_chr_z)
    PC_cc = pca.transform(X_cc_z)

    pc1_chr = PC_chr[:, 0]
    pc1_cc = PC_cc[:, 0]

    # Plot dataframe
    plot_chr = pd.DataFrame({
        "group": clusters_chr.astype(str),
        "PC1": pc1_chr,
        "cohort": "CHR"
    })
    plot_cc = pd.DataFrame({
        "group": "CC",
        "PC1": pc1_cc,
        "cohort": "CC"
    })
    plot_df = pd.concat([plot_chr, plot_cc], ignore_index=True)

    cluster_order = sorted(
        plot_chr["group"].unique(),
        key=lambda x: int(x) if str(x).isdigit() else str(x)
    )
    group_order = cluster_order + ["CC"]

    # ---------------------------------------------
    # Pick palette for this modality
    # ---------------------------------------------
    group_palette = modality_palettes.get(modality, default_palette).copy()

    # Safety check: ensure every group has a color
    missing_groups = [g for g in group_order if g not in group_palette]
    if missing_groups:
        raise ValueError(
            f"Missing colors for modality '{modality}' and groups: {missing_groups}"
        )

    fig, ax = plt.subplots(figsize=(10.5, 6.5))

    # Violin by group color
    sns.violinplot(
        data=plot_df,
        x="group",
        y="PC1",
        hue="group",
        order=group_order,
        hue_order=group_order,
        palette=group_palette,
        dodge=False,
        inner="quartile",
        cut=0,
        bw_adjust=1.0,
        linewidth=1,
        legend=False,
        ax=ax
    )

    # Black points on top
    sns.stripplot(
        data=plot_df,
        x="group",
        y="PC1",
        order=group_order,
        color="black",
        size=3,
        jitter=0.22,
        alpha=0.35,
        ax=ax
    )

    # Bring points forward
    for c in ax.collections:
        c.set_zorder(2)
    for l in ax.lines:
        l.set_zorder(3)

    # Median marker
    medians = plot_df.groupby("group")["PC1"].median()
    x_positions = np.arange(len(group_order))
    med_vals = [medians.loc[g] for g in group_order]
    ax.scatter(
        x_positions, med_vals,
        marker="_", s=180, linewidths=3,
        color="black", zorder=4
    )

    # n labels
    counts = plot_df["group"].value_counts()
    ymin, ymax = ax.get_ylim()
    y_annot = ymin + 0.04 * (ymax - ymin)
    for i, g in enumerate(group_order):
        ax.text(i, y_annot, f"n={int(counts.get(g, 0))}",
                ha="center", va="top", fontsize=14)

    ax.set_xlabel("CHR clusters + CC", fontsize=16)
    ax.set_ylabel("PCA Component 1 score", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    ax.set_title(f"{modality}: CHR cluster PC1 vs CC", fontsize=16)
    ax.grid(axis="y", alpha=0.15)
    sns.despine(ax=ax)

    fig.tight_layout()
    fig.savefig(
        os.path.join(out_dir, f"test_{modality}_PC1_CHR_vs_CC.svg"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    mod_num += 1


### Differences in original variables - final labels

In [ ]:
from sklearn.feature_selection import f_classif
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Create directory for differences in original features plots
os.makedirs(os.path.join(plots_dir, "merged_feature_differences/"), exist_ok=True)

for modality, df in dict_final.items():

    print(f"\n=== Modality: {modality} ===")

    # Extract data and labels
    X = df.drop(columns=['src_subject_id']).values
    clusters = labels_test_final
    feature_names = df.drop(columns=['src_subject_id']).columns

    # --- Feature Ranking (ANOVA F-test) ---
    f_vals, _ = f_classif(X, clusters)
    f_df = (
        pd.DataFrame({'feature': feature_names, 'f_value': f_vals})
        .sort_values('f_value', ascending=False)
    )

    # --- Barplot of Top Features ---
    top_k = 10
    plt.figure(figsize=(10, 6))
    sns.barplot(x='f_value', y='feature', data=f_df.head(top_k), palette=sns.color_palette(n_colors=top_k))
    plt.title(f"{modality} — Top {top_k} Discriminative Features (ANOVA F-value)")
    plt.xlabel("F-value")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()


    # --- Boxplots + Scatter Points of Top Features + CC ---
    top_features = f_df['feature'].head(top_k).tolist()

    max_cols = 3
    n_feats = len(top_features)
    n_rows = int(np.ceil(n_feats / max_cols))

    fig, axes = plt.subplots(n_rows, max_cols, figsize=(max_cols * 6, n_rows * 5))
    axes = axes.flatten()

    cluster_order = [str(x) for x in np.unique(clusters)]
    df_cc = dict_final_cc_test.get(modality)
    cc_available = df_cc is not None
    if cc_available:
        df_cc = df_cc.copy().reindex(columns=df.columns, fill_value=np.nan)
    group_order = cluster_order + (["CC"] if cc_available else [])
    pal = sns.color_palette(n_colors=len(group_order))

    for i, feat in enumerate(top_features):
        ax = axes[i]
        plot_df = pd.DataFrame({
            'group': pd.Series(clusters).astype(str),
            'value': df[feat].values,
        }).dropna()
        if cc_available and feat in df_cc.columns:
            cc_plot_df = pd.DataFrame({
                'group': 'CC',
                'value': df_cc[feat].values,
            }).dropna()
            plot_df = pd.concat([plot_df, cc_plot_df], ignore_index=True)
        sns.boxplot(
            data=plot_df, x='group', y='value', order=group_order, palette=pal, ax=ax
        )
        sns.stripplot(
            data=plot_df, x='group', y='value', order=group_order,
            color='black', size=4, jitter=True, alpha=0.5, ax=ax
        )

        # Larger text sizes
        ax.set_title(feat, fontsize=14)
        ax.set_xlabel("", fontsize=22)
        ax.set_ylabel("", fontsize=22)
        ax.tick_params(axis='both', labelsize=14)

    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f"{modality} — Distribution of Top Features by Cluster and CC\n(with individual data points)",
        y=1.02, fontsize=18
    )

    plt.tight_layout()
    fig.savefig(os.path.join(plots_dir, "merged_feature_differences/", f"{modality}_top_features_with_cc.png"))
    plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# =========================
# PCA aggregated plots per modality + variance explained (first 5 PCs)
# - Respects your existing global theme (NO sns.set_theme here)
# - PC1 by cluster: violin (quartiles) + jitter + median marker + n labels
# - Variance plot: explained variance ratio for PC1..PC5
# - Optional: PC1 loadings (top +/- contributors)
# =========================

out_dir = os.path.join(plots_dir, "merged_feature_pca/")
os.makedirs(out_dir, exist_ok=True)

loadings_dir = os.path.join(out_dir, "pc1_loadings/")
os.makedirs(loadings_dir, exist_ok=True)

variance_dir = os.path.join(out_dir, "variance/")
os.makedirs(variance_dir, exist_ok=True)

mod_num = 0
for modality, df in dict_final.items():
    print(f"\n=== PCA Aggregated Plots test sample — Modality: {modality} ===")

    # Extract data and labels
    X = df.drop(columns=['src_subject_id']).values
    clusters = labels_test_final
    feature_names = df.drop(columns=['src_subject_id']).columns

    # Stable cluster order (customize if you want a specific ordering)
    cluster_order = np.sort(pd.unique(clusters))

    # --- Standardize ---
    Xz = StandardScaler().fit_transform(X)

    # --- PCA for variance (first 5 components) ---
    n_pcs = min(5, Xz.shape[1])  # cannot exceed number of features
    pca_var = PCA(n_components=n_pcs, random_state=0)
    pca_var.fit(Xz)

    evr = pca_var.explained_variance_ratio_
    cum_evr = np.cumsum(evr)

    # --- Also compute PC scores (at least PC1) ---
    pc_scores = pca_var.transform(Xz)  # shape: (n_samples, n_pcs)
    pc1 = pc_scores[:, 0]
    evr1 = float(evr[0])
    print(f"PC1 EVR: {evr1:.2%}")

    plot_df = pd.DataFrame({"cluster": clusters, "PC1": pc1})

    # -------------------------
    # 1) PC1 by cluster (publication-ready, respects global theme)
    # -------------------------
    fig, ax = plt.subplots(figsize=(10.5, 6.5))


    sns.violinplot(
        data=plot_df,
        x="cluster",
        y="PC1",
        order=cluster_order,
        palette=sns.color_palette(n_colors=len(cluster_order)),
        inner="quartile",
        cut=0,
        bw_adjust=1.0,
        linewidth=1,
        ax=ax
    )

    sns.stripplot(
        data=plot_df,
        x="cluster",
        y="PC1",
        order=cluster_order,
        color="black",
        size=3,
        jitter=0.22,
        alpha=0.35,
        ax=ax
    )

    # Median marker per cluster
    medians = plot_df.groupby("cluster")["PC1"].median()
    x_positions = np.arange(len(cluster_order))
    ax.scatter(
        x=x_positions,
        y=[medians.loc[c] for c in cluster_order],
        s=180,
        marker="_",
        linewidths=3
    )

    # Annotate n per cluster near the bottom
    counts = plot_df["cluster"].value_counts()
    ymin, ymax = ax.get_ylim()
    y_annot = ymin + 0.04 * (ymax - ymin)
    for i, c in enumerate(cluster_order):
        ax.text(i, y_annot, f"n={int(counts.loc[c])}", ha="center", va="bottom", fontsize=12)

    ax.set_xlabel("Cluster", fontsize=16)
    ax.set_ylabel("PCA Component 1 score", fontsize=16)

    # Light grid for readability; remove if your global theme already handles grids
    ax.grid(axis="y", alpha=0.15)
    sns.despine(ax=ax)

    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"{modality}_PC1_violin_pubready.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # 2) Variance explained (PC1..PC5): bar + cumulative line
    # -------------------------
    figv, axv = plt.subplots(figsize=(10, 5.5))

    pcs_idx = np.arange(1, n_pcs + 1)
    axv.bar(pcs_idx, evr)  # uses your global matplotlib color cycle
    axv.plot(pcs_idx, cum_evr, marker="o")

    axv.set_xticks(pcs_idx)
    axv.set_xlabel("Principal Component")
    axv.set_ylabel("Explained variance ratio")
    axv.set_title(f"{modality} — PCA Variance Explained (Top {n_pcs} PCs)", pad=12)

    axv.set_ylim(0, max(0.25, evr.max() * 1.2))  # keeps plot readable if EVR is small/large
    axv.grid(axis="y", alpha=0.15)
    sns.despine(ax=axv)

    figv.tight_layout()
    figv.savefig(os.path.join(variance_dir, f"{modality}_variance_top{n_pcs}.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # 3) Optional: PC1 feature loadings (top +/- contributors)
    # -------------------------
    loadings = pca_var.components_[0]  # PC1 loadings
    load_df = pd.DataFrame({"feature": feature_names, "loading": loadings})

    top_n = min(15, len(feature_names) // 2) if len(feature_names) >= 2 else 1
    top_pos = load_df.sort_values("loading", ascending=False).head(top_n)
    top_neg = load_df.sort_values("loading", ascending=True).head(top_n)
    load_plot_df = pd.concat([top_neg, top_pos], axis=0)

    fig2, ax2 = plt.subplots(figsize=(10.5, 7.5))
    sns.barplot(data=load_plot_df, x="loading", y="feature", ax=ax2)
    ax2.axvline(0, linewidth=1)

    ax2.set_title(f"{modality} — PC1 Feature Loadings (Top ±{top_n})", pad=12)
    ax2.set_xlabel("PC1 loading")
    ax2.set_ylabel("")

    ax2.grid(axis="x", alpha=0.15)
    sns.despine(ax=ax2)

    fig2.tight_layout()
    fig2.savefig(os.path.join(loadings_dir, f"{modality}_PC1_loadings_top_pm{top_n}.png"), dpi=300, bbox_inches="tight")
    plt.show()

    mod_num += 1


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ==========================================================
# GLOBAL PCA (ALL modalities merged) -> PC1 by final clusters
# - Merges modalities on src_subject_id (inner-join by default)
# - Standardizes all features
# - PCA -> PC1
# - Publication-ready violin (quartiles) + jitter + median + n
# - Also saves a variance plot (top 5 PCs) for the merged space
# ==========================================================

global_out_dir = os.path.join(plots_dir, "global_pca_all_modalities/")
os.makedirs(global_out_dir, exist_ok=True)

# ---- 1) Merge all modalities into one wide dataframe ----
dfs = []
for modality, df in dict_final.items():
    tmp = df.copy()

    # Prefix feature names with modality to avoid collisions
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})

    dfs.append(tmp)

# Inner join across modalities by subject id (keeps subjects present in ALL modalities)
merged = dfs[0]
for d in dfs[1:]:
    merged = merged.merge(d, on="src_subject_id", how="inner")

print(f"[GLOBAL PCA] Subjects after inner-join across modalities: {merged.shape[0]}")
print(f"[GLOBAL PCA] Total merged features: {merged.shape[1] - 1}")

# ---- 2) Align test integrated labels to the merged subject IDs ----
merged_clusters = merged["src_subject_id"].astype(str).map(test_final_labels_by_subject_id).to_numpy()
if np.any(pd.isna(merged_clusters)):
    missing = merged.loc[pd.isna(merged_clusters), "src_subject_id"].head(5).tolist()
    raise ValueError(f"Missing test integrated labels for merged subjects. Examples: {missing}")

# ---- 3) PCA on all features ----
X = merged.drop(columns=["src_subject_id"]).values

# Standardize before PCA
Xz = StandardScaler().fit_transform(X)

# Fit PCA (enough components to report variance for first 5, but at least 2 if possible)
n_pcs = min(5, Xz.shape[1])
pca = PCA(n_components=n_pcs, random_state=0)
scores = pca.fit_transform(Xz)

pc1 = scores[:, 0]
evr = pca.explained_variance_ratio_
cum_evr = np.cumsum(evr)

print(f"[GLOBAL PCA] PC1 EVR: {evr[0]:.2%}")

# ---- 4) Plot PC1 by cluster (global / all modalities) ----
plot_df = pd.DataFrame({"cluster": merged_clusters, "PC1": pc1})
cluster_order = np.sort(pd.unique(plot_df["cluster"]))

fig, ax = plt.subplots(figsize=(10.5, 6.5))

sns.violinplot(
    data=plot_df,
    x="cluster",
    y="PC1",
    order=cluster_order,
    palette=sns.color_palette(n_colors=len(cluster_order)),
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    ax=ax
)

sns.stripplot(
    data=plot_df,
    x="cluster",
    y="PC1",
    order=cluster_order,
    color="black",
    size=3,
    jitter=0.22,
    alpha=0.35,
    ax=ax
)

# Median marker per cluster
medians = plot_df.groupby("cluster")["PC1"].median()
x_positions = np.arange(len(cluster_order))
ax.scatter(
    x=x_positions,
    y=[medians.loc[c] for c in cluster_order],
    s=180,
    marker="_",
    linewidths=3
)

# Annotate n per cluster near bottom
counts = plot_df["cluster"].value_counts()
ymin, ymax = ax.get_ylim()
y_annot = ymin + 0.04 * (ymax - ymin)
for i, c in enumerate(cluster_order):
    ax.text(i, y_annot, f"n={int(counts.loc[c])}", ha="center", va="bottom", fontsize=12)

ax.set_xlabel("Cluster", fontsize=16)
ax.set_ylabel("PCA Component 1 score", fontsize=16)
ax.grid(axis="y", alpha=0.15)
sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(os.path.join(global_out_dir, "GLOBAL_all_modalities_PC1_violin.png"), dpi=300, bbox_inches="tight")
plt.show()

# ---- 5) Variance explained plot (Top PCs) ----
figv, axv = plt.subplots(figsize=(10, 5.5))
pcs_idx = np.arange(1, n_pcs + 1)

axv.bar(pcs_idx, evr)
axv.plot(pcs_idx, cum_evr, marker="o")

axv.set_xticks(pcs_idx)
axv.set_xlabel("Principal Component")
axv.set_ylabel("Explained variance ratio")
axv.set_title(f"GLOBAL (All Modalities) — PCA Variance Explained (Top {n_pcs} PCs)", pad=12)

axv.grid(axis="y", alpha=0.15)
sns.despine(ax=axv)

figv.tight_layout()
figv.savefig(os.path.join(global_out_dir, f"GLOBAL_all_modalities_variance_top{n_pcs}.png"),
             dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ==========================================================
# 1) SINGLE PLOT: PC1 (per-modality PCA) distributions by modality (hue=integrated cluster)
# 2) ADDITIONAL "GLOBAL" DIFFERENCE: one shared PC1 computed from ALL features across ALL modalities
#    -> a separate global violin plot + optional printout of EVR
#
# IMPORTANT: Test integrated labels are aligned to src_subject_id via
# test_final_labels_by_subject_id, created from the test predictions.
# ==========================================================

out_dir = os.path.join(plots_dir, "merged_feature_pca/")
os.makedirs(out_dir, exist_ok=True)

# --------------------------
# A) Per-modality PC1 plot (your current single plot)
# --------------------------
rows = []
for modality, df in dict_final.items():
    feature_df = df.drop(columns=["src_subject_id"])
    X = feature_df.values
    clusters = df["src_subject_id"].astype(str).map(test_final_labels_by_subject_id).to_numpy()
    if np.any(pd.isna(clusters)):
        missing = df.loc[pd.isna(clusters), "src_subject_id"].head(5).tolist()
        raise ValueError(f"Missing test integrated labels for modality {modality}. Examples: {missing}")

    Xz = StandardScaler().fit_transform(X)
    pca = PCA(n_components=1, random_state=0).fit(Xz)
    pc1 = pca.transform(Xz)[:, 0]
    evr1 = float(pca.explained_variance_ratio_[0])

    tmp = pd.DataFrame({"modality": modality, "cluster": clusters, "PC1": pc1})
    tmp["PC1_EVR"] = evr1
    rows.append(tmp)

    print(f"{modality}: PC1 EVR={evr1:.2%}")

plot_df = pd.concat(rows, ignore_index=True)

modality_order = list(dict_final.keys())
cluster_order = np.sort(plot_df["cluster"].unique())

fig_w = max(18, 2.2 * len(modality_order))
fig_h = 9
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

sns.violinplot(
    data=plot_df,
    x="modality",
    y="PC1",
    hue="cluster",
    order=modality_order,
    hue_order=cluster_order,
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    dodge=True,
    width=0.65,
    ax=ax
)

sns.stripplot(
    data=plot_df,
    x="modality",
    y="PC1",
    hue="cluster",
    order=modality_order,
    hue_order=cluster_order,
    dodge=True,
    jitter=0.18,
    size=2.2,
    alpha=0.18,
    color="black",
    ax=ax
)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:len(cluster_order)], labels[:len(cluster_order)], title="Cluster", frameon=False, loc="upper right")

for x in np.arange(0.5, len(modality_order), 1.0):
    ax.axvline(x, linewidth=0.8, alpha=0.25)

ax.set_xlabel("Modality", fontsize=16)
ax.set_ylabel("PCA Component 1 score", fontsize=16)
ax.set_title("PC1 Distributions by Modality (Integrated Cluster Differences)", fontsize=18, pad=14)
ax.grid(axis="y", alpha=0.15)
sns.despine(ax=ax)

plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=13)
plt.setp(ax.get_yticklabels(), fontsize=13)

fig.tight_layout()
fig.savefig(os.path.join(out_dir, "test_ALL_modalities_PC1_singleplot_violin_big.png"), dpi=300, bbox_inches="tight")
plt.show()


# --------------------------
# B) GLOBAL PCA PC1 across ALL modalities/features (shared PC axis)
# --------------------------

# 1) Merge all modalities into one wide table keyed by src_subject_id
dfs = []
for modality, df in dict_final.items():
    tmp = df.copy()
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})  # avoid name collisions
    dfs.append(tmp)

merged = dfs[0]
for d in dfs[1:]:
    merged = merged.merge(d, on="src_subject_id", how="inner")  # subjects present in ALL modalities

print(f"\n[GLOBAL PCA] Subjects after inner-join: {merged.shape[0]}")
print(f"[GLOBAL PCA] Total merged features: {merged.shape[1] - 1}")

# 2) Align integrated test labels to merged subject IDs
merged_clusters = merged["src_subject_id"].astype(str).map(test_final_labels_by_subject_id).to_numpy()
if np.any(pd.isna(merged_clusters)):
    missing = merged.loc[pd.isna(merged_clusters), "src_subject_id"].head(5).tolist()
    raise ValueError(
        "Some merged test subjects are missing labels in test_final_labels_by_subject_id. "
        f"Examples: {missing}"
    )

# 3) Global PCA (PC1)
X_global = merged.drop(columns=["src_subject_id"]).values
Xg_z = StandardScaler().fit_transform(X_global)

pca_global = PCA(n_components=1, random_state=0).fit(Xg_z)
pc1_global = pca_global.transform(Xg_z)[:, 0]
evr1_global = float(pca_global.explained_variance_ratio_[0])
print(f"[GLOBAL PCA] PC1 EVR: {evr1_global:.2%}")

global_df = pd.DataFrame({
    "cluster": merged_clusters,
    "PC1_global": pc1_global
})

global_cluster_order = np.sort(global_df["cluster"].unique())

# 4) Plot global PC1 difference by integrated cluster
fig2, ax2 = plt.subplots(figsize=(10.5, 6.5))

sns.violinplot(
    data=global_df,
    x="cluster",
    y="PC1_global",
    order=global_cluster_order,
    palette=sns.color_palette(n_colors=len(global_cluster_order)),
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    ax=ax2
)

sns.stripplot(
    data=global_df,
    x="cluster",
    y="PC1_global",
    order=global_cluster_order,
    color="black",
    size=3,
    jitter=0.22,
    alpha=0.30,
    ax=ax2
)

# Median markers
med = global_df.groupby("cluster")["PC1_global"].median()
xpos = np.arange(len(global_cluster_order))
ax2.scatter(
    xpos,
    [med.loc[c] for c in global_cluster_order],
    s=180,
    marker="_",
    linewidths=3
)

# n labels
counts = global_df["cluster"].value_counts()
ymin, ymax = ax2.get_ylim()
y_annot = ymin + 0.04 * (ymax - ymin)
#for i, c in enumerate(global_cluster_order):
#    ax2.text(i, y_annot, f"n={int(counts.loc[c])}", ha="center", va="top", fontsize=16)

ax2.set_xlabel("Integrated cluster", fontsize=20)
ax2.set_ylabel("Global PC1 score (all modalities/features)", fontsize=20)
ax2.set_title(f"Global PC1 Across All Features and Modalities", fontsize=20, pad=14)
ax2.grid(axis="y", alpha=0.15)
ax2.set_xticklabels(ax2.get_xticklabels(), fontsize=16)
ax2.set_yticklabels(ax2.get_yticklabels(), fontsize=16)
sns.despine(ax=ax2)

fig2.tight_layout()
fig2.savefig(os.path.join(out_dir, "GLOBAL_all_modalities_PC1_violin.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ==========================================================
# 1) SINGLE PLOT: PC1 (per-modality PCA) distributions by modality
#    - CHR integrated clusters + CC group
# 2) GLOBAL PCA across ALL modalities/features
#    - CHR integrated clusters + CC group
# ==========================================================

out_dir = os.path.join(plots_dir, "merged_feature_pca_chr_vs_cc/")
os.makedirs(out_dir, exist_ok=True)

# ---------- custom colors ----------
cluster_colors = ["#327D6D", "#7FE3CD", "#F2B134", "#B65FCF", "#E76F51", "#8DA0CB"]
CC_COLOR = "#065982ff"

# --------------------------
# A) Per-modality PC1 plot: CHR clusters + CC
# --------------------------


rows = []

for modality, df_chr in dict_final.items():
    # CHR
    X_chr_df = df_chr.drop(columns=["src_subject_id"]).copy()
    grp_chr_raw = df_chr["src_subject_id"].astype(str).map(test_final_labels_by_subject_id)
    if np.any(pd.isna(grp_chr_raw)):
        missing = df_chr.loc[pd.isna(grp_chr_raw), "src_subject_id"].head(5).tolist()
        raise ValueError(f"{modality}: missing test integrated labels for CHR rows. Examples: {missing}")
    grp_chr = grp_chr_raw.astype(str).to_numpy()

    # CC (already transformed via apply_preprocessing_to_new_data)
    df_cc = dict_final_cc_test[modality]
    X_cc_df = df_cc.drop(columns=["src_subject_id"]).copy()
    X_cc_df = X_cc_df.reindex(columns=X_chr_df.columns)  # strict CHR feature order

    # Fit scaler+PCA on CHR only
    scaler = StandardScaler()
    X_chr_z = scaler.fit_transform(X_chr_df.values)
    X_cc_z = scaler.transform(X_cc_df.values)

    pca = PCA(n_components=1, random_state=0).fit(X_chr_z)
    pc1_chr = pca.transform(X_chr_z)[:, 0]
    pc1_cc = pca.transform(X_cc_z)[:, 0]
    evr1 = float(pca.explained_variance_ratio_[0])

    print(f"{modality}: CHR-fitted PC1 EVR={evr1:.2%}")

    tmp_chr = pd.DataFrame({
        "modality": modality,
        "group": grp_chr,
        "PC1": pc1_chr,
        "cohort": "CHR"
    })
    tmp_cc = pd.DataFrame({
        "modality": modality,
        "group": "CC",
        "PC1": pc1_cc,
        "cohort": "CC"
    })
    tmp = pd.concat([tmp_chr, tmp_cc], ignore_index=True)
    tmp["PC1_EVR_chrfit"] = evr1
    rows.append(tmp)

plot_df = pd.concat(rows, ignore_index=True)

modality_order = list(final_metrics["data"].keys())
chr_labels_all = pd.Series(plot_df.loc[plot_df["cohort"] == "CHR", "group"].astype(str).unique())
chr_cluster_order = sorted(chr_labels_all, key=lambda x: int(x) if str(x).isdigit() else str(x))
group_order = chr_cluster_order + ["CC"]

group_palette = {g: cluster_colors[i % len(cluster_colors)] for i, g in enumerate(chr_cluster_order)}
group_palette["CC"] = CC_COLOR

fig_w = max(18, 2.2 * len(modality_order))
fig_h = 6
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

sns.violinplot(
    data=plot_df,
    x="modality",
    y="PC1",
    hue="group",
    order=modality_order,
    hue_order=group_order,
    palette=group_palette,
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    dodge=True,
    width=0.8,
    ax=ax
)

ax.set_xlim(-0.55, len(modality_order) - 0.45)

# --- manual black points centered inside each dodged violin ---
rng = np.random.default_rng(0)

n_hue = len(group_order)
violin_width = 0.8                      # must match sns.violinplot(width=0.65)
sub_width = violin_width / n_hue         # width allotted to each group within a modality
jitter_scale = sub_width * 0.28          # small jitter within each subgroup

for i, modality in enumerate(modality_order):
    for j, group in enumerate(group_order):
        vals = plot_df.loc[
            (plot_df["modality"] == modality) & (plot_df["group"] == group),
            "PC1"
        ].to_numpy()

        if len(vals) == 0:
            continue

        # exact center of this group's violin within this modality
        center = i - violin_width / 2 + (j + 0.5) * sub_width

        # small symmetric jitter around that center
        x = center + rng.uniform(-jitter_scale, jitter_scale, size=len(vals))

        ax.scatter(
            x,
            vals,
            color="black",
            s=10,
            alpha=0.22,
            zorder=3,
            linewidths=0
        )

# Legend cleanup (keep one)
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:len(group_order)],
    labels[:len(group_order)],
    title="Group",
    frameon=False,
    loc="upper right"
)

for x in np.arange(0.5, len(modality_order), 1.0):
    ax.axvline(x, linewidth=0.8, alpha=0.25)

ax.set_xlabel("Modality", fontsize=16)
ax.set_ylabel("PC1 score (CHR-fitted PCA)", fontsize=16)
ax.set_title("PC1 Distributions by Modality (CHR integrated clusters + CC)", fontsize=18, pad=14)
ax.tick_params(axis="both", labelsize=14)
ax.grid(axis="y", alpha=0.15)
sns.despine(ax=ax)

plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=13)
plt.setp(ax.get_yticklabels(), fontsize=13)

fig.tight_layout()
fig.savefig(os.path.join(out_dir, "test_ALL_modalities_PC1_singleplot_violin_CHR_vs_CC.svg"), dpi=300, bbox_inches="tight")
plt.show()

# --------------------------
# B) GLOBAL PCA PC1 across ALL modalities/features: CHR clusters + CC
# --------------------------

# CHR merged wide
dfs_chr = []
for modality, df in dict_final.items():
    tmp = df.copy()
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})
    dfs_chr.append(tmp)

merged_chr = dfs_chr[0]
for d in dfs_chr[1:]:
    merged_chr = merged_chr.merge(d, on="src_subject_id", how="inner")

print(f"\n[GLOBAL PCA] CHR subjects after inner-join: {merged_chr.shape[0]}")
print(f"[GLOBAL PCA] CHR merged features: {merged_chr.shape[1] - 1}")

# CC merged wide
dfs_cc = []
for modality, df in dict_final_cc_test.items():
    tmp = df.copy()
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})
    dfs_cc.append(tmp)

merged_cc = dfs_cc[0]
for d in dfs_cc[1:]:
    merged_cc = merged_cc.merge(d, on="src_subject_id", how="inner")

print(f"[GLOBAL PCA] CC subjects after inner-join: {merged_cc.shape[0]}")
print(f"[GLOBAL PCA] CC merged features: {merged_cc.shape[1] - 1}")

# Align CHR test labels to merged CHR IDs
chr_clusters_raw = merged_chr["src_subject_id"].astype(str).map(test_final_labels_by_subject_id)
if np.any(pd.isna(chr_clusters_raw)):
    missing = merged_chr.loc[pd.isna(chr_clusters_raw), "src_subject_id"].head(5).tolist()
    raise ValueError(f"Missing test integrated labels for merged CHR IDs. Examples: {missing}")
chr_clusters = chr_clusters_raw.astype(str).to_numpy()

# Strict CC feature alignment to CHR merged features
chr_feature_cols = [c for c in merged_chr.columns if c != "src_subject_id"]
cc_feature_cols = [c for c in merged_cc.columns if c != "src_subject_id"]
missing_in_cc = [c for c in chr_feature_cols if c not in cc_feature_cols]
if missing_in_cc:
    raise ValueError(f"CC missing {len(missing_in_cc)} global features. Example: {missing_in_cc[:10]}")

X_chr_global = merged_chr[chr_feature_cols].values
X_cc_global = merged_cc.reindex(columns=chr_feature_cols).values

# CHR-fitted global PCA
Xg_scaler = StandardScaler()
Xg_chr_z = Xg_scaler.fit_transform(X_chr_global)
Xg_cc_z = Xg_scaler.transform(X_cc_global)

pca_global = PCA(n_components=1, random_state=0).fit(Xg_chr_z)
pc1_chr_global = pca_global.transform(Xg_chr_z)[:, 0]
pc1_cc_global = pca_global.transform(Xg_cc_z)[:, 0]
evr1_global = float(pca_global.explained_variance_ratio_[0])
print(f"[GLOBAL PCA] CHR-fitted PC1 EVR: {evr1_global:.2%}")

global_df = pd.concat([
    pd.DataFrame({"group": chr_clusters, "PC1_global": pc1_chr_global, "cohort": "CHR"}),
    pd.DataFrame({"group": "CC", "PC1_global": pc1_cc_global, "cohort": "CC"})
], ignore_index=True)

global_group_order = chr_cluster_order + ["CC"]

fig2, ax2 = plt.subplots(figsize=(10.5, 6.5))

sns.violinplot(
    data=global_df,
    x="group",
    y="PC1_global",
    hue="group",
    order=global_group_order,
    hue_order=global_group_order,
    palette=group_palette,
    dodge=False,
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    legend=False,
    ax=ax2
)

sns.stripplot(
    data=global_df,
    x="group",
    y="PC1_global",
    order=global_group_order,
    color="black",
    size=3,
    jitter=0.22,
    alpha=0.30,
    ax=ax2
)

med = global_df.groupby("group")["PC1_global"].median()
xpos = np.arange(len(global_group_order))
ax2.scatter(xpos, [med.loc[g] for g in global_group_order], s=180, marker="_", linewidths=3, color="black", zorder=4)

counts = global_df["group"].value_counts()
ymin, ymax = ax2.get_ylim()
y_annot = ymin + 0.04 * (ymax - ymin)
for i, g in enumerate(global_group_order):
    ax2.text(i, y_annot, f"n={int(counts.get(g, 0))}", ha="center", va="top", fontsize=16)

ax2.set_xlabel("Group", fontsize=20)
ax2.set_ylabel("Global PC1 score (CHR-fitted)", fontsize=20)
ax2.set_title("Global PC1 Across All Features and Modalities (CHR clusters + CC)", fontsize=20, pad=14)
ax2.grid(axis="y", alpha=0.15)
ax2.tick_params(axis="x", labelsize=16)
ax2.tick_params(axis="y", labelsize=16)
sns.despine(ax=ax2)

fig2.tight_layout()
fig2.savefig(os.path.join(out_dir, "test_GLOBAL_all_modalities_PC1_violin_CHR_vs_CC.svg"), dpi=300, bbox_inches="tight")
plt.show()


### Differences in categorical variables - individual labels

In [ ]:
def add_metadata_and_clusters(modality_data, data_full, mod_num):
    """
    Merge cluster labels into full metadata using src_subject_id from the
    modality dataframe that was clustered. The notebook calls this function
    once per modality, passing dict_final[modality].
    """
    clusters = pd.Series(labels_test_modalities[mod_num]).reset_index(drop=True)

    if isinstance(modality_data, pd.DataFrame):
        modality_df = modality_data
    elif isinstance(modality_data, dict):
        if len(modality_data) == 0:
            raise ValueError("modality_data is empty; cannot extract subject IDs.")
        first_modality = next(iter(modality_data))
        modality_df = modality_data[first_modality]
    else:
        raise TypeError(
            "modality_data must be a pandas DataFrame or a dict of modality DataFrames; "
            f"got {type(modality_data).__name__}."
        )

    if modality_df.empty:
        raise ValueError("modality_data is empty; cannot extract subject IDs.")
    if "src_subject_id" not in modality_df.columns:
        raise KeyError("modality_data must include a 'src_subject_id' column.")

    subj_ids = modality_df["src_subject_id"].reset_index(drop=True)

    # Sanity check: should match cluster array length
    if len(subj_ids) != len(clusters):
        print(f"Warning: {len(subj_ids)} subject IDs vs {len(clusters)} cluster labels.")
        min_len = min(len(subj_ids), len(clusters))
        subj_ids = subj_ids.iloc[:min_len]
        clusters = clusters.iloc[:min_len]
        print(f"Trimmed both to {min_len} entries to align.")

    cluster_df = pd.DataFrame({
        "src_subject_id": subj_ids,
        "Cluster": clusters,
    })

    merged = pd.merge(data_full, cluster_df, on="src_subject_id", how="left")

    print(f"Merged clusters for {merged['Cluster'].notna().sum()} subjects (out of {len(merged)} total).")
    return merged

def chi_square_comparison(df, group_col, label_col, title_prefix):
    """
    Perform chi-square test and plot grouped bar chart.
    """
    # Drop missing values
    df = df.dropna(subset=[group_col, label_col])

    # Make a copy for plotting / stats so we can safely relabel
    df_plot = df.copy()

    # Rename 'HC' -> 'CC' only for phenotype (for plotting and stats)
    if label_col == 'phenotype':
        df_plot[label_col] = df_plot[label_col].replace({'HC': 'CC'})

    # 1. Summarize counts
    summary_df = (
        df_plot.groupby([group_col, label_col])
        .size()
        .reset_index(name='n')
    )

    # 2. Chi-square test
    tbl = pd.crosstab(df_plot[group_col], df_plot[label_col])
    chi2, pval, dof, expected = chi2_contingency(tbl)
    pval = round(pval, 3)

    # 3. Grouped bar chart
    plt.figure(figsize=(8, 6))
    sns.barplot(
        data=summary_df,
        x=group_col,
        y='n',
        hue=label_col,
        dodge=True
    )
    plt.title(f"{title_prefix}\n(Chi-square p = {pval})")
    plt.xlabel("Cluster")
    plt.ylabel("Count")
    sns.despine()
    plt.tight_layout()
    plt.show()


mod_num = 0
for modality in dict_final.keys():
    print(f"\n=== Analyzing categorical differences for modality: {modality} ===")

    # Merge cluster labels into full data
    df = add_metadata_and_clusters(dict_final[modality], test_data, mod_num)

    # Compare by phenotype (CHR vs CC)
    if 'phenotype' in df.columns:
        chi_square_comparison(
            df=df,
            group_col='Cluster',
            label_col='phenotype',
            title_prefix=f"Comparison of CHR vs CC per Subgroup",
        )

    # Compare by site
    if 'Site' in df.columns:
        chi_square_comparison(
            df=df,
            group_col='Cluster',
            label_col='Site',
            title_prefix=f"Comparison of Site Distribution per Subgroup",
        )

    # Optional: extend for other categorical variables
    for col in ['sips_bips_scr_lifetime', 'sips_aps_scr_lifetime', 'sips_grd_scr_lifetime']:
        if col in df.columns:
            chi_square_comparison(
                df=df,
                group_col='Cluster',
                label_col=col,
                title_prefix=f"Comparison of {col} per Subgroup",
            )
    mod_num = mod_num + 1




### Mapping modalities -> final

In [ ]:
import numpy as np
import pandas as pd

new_test_labels_by_modality = {}

for i, modality in enumerate(modality_names):
    df = dict_final[modality]
    labels = labels_test_modalities[i]

    # Validate alignment
    if len(labels) != len(df):
        raise ValueError(
            f"Length mismatch for {modality}: labels={len(labels)} vs df={len(df)}"
        )

    labels_s = pd.Series(labels, index=df.index)

    uniq = labels_s.dropna().unique()
    if len(uniq) < 2:
        continue

    # Features only (avoid double mean of empty/invalid frames)
    X = df.drop(columns=['src_subject_id'], errors='ignore')
    if X.shape[1] == 0:
        raise ValueError(f"{modality}: no feature columns after dropping src_subject_id")

    # Cluster-specific means
    cluster_means = labels_s.groupby(labels_s).apply(
        lambda s: X.loc[s.index].to_numpy().mean()
    )

    # Choose the "highest mean" cluster as the high_cluster
    high_cluster = cluster_means.sort_values(ascending=False).index[0]

    # Map to severity labels (your modality-specific inversion kept)
    if modality in ("Functioning", "Cognition"):
        # higher score means *lower* severity (per your rule)
        new_labels = np.where(labels_s == high_cluster, "low_severity", "high_severity")
    else:
        new_labels = np.where(labels_s == high_cluster, "high_severity", "low_severity")

    new_test_labels_by_modality[modality] = new_labels.tolist()


In [ ]:
domain_map(
    new_labels_by_modality=new_test_labels_by_modality,
    final_labels=labels_test_final,
    stage_order=["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition"],
    final_name="Integrated",
    top_token="high_severity",          # now HIGH is on top
    bottom_token="low_severity",
    invert_final=False,
    color_for_top_final="#005341",      # top-like final ribbons
    color_for_bottom_final="#B3D9D5",   # bottom-like final ribbons
    add_gap_in_final=True,
    gap_weight=20,
    plots_dir=plots_dir,
    save_file_name = "Parcats_by_final_test.pdf"
)

In [ ]:
import pandas as pd

stage_order = ["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition"]

# Pick subject_id vector from one modality (must share row order with labels)
subject_ids_test = dict_final[stage_order[0]]['src_subject_id'].astype(str).reset_index(drop=True)

# Build the path dataframe
df_paths_test = pd.DataFrame({"src_subject_id": subject_ids_test})
for stage in stage_order:
    df_paths_test[stage] = pd.Series(new_test_labels_by_modality[stage]).astype(str).reset_index(drop=True)

df_paths_test["final"] = pd.Series(labels_test_final).astype(str).reset_index(drop=True)

# Sanity checks
N = len(df_paths_test)
assert all(len(new_test_labels_by_modality[s]) == N for s in stage_order), "Label lengths don't match subject_ids length"
assert len(labels_test_final) == N, "Final labels length doesn't match subject_ids length"

df_paths_test


In [ ]:
# Use the shared canonical stream format: Domain=label → ... → final=label.
# This must match the discovery-side stream labels before comparing presence or mappings.
stream_summary_test_result = summarize_streams(df_paths_test, stage_order, top_k=100, sample_ids=12)
stream_summary_test = (
    stream_summary_test_result[0]
    if isinstance(stream_summary_test_result, tuple)
    else stream_summary_test_result
)

if not isinstance(stream_summary_test, pd.DataFrame):
    raise TypeError(
        "summarize_streams must return a DataFrame or a tuple whose first item is a DataFrame; "
        f"got {type(stream_summary_test).__name__}."
    )

stream_summary_test


### Comparison between discovery and test in cluster mapping

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import jensenshannon

# Optional (for sankey). If plotly isn't installed, sankey functions will gracefully fail.
try:
    import plotly.graph_objects as go
    _HAS_PLOTLY = True
except Exception:
    _HAS_PLOTLY = False


ARROW_PAT = re.compile(r"\s*(?:→|->)\s*")

def parse_stream(stream_str):
    """
    Parse stream like:
      "Psychoticism=low_severity → Detachment=high_severity → ... → final=1"
    into ordered list of (domain, label).
    """
    if pd.isna(stream_str):
        return []
    parts = [p.strip() for p in ARROW_PAT.split(str(stream_str).strip()) if p.strip()]
    out = []
    for p in parts:
        if "=" in p:
            dom, lab = p.split("=", 1)
            out.append((dom.strip(), lab.strip()))
        else:
            out.append((p.strip(), "<NA>"))
    return out


def infer_stage_order(df, stream_col="stream"):
    """
    Infer stage order from the first non-null stream.
    Assumes all streams follow the same domain order.
    """
    s = df[stream_col].dropna().astype(str)
    if s.empty:
        raise ValueError("No streams found to infer stage order.")
    path = parse_stream(s.iloc[0])
    return [d for d, _ in path]


def _normalize(v, eps=1e-12):
    v = np.asarray(v, dtype=float)
    s = v.sum()
    if s <= 0:
        return np.ones_like(v) / max(1, len(v))
    return v / (s + eps)


# ------------------------------------------------------------
# 1) Prefix-tree comparison: compare P(next | prefix)
# ------------------------------------------------------------

def build_prefix_next(df, stream_col="stream", n_col="n"):
    """
    Returns:
      prefix_next: dict[prefix_tuple_of_(domain,label)] -> dict[next_token_(domain,label)|<END>] -> weight
      prefix_mass: dict[prefix] -> total weight passing through prefix
    Where prefix is a tuple of (domain,label) tokens, e.g.:
      prefix = (("Psychoticism","low_severity"), ("Detachment","high_severity"))
    and next token is the next (domain,label) or "<END>".
    """
    prefix_next = {}
    prefix_mass = {}

    for _, row in df.iterrows():
        w = float(row.get(n_col, 1.0))
        path = parse_stream(row[stream_col])
        if not path:
            continue

        # For each prefix (including empty prefix), record what comes next
        for i in range(len(path)):
            prefix = tuple(path[:i])  # empty prefix for i=0
            nxt = path[i]             # next token
            prefix_mass[prefix] = prefix_mass.get(prefix, 0.0) + w
            d = prefix_next.setdefault(prefix, {})
            d[nxt] = d.get(nxt, 0.0) + w

        # terminal transition from full path to END
        full_prefix = tuple(path)
        prefix_mass[full_prefix] = prefix_mass.get(full_prefix, 0.0) + w
        d = prefix_next.setdefault(full_prefix, {})
        d["<END>"] = d.get("<END>", 0.0) + w

    return prefix_next, prefix_mass


def compare_prefix_structure(df_disc, df_test, stream_col="stream", n_col="n", eps=1e-12):
    """
    For each prefix, compare the conditional distribution over next tokens:
        P_disc(next | prefix)  vs  P_test(next | prefix)

    Returns DataFrame with:
      - prefix_str
      - depth
      - js_next (Jensen–Shannon distance on next-step distributions)
      - mass_disc / mass_test (how much data passes through prefix; useful for weighting but not "size-equality")
      - top_next_disc / top_next_test (most likely next step)
      - support_next_overlap (Jaccard on next-token supports)
    """
    pn_d, pm_d = build_prefix_next(df_disc, stream_col, n_col)
    pn_t, pm_t = build_prefix_next(df_test, stream_col, n_col)

    prefixes = set(pn_d.keys()) | set(pn_t.keys())

    rows = []
    for pref in prefixes:
        nd = pn_d.get(pref, {})
        nt = pn_t.get(pref, {})

        keys = set(nd.keys()) | set(nt.keys())
        # Align next-token vectors
        vd = np.array([nd.get(k, 0.0) for k in keys], dtype=float)
        vt = np.array([nt.get(k, 0.0) for k in keys], dtype=float)

        pd_ = _normalize(vd, eps=eps)
        pt_ = _normalize(vt, eps=eps)

        js = float(jensenshannon(pd_ + eps, pt_ + eps, base=2.0))

        # Next-token support overlap (presence/absence, structure)
        supp_d = {k for k, v in nd.items() if v > 0}
        supp_t = {k for k, v in nt.items() if v > 0}
        supp_j = len(supp_d & supp_t) / max(1, len(supp_d | supp_t))

        # Most likely next token in each
        top_d = max(nd.items(), key=lambda x: x[1])[0] if nd else None
        top_t = max(nt.items(), key=lambda x: x[1])[0] if nt else None

        def tok_str(tok):
            if tok == "<END>":
                return "<END>"
            if tok is None:
                return "<NONE>"
            return f"{tok[0]}={tok[1]}"

        prefix_str = " → ".join([f"{d}={l}" for d, l in pref]) if pref else "<START>"
        rows.append({
            "prefix_str": prefix_str,
            "depth": len(pref),
            "js_next": js,
            "mass_disc": pm_d.get(pref, 0.0),
            "mass_test": pm_t.get(pref, 0.0),
            "top_next_disc": tok_str(top_d),
            "top_next_test": tok_str(top_t),
            "support_next_jaccard": supp_j,
            "prefix_exists_in_disc": pref in pn_d,
            "prefix_exists_in_test": pref in pn_t,
        })

    out = pd.DataFrame(rows)
    # A useful default sorting: prioritize structurally-different AND commonly-used prefixes
    out["mass_min"] = np.minimum(out["mass_disc"], out["mass_test"])
    out = out.sort_values(["mass_min", "js_next"], ascending=[False, False]).reset_index(drop=True)
    return out


def plot_top_prefix_differences(prefix_report, top_n=20, min_depth=1):
    """
    Barh plot of top prefixes by JS(next) after filtering.
    """
    d = prefix_report[prefix_report["depth"] >= min_depth].copy()
    d = d.sort_values("js_next", ascending=False).head(top_n)

    plt.figure(figsize=(11, max(4, 0.35 * len(d))))
    y = np.arange(len(d))[::-1]
    plt.barh(y, d["js_next"].iloc[::-1].to_numpy())
    plt.yticks(y, d["prefix_str"].iloc[::-1].to_list(), fontsize=9)
    plt.xlabel("Jensen–Shannon distance of P(next | prefix)")
    plt.title("Most structurally different prefixes (next-step rule)")
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 2) Full mapping to final: compare P(final | modalities-prefix)
# ------------------------------------------------------------

def final_mapping_table(df, stream_col="stream", n_col="n", final_domain="final"):
    """
    Builds a table over modality-prefixes (everything up to but excluding final)
    with P(final=label | prefix) computed within each dataset.

    Returns DataFrame:
      prefix_str, final_label, weight
    and also a pivoted table of P(final=...) by prefix.
    """
    rows = []
    for _, row in df.iterrows():
        w = float(row.get(n_col, 1.0))
        path = parse_stream(row[stream_col])
        if not path:
            continue

        # split into prefix (before final) and final token
        final_tokens = [t for t in path if t[0] == final_domain]
        if not final_tokens:
            # If final isn't explicitly present, skip
            continue
        final_tok = final_tokens[-1]  # in case of duplicates, take last
        final_label = final_tok[1]

        # prefix = all tokens before the final token position (first occurrence)
        # more robust: take all tokens except the final-domain token(s)
        prefix = tuple([t for t in path if t[0] != final_domain])

        prefix_str = " → ".join([f"{d}={l}" for d, l in prefix]) if prefix else "<NO_MODALITIES>"
        rows.append({"prefix_str": prefix_str, "final_label": final_label, "w": w})

    long = pd.DataFrame(rows)
    if long.empty:
        return long, pd.DataFrame()

    # Compute conditional probabilities P(final_label | prefix)
    grp = long.groupby(["prefix_str", "final_label"], dropna=False)["w"].sum().reset_index()
    totals = grp.groupby("prefix_str")["w"].sum().reset_index().rename(columns={"w": "w_total"})
    grp = grp.merge(totals, on="prefix_str", how="left")
    grp["p_final_given_prefix"] = grp["w"] / grp["w_total"]

    pivot = grp.pivot_table(index="prefix_str", columns="final_label", values="p_final_given_prefix", fill_value=0.0)
    return grp, pivot


def compare_final_mapping(df_disc, df_test, stream_col="stream", n_col="n", final_domain="final"):
    """
    Compare P(final | modalities) between discovery and test.
    Returns a table with per-prefix deltas per final label plus summary metrics.
    """
    long_d, piv_d = final_mapping_table(df_disc, stream_col, n_col, final_domain)
    long_t, piv_t = final_mapping_table(df_test, stream_col, n_col, final_domain)

    empty_out = pd.DataFrame(columns=["prefix_str", "js_final_given_prefix", "w_disc", "w_test", "w_min"])
    if piv_d.empty or piv_t.empty:
        return empty_out, {"note": "No final mapping found (missing final tokens?)", "num_prefixes_union": 0}

    # align
    idx = sorted(set(piv_d.index) | set(piv_t.index))
    cols = sorted(set(piv_d.columns) | set(piv_t.columns))
    A = piv_d.reindex(index=idx, columns=cols).fillna(0.0)
    B = piv_t.reindex(index=idx, columns=cols).fillna(0.0)

    # Delta per final label
    delta = (B - A)
    out = delta.copy()
    out.columns = [f"delta_final={c}" for c in out.columns]
    out.insert(0, "prefix_str", out.index)

    # A per-prefix summary: JS distance between final distributions for that prefix
    js_list = []
    for i in range(A.shape[0]):
        p = _normalize(A.iloc[i].to_numpy())
        q = _normalize(B.iloc[i].to_numpy())
        js_list.append(float(jensenshannon(p + 1e-12, q + 1e-12, base=2.0)))
    out["js_final_given_prefix"] = js_list

    # Add weights: how common the prefix is (within each dataset)
    wD = long_d.groupby("prefix_str")["w"].sum() if not long_d.empty else pd.Series(dtype=float)
    wT = long_t.groupby("prefix_str")["w"].sum() if not long_t.empty else pd.Series(dtype=float)
    out["w_disc"] = out["prefix_str"].map(wD).fillna(0.0)
    out["w_test"] = out["prefix_str"].map(wT).fillna(0.0)
    out["w_min"] = np.minimum(out["w_disc"], out["w_test"])

    # Sort by (common prefixes) then by biggest JS shift in final mapping
    out = out.sort_values(["w_min", "js_final_given_prefix"], ascending=[False, False]).reset_index(drop=True)

    # Global summary metric: weighted average JS over prefixes (weights = w_min)
    w = out["w_min"].to_numpy()
    if w.sum() > 0:
        global_js = float(np.sum(out["js_final_given_prefix"].to_numpy() * w) / w.sum())
    else:
        global_js = float(out["js_final_given_prefix"].mean())

    metrics = {
        "num_prefixes_union": len(out),
        "weighted_js_final_given_prefix": global_js,
        "final_labels": cols,
    }
    return out, metrics


def plot_top_final_mapping_shifts(final_cmp, top_n=20):
    required = {"prefix_str", "js_final_given_prefix"}
    if final_cmp is None or final_cmp.empty:
        print("No final-mapping shifts to plot.")
        return None
    missing = required - set(final_cmp.columns)
    if missing:
        print(f"No final-mapping shifts to plot; missing columns: {sorted(missing)}")
        return None
    d = final_cmp.head(top_n).copy()
    plt.figure(figsize=(11, max(4, 0.35 * len(d))))
    y = np.arange(len(d))[::-1]
    plt.barh(y, d["js_final_given_prefix"].iloc[::-1].to_numpy())
    plt.yticks(y, d["prefix_str"].iloc[::-1].to_list(), fontsize=9)
    plt.xlabel("JS distance between P(final | prefix) in test vs discovery")
    plt.title("Prefixes with biggest changes in final mapping")
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 3) Full stream presence + rank overlap
# ------------------------------------------------------------

def stream_presence_and_topk(df_disc, df_test, stream_col="stream", n_col="n", topk=30):
    A = set(df_disc[stream_col].dropna().astype(str))
    B = set(df_test[stream_col].dropna().astype(str))

    presence = {
        "unique_streams_disc": len(A),
        "unique_streams_test": len(B),
        "stream_jaccard_presence": len(A & B) / max(1, len(A | B)),
        "coverage_disc_in_test": len(A & B) / max(1, len(A)),
        "coverage_test_in_disc": len(A & B) / max(1, len(B)),
    }

    pD = df_disc.groupby(stream_col)[n_col].sum().sort_values(ascending=False)
    pT = df_test.groupby(stream_col)[n_col].sum().sort_values(ascending=False)
    pD = pD / pD.sum()
    pT = pT / pT.sum()

    topD = set(pD.index[: min(topk, len(pD))].astype(str))
    topT = set(pT.index[: min(topk, len(pT))].astype(str))

    presence[f"top{topk}_jaccard_by_rank"] = len(topD & topT) / max(1, len(topD | topT))
    presence["top_disc_only"] = sorted(list(topD - topT))[:10]
    presence["top_test_only"] = sorted(list(topT - topD))[:10]
    return presence, pD, pT


# ------------------------------------------------------------
# 4) Sankey (full structure) per dataset
# ------------------------------------------------------------

def sankey_from_streams(df, stream_col="stream", n_col="n", max_edges=200):
    """
    Build a Sankey graph from full streams.
    Nodes are stage-specific label tokens: f"{domain}={label}".
    Edges connect consecutive tokens. We prune to max_edges by weight.
    """
    if not _HAS_PLOTLY:
        raise RuntimeError("Plotly not installed; cannot draw sankey.")

    edge_w = {}
    for _, row in df.iterrows():
        w = float(row.get(n_col, 1.0))
        path = parse_stream(row[stream_col])
        toks = [f"{d}={l}" for d, l in path]
        for a, b in zip(toks[:-1], toks[1:]):
            edge_w[(a, b)] = edge_w.get((a, b), 0.0) + w

    # prune
    edges = sorted(edge_w.items(), key=lambda x: -x[1])[:max_edges]
    nodes = {}
    def nid(x):
        if x not in nodes:
            nodes[x] = len(nodes)
        return nodes[x]

    src, tgt, val = [], [], []
    for (a, b), w in edges:
        src.append(nid(a))
        tgt.append(nid(b))
        val.append(w)

    labels = [None] * len(nodes)
    for k, i in nodes.items():
        labels[i] = k

    fig = go.Figure(data=[go.Sankey(
        node=dict(label=labels, pad=12, thickness=12),
        link=dict(source=src, target=tgt, value=val),
    )])
    return fig


# ------------------------------------------------------------
# Master runner
# ------------------------------------------------------------

def full_structure_report(stream_summary, stream_summary_test, stream_col="stream", n_col="n", topk=30, final_domain="final"):
    # Presence + top-k overlap
    presence, pD, pT = stream_presence_and_topk(stream_summary, stream_summary_test, stream_col, n_col, topk=topk)

    # Prefix-tree structural differences
    prefix_report = compare_prefix_structure(stream_summary, stream_summary_test, stream_col, n_col)

    # Full mapping to final
    final_cmp, final_metrics = compare_final_mapping(stream_summary, stream_summary_test, stream_col, n_col, final_domain)

    return {
        "presence_metrics": presence,
        "p_stream_disc": pD,
        "p_stream_test": pT,
        "prefix_report": prefix_report,
        "final_mapping_compare": final_cmp,
        "final_mapping_metrics": final_metrics,
    }


# -----------------------------
# Example usage
# -----------------------------
# rep = full_structure_report(stream_summary, stream_summary_test, topk=30, final_domain="final")
# rep["presence_metrics"]
# rep["prefix_report"].head(30)  # structural differences by prefix
# plot_top_prefix_differences(rep["prefix_report"], top_n=20, min_depth=1)
#
# rep["final_mapping_metrics"]
# rep["final_mapping_compare"].head(30)
# plot_top_final_mapping_shifts(rep["final_mapping_compare"], top_n=20)
#
# # Sankey (if plotly installed)
# if _HAS_PLOTLY:
#     figD = sankey_from_streams(stream_summary, max_edges=250)
#     figD.update_layout(title_text="Discovery stream structure (Sankey)")
#     figD.show()
#     figT = sankey_from_streams(stream_summary_test, max_edges=250)
#     figT.update_layout(title_text="Test stream structure (Sankey)")
#     figT.show()


In [ ]:
def summarize_streams_for_comparison(df_paths, stage_order, top_k=100, sample_ids=12, final_col="final"):
    """Build canonical stream labels for discovery/test comparison."""
    group_cols = list(stage_order) + ([final_col] if final_col in df_paths.columns else [])
    total = len(df_paths)

    g = (
        df_paths
        .groupby(group_cols, dropna=False)
        .agg(
            n=("src_subject_id", "size"),
            example_ids=("src_subject_id", lambda x: list(x.head(sample_ids))),
        )
        .reset_index()
    )
    g["pct"] = (g["n"] / total * 100).round(2) if total else 0.0

    def make_stream_label(row):
        parts = [f"{stage}={row[stage]}" for stage in stage_order]
        if final_col in row.index:
            parts.append(f"{final_col}={row[final_col]}")
        return " → ".join(parts)

    g["stream"] = g.apply(make_stream_label, axis=1)
    g = g.sort_values("n", ascending=False).head(top_k)

    cols = ["n", "pct", "stream", "example_ids"] + group_cols
    return g[cols]

# Rebuild both sides here so the comparison cannot reuse stale stream labels
# from an older imported Utils.summarize_streams implementation.
stream_summary = summarize_streams_for_comparison(df_paths, stage_order, top_k=100, sample_ids=12)
stream_summary_test = summarize_streams_for_comparison(df_paths_test, stage_order, top_k=100, sample_ids=12)

rep = full_structure_report(stream_summary, stream_summary_test, topk=100, final_domain="final")

rep["presence_metrics"]


In [ ]:
# Where does the pathway grammar differ most?
rep["prefix_report"].head(25)
plot_top_prefix_differences(rep["prefix_report"], top_n=25, min_depth=1)


In [ ]:
# Does the multimodal signature map to the same final label?
rep["final_mapping_metrics"]
rep["final_mapping_compare"].head(25)
plot_top_final_mapping_shifts(rep["final_mapping_compare"], top_n=25)


In [ ]:
import pandas as pd
import numpy as np

def all_streams_table(stream_summary, stream_summary_test, stream_col="stream", n_col="n"):
    # Aggregate in case there are duplicate stream rows
    disc = (stream_summary
            .groupby(stream_col, dropna=False)[n_col]
            .sum()
            .rename("n_disc")
            .to_frame())

    test = (stream_summary_test
            .groupby(stream_col, dropna=False)[n_col]
            .sum()
            .rename("n_test")
            .to_frame())

    # Outer join gives union of streams
    tbl = disc.join(test, how="outer").fillna(0)

    # Add proportions within each dataset
    N_disc = tbl["n_disc"].sum()
    N_test = tbl["n_test"].sum()

    tbl["p_disc"] = tbl["n_disc"] / N_disc if N_disc > 0 else np.nan
    tbl["p_test"] = tbl["n_test"] / N_test if N_test > 0 else np.nan

    # Helpful comparisons
    tbl["delta_p"] = tbl["p_test"] - tbl["p_disc"]
    tbl["abs_delta_p"] = tbl["delta_p"].abs()
    tbl["log2_fc"] = np.log2((tbl["p_test"] + 1e-12) / (tbl["p_disc"] + 1e-12))

    # Make stream a real column, sort by biggest shift
    tbl = tbl.reset_index().rename(columns={stream_col: "stream"})
    tbl = tbl.sort_values("abs_delta_p", ascending=False).reset_index(drop=True)

    return tbl

# Usage
streams_tbl = all_streams_table(stream_summary, stream_summary_test)
streams_tbl.head(100)


# Post-Pipeline Paper 1 Analyses

These cells bring the post-pipeline analyses from `PrepareData_demtable_paper1.Rmd` into the clinical notebook: demographic/sample tables, subgroup difference summaries, mixed heatmaps, site/recruitment checks, and included-vs-excluded diagnostics. Outputs are written under `plots_dir/post_pipeline_paper1` so they stay with the pipeline run.


In [ ]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

POST_ANALYSIS_DIR = Path(plots_dir) / "post_pipeline_paper1"
POST_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

LABELS_DIR = Path("path/to/results/study_1/release/Labels")
DISCOVERY_LABELS_FILE = LABELS_DIR / "discovery_labels_clin_multiclust.csv"
TEST_LABELS_FILE = LABELS_DIR / "test_labels_clin_multiclust.csv"
LABELS_DIR.mkdir(parents=True, exist_ok=True)

DICTIONARY_DIR_DIFF = "path/to/project/Feature selection/Complete_dictionary_differences.xlsx"
SUBGROUP_VARS = ["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition", "final"]


def _subject_ids_from_modality(data_by_modality, stage_order, subject_id_column="src_subject_id"):
    first_stage = stage_order[0]
    if first_stage not in data_by_modality:
        raise KeyError(f"Missing {first_stage} in modality data.")
    return data_by_modality[first_stage][subject_id_column].astype(str).reset_index(drop=True)


def build_label_frame_from_current_objects(sample):
    """Fallback only: rebuild labels if the saved release label CSV is unavailable."""
    if sample == "discovery":
        if "df_paths" in globals():
            label_df = df_paths.copy()
        else:
            subject_ids = _subject_ids_from_modality(final_metrics["data"], stage_order)
            label_df = pd.DataFrame({"src_subject_id": subject_ids})
            for stage in stage_order:
                label_df[stage] = pd.Series(new_labels_by_modality[stage]).astype(str).reset_index(drop=True)
            label_df["final"] = pd.Series(final_metrics["final_labels"]).astype(str).reset_index(drop=True)
    elif sample == "test":
        if "df_paths_test" in globals():
            label_df = df_paths_test.copy()
        else:
            subject_ids = _subject_ids_from_modality(dict_final, stage_order)
            label_df = pd.DataFrame({"src_subject_id": subject_ids})
            for stage in stage_order:
                label_df[stage] = pd.Series(new_test_labels_by_modality[stage]).astype(str).reset_index(drop=True)
            label_df["final"] = pd.Series(labels_test_final).astype(str).reset_index(drop=True)
    else:
        raise ValueError("sample must be 'discovery' or 'test'")

    keep = ["src_subject_id"] + [c for c in SUBGROUP_VARS if c in label_df.columns]
    label_df = label_df[keep].copy()
    label_df["src_subject_id"] = label_df["src_subject_id"].astype(str)
    return label_df


def load_saved_label_frame(path, sample):
    """Match the Rmd: read the saved label CSVs produced by the final label-export cell."""
    if path.exists():
        label_df = pd.read_csv(path)
        source = str(path)
    else:
        print(f"WARNING: {path} does not exist; rebuilding {sample} labels from current notebook objects.")
        label_df = build_label_frame_from_current_objects(sample)
        label_df.to_csv(path, index=False)
        source = f"rebuilt and saved to {path}"

    required = ["src_subject_id"] + SUBGROUP_VARS
    missing = [col for col in required if col not in label_df.columns]
    if missing:
        raise KeyError(f"{sample} labels are missing required columns: {missing}. Source: {source}")

    label_df = label_df[required].copy()
    label_df["src_subject_id"] = label_df["src_subject_id"].astype(str)
    for col in SUBGROUP_VARS:
        label_df[col] = label_df[col].astype(str)
    print(f"Loaded {sample} labels from {source}: {label_df.shape}")
    return label_df


def _resolve_post_analysis_data():
    """Return the raw merged dataframe used by the Rmd as `merged_data`."""
    for name in ["merged_data", "data_all", "data"]:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame) and "src_subject_id" in obj.columns:
            print(f"Using `{name}` as the Rmd-style merged_data dataframe.")
            return obj.copy()

    merged_path = Path(
        "path/to/"
        "multiclust_data/merged_data.csv"
    )
    if merged_path.exists():
        print(f"Reloading Rmd-style merged_data from {merged_path}.")
        return pd.read_csv(merged_path)

    available = {
        name: type(globals().get(name)).__name__
        for name in ["merged_data", "data_all", "data"]
        if name in globals()
    }
    raise KeyError(
        "Could not find the Rmd-style merged_data dataframe with `src_subject_id`. "
        f"Available candidates: {available}"
    )


discovery_labels = load_saved_label_frame(DISCOVERY_LABELS_FILE, "discovery")
test_labels = load_saved_label_frame(TEST_LABELS_FILE, "test")

data_for_post = _resolve_post_analysis_data()
data_for_post["src_subject_id"] = data_for_post["src_subject_id"].astype(str)

# Match the Rmd's `data_finalSample = merged_data`, then sample assignment and filter.
data_final_sample = data_for_post.copy()
data_final_sample["sample"] = np.where(
    data_final_sample["src_subject_id"].isin(discovery_labels["src_subject_id"]),
    "Discovery",
    "Test",
)
data_final_sample = data_final_sample.loc[
    data_final_sample["src_subject_id"].isin(discovery_labels["src_subject_id"]) |
    data_final_sample["src_subject_id"].isin(test_labels["src_subject_id"])
].copy()

# Match the Rmd's inner merge: merge(discovery_labels, merged_data, by='src_subject_id').
Labels_data_disc = discovery_labels.merge(data_for_post, on="src_subject_id", how="inner")
Labels_data_test = test_labels.merge(data_for_post, on="src_subject_id", how="inner")

print("Discovery labels:", discovery_labels.shape, "merged:", Labels_data_disc.shape)
print("Test labels:", test_labels.shape, "merged:", Labels_data_test.shape)
print("Final sample:", data_final_sample.shape)
print("Post-analysis output directory:", POST_ANALYSIS_DIR)


## Shared Helpers

The Rmd used a custom R `demographic_table()` function. The helpers below reproduce the same analysis intent in Python: compare variables across grouping columns, report effect sizes, apply BH/FDR correction, and export notebook-friendly CSV tables plus figure files.


In [ ]:
try:
    from scipy.stats import chi2_contingency as _scipy_chi2_contingency
    from scipy.stats import kruskal as _scipy_kruskal
    _HAS_SCIPY_STATS = True
except Exception:
    _HAS_SCIPY_STATS = False
    _scipy_chi2_contingency = None
    _scipy_kruskal = None


def _bh_fdr(pvalues):
    p = pd.to_numeric(pd.Series(pvalues), errors="coerce")
    out = pd.Series(np.nan, index=p.index, dtype=float)
    valid = p.dropna()
    m = len(valid)
    if m == 0:
        return out
    order = np.argsort(valid.values)
    ranked = valid.values[order]
    adj = ranked * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    out.loc[valid.index[order]] = adj
    return out


def _load_dictionary_labels(dictionary_path=DICTIONARY_DIR_DIFF):
    try:
        d = pd.read_excel(dictionary_path)
    except Exception as exc:
        print(f"Could not read dictionary at {dictionary_path}: {exc}")
        return {}, []
    name_col = "ElementName" if "ElementName" in d.columns else d.columns[0]
    label_col = next((c for c in ["ElementDescription", "Description", "Label", "Variable", "VariableName"] if c in d.columns), name_col)
    labels = dict(zip(d[name_col].astype(str), d[label_col].astype(str)))
    ordered = [v for v in d[name_col].astype(str).tolist()]
    return labels, ordered


VARIABLE_LABELS, DICTIONARY_VARIABLE_ORDER = _load_dictionary_labels()


def _label_for_variable(variable):
    return VARIABLE_LABELS.get(str(variable), str(variable))


def _analysis_variables(df, comparison_cols=None, prefer_dictionary=True):
    comparison_cols = set(comparison_cols or [])
    blocked = {
        "src_subject_id", "sample", "phenotype", "Site", "sites", "chrrecruit",
        *SUBGROUP_VARS, *comparison_cols,
    }
    if prefer_dictionary and DICTIONARY_VARIABLE_ORDER:
        candidates = [v for v in DICTIONARY_VARIABLE_ORDER if v in df.columns]
    else:
        candidates = list(df.columns)
    return [c for c in candidates if c not in blocked]


def _is_binary_series(x):
    vals = pd.Series(x).dropna()
    if vals.empty:
        return False
    if pd.api.types.is_numeric_dtype(vals):
        u = set(pd.to_numeric(vals, errors="coerce").dropna().unique())
        return len(u) <= 2 and u.issubset({0, 1})
    return vals.astype(str).nunique(dropna=True) == 2


def _variable_type(x):
    vals = pd.Series(x).dropna()
    if vals.empty or vals.nunique(dropna=True) < 2:
        return None
    if _is_binary_series(vals):
        return "binary"
    numeric = pd.to_numeric(vals, errors="coerce")
    if numeric.notna().mean() >= 0.90 and numeric.nunique(dropna=True) > 2:
        return "continuous"
    return "categorical"


def _cramers_v_from_table(tab):
    arr = np.asarray(tab, dtype=float)
    n = arr.sum()
    if n == 0 or min(arr.shape) < 2:
        return np.nan
    expected = np.outer(arr.sum(axis=1), arr.sum(axis=0)) / n
    with np.errstate(divide="ignore", invalid="ignore"):
        stat = np.nansum((arr - expected) ** 2 / expected)
    denom = n * (min(arr.shape) - 1)
    return float(np.sqrt(stat / denom)) if denom > 0 else np.nan


def _chi_square_test(tab):
    arr = np.asarray(tab, dtype=float)
    if arr.shape[0] < 2 or arr.shape[1] < 2 or arr.sum() == 0:
        return np.nan, np.nan, np.nan
    if _HAS_SCIPY_STATS:
        stat, p, _, _ = _scipy_chi2_contingency(arr)
    else:
        expected = np.outer(arr.sum(axis=1), arr.sum(axis=0)) / arr.sum()
        with np.errstate(divide="ignore", invalid="ignore"):
            stat = np.nansum((arr - expected) ** 2 / expected)
        p = np.nan
    return float(stat), float(p), _cramers_v_from_table(arr)


def _rank_anova_or_kruskal(groups):
    clean_groups = [pd.to_numeric(pd.Series(g), errors="coerce").dropna().to_numpy() for g in groups]
    clean_groups = [g for g in clean_groups if len(g) > 0]
    if len(clean_groups) < 2:
        return np.nan, np.nan, np.nan
    if _HAS_SCIPY_STATS:
        stat, p = _scipy_kruskal(*clean_groups)
    else:
        values = np.concatenate(clean_groups)
        ranks = pd.Series(values).rank(method="average").to_numpy()
        labels = np.concatenate([[i] * len(g) for i, g in enumerate(clean_groups)])
        grand = ranks.mean()
        ss_between = sum((ranks[labels == i].mean() - grand) ** 2 * (labels == i).sum() for i in np.unique(labels))
        ss_total = ((ranks - grand) ** 2).sum()
        stat = ss_between / max(1, len(clean_groups) - 1)
        p = np.nan
    n = sum(len(g) for g in clean_groups)
    eta_like = float(stat / max(stat + n - len(clean_groups), 1e-12)) if np.isfinite(stat) else np.nan
    return float(stat), float(p), eta_like


def summarize_group_differences(df, comparisons, variables=None, output_prefix=None):
    rows = []
    comparisons = [c for c in comparisons if c in df.columns]
    variables = variables or _analysis_variables(df, comparison_cols=comparisons)

    for comp in comparisons:
        d = df.loc[df[comp].notna()].copy()
        groups = d[comp].astype(str)
        if groups.nunique(dropna=True) < 2:
            continue
        for variable in variables:
            if variable not in d.columns or variable == comp:
                continue
            vt = _variable_type(d[variable])
            if vt is None:
                continue

            sub = pd.DataFrame({"group": groups, "value": d[variable]}).dropna(subset=["value"])
            if sub["group"].nunique() < 2:
                continue

            if vt == "continuous":
                grouped = [g["value"] for _, g in sub.groupby("group", sort=True)]
                stat, p, effect = _rank_anova_or_kruskal(grouped)
                summary = sub.assign(value=pd.to_numeric(sub["value"], errors="coerce")).groupby("group")["value"].agg(["count", "mean", "std", "median"]).reset_index()
                test = "Kruskal-Wallis"
                effect_name = "eta2_H"
            else:
                tab = pd.crosstab(sub["group"].astype(str), sub["value"].astype(str))
                stat, p, effect = _chi_square_test(tab)
                summary = tab.reset_index()
                test = "Chi-square"
                effect_name = "cramers_v"

            rows.append({
                "comparison": comp,
                "variable": variable,
                "label": _label_for_variable(variable),
                "var_type": vt,
                "test": test,
                "statistic": stat,
                "p": p,
                "effect_size": effect,
                "effect_size_name": effect_name,
                "n": int(sub.shape[0]),
                "n_groups": int(sub["group"].nunique()),
                "summary": summary.to_json(orient="records"),
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["fdr"] = out.groupby("comparison")["p"].transform(_bh_fdr)
    out = out.sort_values(["comparison", "fdr", "p", "label"], na_position="last").reset_index(drop=True)
    if output_prefix:
        out.to_csv(POST_ANALYSIS_DIR / f"{output_prefix}_difference_summary_long.csv", index=False)
        compact = out.assign(
            cell=lambda x: x.apply(
                lambda r: f"ES={r['effect_size']:.3g}; FDR={r['fdr']:.3g}" if pd.notna(r["fdr"]) else "",
                axis=1,
            )
        ).pivot_table(index="label", columns="comparison", values="cell", aggfunc="first").reset_index()
        compact.to_csv(POST_ANALYSIS_DIR / f"{output_prefix}_difference_summary_compact.csv", index=False)
        display(compact)
    return out


def demographic_summary_table(df, comparison, variables=None, output_name=None):
    variables = variables or _analysis_variables(df, comparison_cols=[comparison])
    rows = []
    d = df.loc[df[comparison].notna()].copy()
    for variable in variables:
        if variable not in d.columns:
            continue
        vt = _variable_type(d[variable])
        if vt is None:
            continue
        if vt == "continuous":
            for group, sub in d.groupby(d[comparison].astype(str)):
                x = pd.to_numeric(sub[variable], errors="coerce")
                rows.append({
                    "comparison": comparison,
                    "group": group,
                    "variable": variable,
                    "label": _label_for_variable(variable),
                    "level": "",
                    "type": vt,
                    "n": int(x.notna().sum()),
                    "summary": f"{x.mean():.2f} ({x.std():.2f})" if x.notna().any() else "",
                })
        else:
            for (group, level), n in d.groupby([d[comparison].astype(str), d[variable].astype(str)], dropna=False).size().items():
                denom = int((d[comparison].astype(str) == group).sum())
                rows.append({
                    "comparison": comparison,
                    "group": group,
                    "variable": variable,
                    "label": _label_for_variable(variable),
                    "level": level,
                    "type": vt,
                    "n": int(n),
                    "summary": f"{int(n)} ({(n / denom * 100) if denom else np.nan:.1f}%)",
                })
    out = pd.DataFrame(rows)
    if output_name:
        out.to_csv(POST_ANALYSIS_DIR / output_name, index=False)
    return out


## Demographic Tables


In [ ]:
# Rmd equivalent: full sample demographic table by phenotype.
dem_table_full_sample = demographic_summary_table(
    data_for_post,
    comparison="phenotype",
    output_name="dem_table_full_sample_by_phenotype.csv",
)
display(dem_table_full_sample.head(30))

# Rmd equivalent: final clustered sample, Discovery vs Test.
dem_table_final_sample = demographic_summary_table(
    data_final_sample,
    comparison="sample",
    output_name="dem_table_final_sample_discovery_vs_test.csv",
)
display(dem_table_final_sample.head(30))


## Label Difference Summaries


In [ ]:
# Rmd equivalent: subgroup demographic/difference summaries for discovery and test samples.
discovery_difference_summary = summarize_group_differences(
    Labels_data_disc,
    comparisons=SUBGROUP_VARS,
    output_prefix="discovery_labels",
)

test_difference_summary = summarize_group_differences(
    Labels_data_test,
    comparisons=SUBGROUP_VARS,
    output_prefix="test_labels",
)

print("Discovery difference rows:", len(discovery_difference_summary))
print("Test difference rows:", len(test_difference_summary))


## Mixed Heatmaps of Subgroup Differences


In [ ]:
def _positive_binary_level(x):
    vals = pd.Series(x).dropna().astype(str).unique().tolist()
    if len(vals) < 2:
        return None
    positives = {"1", "yes", "true", "positive", "present", "high", "case", "chr"}
    for value in vals:
        if value.lower() in positives:
            return value
    return sorted(vals)[1] if len(vals) > 1 else sorted(vals)[0]


def _format_fdr_text(value):
    if pd.isna(value):
        return ""
    if value < 0.001:
        return "<0.001"
    return f"{value:.3f}"


def _stars_from_fdr(value):
    if pd.isna(value):
        return ""
    if value < 0.001:
        return "***"
    if value < 0.01:
        return "**"
    if value < 0.05:
        return "*"
    return ""


def _load_dictionary_variable_map(dictionary_path=DICTIONARY_DIR_DIFF):
    dict_df = pd.read_excel(dictionary_path)
    required = {"ElementName", "Modality"}
    missing = required.difference(dict_df.columns)
    if missing:
        raise KeyError(f"Dictionary is missing required columns for Rmd-style heatmap ordering: {sorted(missing)}")

    aliases_col = "Aliases" if "Aliases" in dict_df.columns else None
    rows = []
    for i, row in dict_df.reset_index(drop=True).iterrows():
        element = str(row["ElementName"]) if pd.notna(row["ElementName"]) else ""
        modality = str(row["Modality"]) if pd.notna(row["Modality"]) else "Other / Unmapped"
        if element:
            rows.append({"key": element, "Modality": modality, "dict_row": i, "priority": 1, "element_short": element})
        if aliases_col:
            aliases = row.get(aliases_col)
            if pd.notna(aliases):
                for alias in re.split(r"[;|,\n\r]+", str(aliases)):
                    alias = alias.strip()
                    if alias:
                        rows.append({"key": alias, "Modality": modality, "dict_row": i, "priority": 2, "element_short": element or alias})
    lookup = pd.DataFrame(rows)
    if lookup.empty:
        return pd.DataFrame(columns=["variable", "Modality", "dict_row", "element_short"])
    lookup = lookup.sort_values(["key", "priority", "dict_row"]).drop_duplicates("key", keep="first")
    return lookup.rename(columns={"key": "variable"})[["variable", "Modality", "dict_row", "element_short"]]


def build_mixed_heatmap_data_rmd_style(df, diff_summary, comparisons=SUBGROUP_VARS):
    """Python analogue of the Rmd's build_mixed_heatmap_data()."""
    if diff_summary.empty:
        return pd.DataFrame()

    rows = []
    use = diff_summary.loc[diff_summary["comparison"].isin(comparisons)].copy()
    # Keep all variables returned by the demographic/difference table, as in the Rmd.
    use = use.sort_values(["comparison", "fdr", "p", "variable"], na_position="last")

    for comp in comparisons:
        comp_rows = use.loc[use["comparison"] == comp]
        if comp not in df.columns or comp_rows.empty:
            print(f"Skipping comparison {comp!r}: not found in data or no difference rows.")
            continue

        df2 = df.loc[df[comp].notna()].copy()
        df2[comp] = df2[comp].astype(str)
        grp_levels = list(pd.unique(df2[comp]))

        for _, r in comp_rows.iterrows():
            variable = r["variable"]
            if variable not in df2.columns:
                continue
            vt = r["var_type"]
            x = df2[variable]
            for k, group in enumerate(grp_levels, start=1):
                sub = df2.loc[df2[comp] == group]
                if vt == "continuous":
                    value = pd.to_numeric(sub[variable], errors="coerce").mean()
                elif vt == "binary":
                    if pd.api.types.is_numeric_dtype(x):
                        value = pd.to_numeric(sub[variable], errors="coerce").mean()
                    else:
                        pos = _positive_binary_level(x)
                        value = (sub[variable].astype(str) == pos).mean() if pos is not None else np.nan
                else:
                    value = np.nan
                rows.append({
                    "comparison": comp,
                    "x": f"{comp} | {group}",
                    "k": k,
                    "variable": variable,
                    "label": r["label"],
                    "var_type": vt,
                    "value": value,
                    "fdr_num": r["fdr"],
                    "fdr_text": _format_fdr_text(r["fdr"]),
                })

    heat = pd.DataFrame(rows)
    if heat.empty:
        return heat

    heat["value_plot_cont"] = np.nan
    heat["value_plot_bin"] = np.nan
    cont = heat["var_type"].eq("continuous")
    if cont.any():
        def _zscore_or_zero(s):
            std = s.std(ddof=0)
            if not np.isfinite(std) or std == 0:
                return pd.Series(0.0, index=s.index)
            return (s - s.mean()) / std
        heat.loc[cont, "value_plot_cont"] = heat.loc[cont].groupby("label")["value"].transform(_zscore_or_zero)
    bin_mask = heat["var_type"].eq("binary")
    heat.loc[bin_mask, "value_plot_bin"] = heat.loc[bin_mask, "value"]
    return heat


def add_modality_spacers_rmd_style(heat_df, spacer_suffix=" "):
    if heat_df.empty:
        return heat_df
    df = heat_df.copy()
    df["modality_x"] = df["x"].astype(str).str.replace(r"\|.*$", "", regex=True).str.strip()
    modalities = list(pd.unique(df["modality_x"]))
    spacer_rows = []
    for modality in modalities[:-1]:
        labels = df.loc[df["modality_x"] == modality, "label"].drop_duplicates()
        for label in labels:
            spacer_rows.append({
                "comparison": np.nan,
                "x": f"{modality}{spacer_suffix}",
                "k": np.nan,
                "variable": np.nan,
                "label": label,
                "var_type": "spacer",
                "value": np.nan,
                "fdr_num": np.nan,
                "fdr_text": "",
                "value_plot_cont": np.nan,
                "value_plot_bin": np.nan,
                "modality_x": modality,
            })
    if spacer_rows:
        df = pd.concat([df, pd.DataFrame(spacer_rows)], ignore_index=True)

    x_levels = []
    for modality in modalities:
        xs = df.loc[(df["modality_x"] == modality) & df["x"].astype(str).str.contains(r"\|", regex=True), "x"].drop_duplicates().tolist()
        x_levels.extend(xs)
        if modality != modalities[-1]:
            x_levels.append(f"{modality}{spacer_suffix}")
    x_pos = {}
    cur = 0.0
    for x in x_levels:
        if "|" in str(x):
            cur += 1.0
        else:
            cur += 0.20
        x_pos[x] = cur
    df["x"] = pd.Categorical(df["x"], categories=x_levels, ordered=True)
    df["x_pos"] = df["x"].astype(str).map(x_pos)
    return df.drop(columns=["modality_x"], errors="ignore")


def _heatmap_variable_map(heat_df, dictionary_path=DICTIONARY_DIR_DIFF):
    lookup = _load_dictionary_variable_map(dictionary_path)
    vars_present = pd.DataFrame({"variable": heat_df["variable"].dropna().astype(str).drop_duplicates()})
    mapped = vars_present.merge(lookup, on="variable", how="left")
    mapped["Modality"] = mapped["Modality"].fillna("Other / Unmapped")
    mapped["dict_row"] = mapped["dict_row"].fillna(np.inf)
    mapped["element_short"] = mapped["element_short"].fillna(mapped["variable"])
    return mapped


def _blocked_panel_data(heat_df, var_type, modality_order=None):
    modality_order = modality_order or ["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition", "Demographics", "study_info"]
    df = heat_df.loc[heat_df["var_type"].eq(var_type)].copy()
    if df.empty:
        return df, [], []

    var_map = _heatmap_variable_map(df)
    df = df.merge(var_map, on="variable", how="left")
    mods_in_panel = var_map["Modality"].dropna().drop_duplicates().tolist()
    mods_present = [m for m in modality_order if m in mods_in_panel] + [m for m in mods_in_panel if m not in modality_order and m != "Other / Unmapped"]
    if "Other / Unmapped" in mods_in_panel:
        mods_present.append("Other / Unmapped")

    y_rows = []
    for modality in mods_present:
        vars_m = (
            var_map.loc[var_map["Modality"] == modality]
            .sort_values(["dict_row", "element_short", "variable"])
        )
        if vars_m.empty:
            continue
        y_rows.append({"y_id": f"##{modality}", "y_label": modality, "is_header": True, "variable": None})
        for _, row in vars_m.iterrows():
            y_rows.append({
                "y_id": f"{modality}__{row['variable']}",
                "y_label": f"  {row['element_short']}",
                "is_header": False,
                "variable": row["variable"],
            })
    y_map = pd.DataFrame(y_rows)
    y_levels = y_map["y_id"].tolist()
    y_pos = {y: len(y_levels) - 1 - i for i, y in enumerate(y_levels)}
    label_map = dict(zip(y_map["y_id"], y_map["y_label"]))
    header_set = set(y_map.loc[y_map["is_header"], "y_id"])

    df["y_id"] = df["Modality"].astype(str) + "__" + df["variable"].astype(str)
    df["y_pos"] = df["y_id"].map(y_pos)
    return df, y_levels, [label_map[y] for y in y_levels], header_set


def plot_mixed_heatmap_split_rmd_style(
    heat_df,
    title="My heatmap ordered by modality",
    filename="heatmap_group_means_ordered.svg",
    width=14,
    height=18,
    x_text_size=11,
    y_text_size=11,
    p_text_size=8,
    categorical_grey="#d9d9d9",
):
    if heat_df.empty:
        print("No heatmap data produced.")
        return None

    x_levels = [x for x in heat_df["x"].cat.categories if "|" in str(x)]
    x_pos_df = heat_df[["x", "x_pos"]].drop_duplicates("x").copy()
    x_pos_df["x_key"] = x_pos_df["x"].astype(str)
    x_pos = x_pos_df.set_index("x_key")["x_pos"].to_dict()
    x_tick_pos = [x_pos[str(x)] for x in x_levels if str(x) in x_pos]
    if not x_tick_pos:
        raise ValueError("No real heatmap x-axis columns were found after adding spacer columns.")

    panels = [
        ("continuous", "Continuous (means)", "value_plot_cont", "Blues", "Mean"),
        ("binary", "Binary (proportions)", "value_plot_bin", "YlOrRd", "Prop"),
        ("categorical", "Categorical", None, None, None),
    ]
    panel_data = []
    heights = []
    for var_type, subtitle, value_col, cmap, legend_title in panels:
        dfp, y_levels, y_labels, header_set = _blocked_panel_data(heat_df, var_type)
        panel_data.append((var_type, subtitle, value_col, cmap, legend_title, dfp, y_levels, y_labels, header_set))
        heights.append(max(1, len(y_levels)))

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(width, height),
        gridspec_kw={"height_ratios": heights},
        sharex=False,
    )
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])

    for ax, (var_type, subtitle, value_col, cmap, legend_title, dfp, y_levels, y_labels, header_set) in zip(axes, panel_data):
        ax.set_title(subtitle, loc="left", fontsize=16)
        ax.set_xlim(min(x_tick_pos) - 0.6, max(x_tick_pos) + 0.6)
        ax.set_ylim(-0.5, max(0, len(y_levels) - 0.5))
        ax.set_yticks(range(len(y_levels)))
        ax.set_yticklabels(list(reversed(y_labels)), fontsize=y_text_size)
        for tick, y_id in zip(ax.get_yticklabels(), reversed(y_levels)):
            if y_id in header_set:
                tick.set_fontweight("bold")
                tick.set_fontsize(13)
        ax.tick_params(axis="y", length=0)
        ax.spines[["top", "right"]].set_visible(False)

        if dfp.empty:
            ax.text(0.5, 0.5, f"No {var_type} variables", ha="center", va="center", transform=ax.transAxes)
            continue

        real = dfp.loc[dfp["y_pos"].notna()].copy()
        if var_type == "categorical":
            for _, row in real.iterrows():
                ax.add_patch(plt.Rectangle((row["x_pos"] - 0.5, row["y_pos"] - 0.45), 1.0, 0.9, facecolor=categorical_grey, edgecolor="white", linewidth=0.3))
        else:
            values = pd.to_numeric(real[value_col], errors="coerce")
            if var_type == "binary":
                norm = plt.Normalize(0, 1)
            else:
                finite = values[np.isfinite(values)]
                vmax = max(abs(finite.min()), abs(finite.max())) if len(finite) else 1
                norm = plt.Normalize(-vmax, vmax)
            cm = plt.get_cmap(cmap)
            for idx, row in real.iterrows():
                val = pd.to_numeric(row[value_col], errors="coerce")
                face = "white" if pd.isna(val) else cm(norm(float(val)))
                ax.add_patch(plt.Rectangle((row["x_pos"] - 0.5, row["y_pos"] - 0.45), 1.0, 0.9, facecolor=face, edgecolor="white", linewidth=0.3))
            sm = plt.cm.ScalarMappable(norm=norm, cmap=cm)
            cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.01)
            cbar.set_label(legend_title, fontsize=12)

        # Rmd-style annotation: stars on first group, FDR text on second group.
        for _, row in real.iterrows():
            label = ""
            if row.get("k") == 1:
                label = _stars_from_fdr(row.get("fdr_num"))
            elif row.get("k") == 2 and str(row.get("fdr_text", "")):
                label = str(row.get("fdr_text"))
            if label:
                ax.text(row["x_pos"], row["y_pos"], label, ha="center", va="center", color="white", fontsize=p_text_size,
                        bbox=dict(boxstyle="round,pad=0.08", facecolor="black", edgecolor="none", alpha=0.65))

    for ax in axes[:-1]:
        ax.set_xticks([])
        ax.tick_params(axis="x", length=0)
    axes[-1].set_xticks(x_tick_pos)
    axes[-1].set_xticklabels(x_levels, rotation=45, ha="right", fontsize=x_text_size)
    axes[-1].set_xlabel("Modality | group", fontsize=14)
    fig.suptitle(title, fontsize=20, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.985])

    out = POST_ANALYSIS_DIR / filename
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out)
    return fig


heat_df = build_mixed_heatmap_data_rmd_style(
    Labels_data_disc,
    discovery_difference_summary,
    comparisons=SUBGROUP_VARS,
)
heat_df_spaced = add_modality_spacers_rmd_style(heat_df)
heat_df_spaced.to_csv(POST_ANALYSIS_DIR / "heatmap_group_means_ordered_data.csv", index=False)

map_fig = plot_mixed_heatmap_split_rmd_style(
    heat_df_spaced,
    title="My heatmap ordered by modality",
    filename="heatmap_group_means_ordered.svg",
    width=14,
    height=18,
    p_text_size=8,
    x_text_size=11,
    y_text_size=11,
)

# Test-sample difference summary is kept as a table in the Rmd; write matching long/compact tables above.


## Site and Recruitment Source


In [ ]:
def site_recruitment_overview_rmd_style(merged_data):
    """Match the Rmd site/recruitment section: merged_data -> sites factor + chrrecruit factor."""
    d = merged_data.copy()
    if "Site" not in d.columns or "chrrecruit" not in d.columns:
        print("Site/recruitment columns are not available in merged_data.")
        return None
    d = d.loc[d["Site"].notna() & d["chrrecruit"].notna()].copy()
    d["sites"] = d["Site"].astype(str)
    d["chrrecruit"] = d["chrrecruit"].astype(str)

    tab = pd.crosstab(d["sites"], d["chrrecruit"])
    stat, p, v = _chi_square_test(tab)
    expected = pd.DataFrame(
        np.outer(tab.sum(axis=1), tab.sum(axis=0)) / tab.to_numpy().sum(),
        index=tab.index,
        columns=tab.columns,
    )
    print("Site by recruitment contingency table")
    display(tab)
    print({"chi_square": stat, "p": p, "cramers_v": v})
    tab.to_csv(POST_ANALYSIS_DIR / "site_by_recruitment_contingency.csv")
    expected.to_csv(POST_ANALYSIS_DIR / "site_by_recruitment_expected_counts.csv")

    plot_df = tab.stack().rename("n").reset_index()
    plot_df["prop"] = plot_df.groupby("sites")["n"].transform(lambda s: s / s.sum())
    plot_df.to_csv(POST_ANALYSIS_DIR / "site_recruitment_plot_data.csv", index=False)

    # Rmd `site_recruitment_dist.pdf`: stacked bars, proportions within site.
    prop = tab.div(tab.sum(axis=1), axis=0).fillna(0)
    fig, ax = plt.subplots(figsize=(20, 20))
    bottom = np.zeros(len(prop))
    for col in prop.columns:
        ax.bar(prop.index.astype(str), prop[col].values, bottom=bottom, label=str(col))
        bottom += prop[col].values
    ax.set_ylim(0, 1)
    ax.set_xlabel("Site")
    ax.set_ylabel("Proportion within site")
    ax.set_title("Recruitment method distribution by site")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(title="Recruitment method", bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    out = POST_ANALYSIS_DIR / "site_recruitment_dist.pdf"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

    # Rmd `site_recruitment.pdf`: tile heatmap of within-site proportions.
    fig, ax = plt.subplots(figsize=(20, 20))
    im = ax.imshow(prop.values, aspect="auto", cmap="viridis", vmin=0, vmax=max(1e-9, np.nanmax(prop.values)))
    ax.set_xticks(range(prop.shape[1]))
    ax.set_xticklabels(prop.columns, rotation=45, ha="right")
    ax.set_yticks(range(prop.shape[0]))
    ax.set_yticklabels(prop.index)
    ax.set_xlabel("Recruitment method")
    ax.set_ylabel("Site")
    ax.set_title("Within-site recruitment proportions")
    fig.colorbar(im, ax=ax, label="Within-site %")
    fig.tight_layout()
    out = POST_ANALYSIS_DIR / "site_recruitment.pdf"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

    # Rmd `site_recruitment2.pdf`: ordered tile heatmap. Use hierarchical ordering if scipy is available.
    row_order = list(prop.index)
    col_order = list(prop.columns)
    try:
        from scipy.cluster.hierarchy import linkage, leaves_list
        from scipy.spatial.distance import pdist
        if prop.shape[0] > 1:
            row_order = prop.index[leaves_list(linkage(pdist(prop.values), method="average"))].tolist()
        if prop.shape[1] > 1:
            col_order = prop.columns[leaves_list(linkage(pdist(prop.values.T), method="average"))].tolist()
    except Exception as exc:
        print("Could not compute clustered site/recruitment ordering; using original order:", exc)
    prop_ord = prop.loc[row_order, col_order]
    fig, ax = plt.subplots(figsize=(20, 20))
    im = ax.imshow(prop_ord.values, aspect="auto", cmap="viridis", vmin=0, vmax=max(1e-9, np.nanmax(prop_ord.values)))
    ax.set_xticks(range(prop_ord.shape[1]))
    ax.set_xticklabels(prop_ord.columns, rotation=45, ha="right")
    ax.set_yticks(range(prop_ord.shape[0]))
    ax.set_yticklabels(prop_ord.index)
    ax.set_xlabel("Recruitment method")
    ax.set_ylabel("Site")
    fig.colorbar(im, ax=ax, label="Within-site %")
    fig.tight_layout()
    out = POST_ANALYSIS_DIR / "site_recruitment2.pdf"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out)
    return tab

site_recruitment_table = site_recruitment_overview_rmd_style(data_for_post)


In [ ]:
def subgroup_factor_long(df, factor_col, subgroup_vars=SUBGROUP_VARS):
    keep_subgroups = [c for c in subgroup_vars if c in df.columns]
    if factor_col not in df.columns or not keep_subgroups:
        return pd.DataFrame()
    long = df[[factor_col] + keep_subgroups].copy()
    long[factor_col] = long[factor_col].astype(str)
    out = long.melt(id_vars=[factor_col], value_vars=keep_subgroups, var_name="subgroup_type", value_name="subgroup_label")
    out = out.dropna(subset=[factor_col, "subgroup_label"])
    out = out.loc[~out[factor_col].isin(["nan", "None", ""])]
    out = out.loc[~out["subgroup_label"].astype(str).isin(["nan", "None", ""])]
    return out


def test_subgroup_by_factor(df, factor_col, prefix, factor_label=None):
    factor_label = factor_label or factor_col
    long = subgroup_factor_long(df, factor_col)
    if long.empty:
        print(f"No data for {factor_col}")
        return pd.DataFrame(), long

    rows = []
    for subgroup_type, sub in long.groupby("subgroup_type", sort=False):
        tab = pd.crosstab(sub[factor_col].astype(str), sub["subgroup_label"].astype(str))
        stat, p, v = _chi_square_test(tab)
        rows.append({
            "subgroup_type": subgroup_type,
            "statistic": stat,
            "p_value": p,
            "cramers_v": v,
            "n": int(tab.to_numpy().sum()),
            "n_factor_levels": tab.shape[0],
            "n_labels": tab.shape[1],
        })
    results = pd.DataFrame(rows)
    results["p_adj"] = _bh_fdr(results["p_value"])
    results = results.sort_values("p_adj", na_position="last")
    results.to_csv(POST_ANALYSIS_DIR / f"{prefix}_subgroup_tests.csv", index=False)
    display(results)

    plot_df = (
        long.groupby(["subgroup_type", factor_col, "subgroup_label"])
        .size()
        .rename("n")
        .reset_index()
    )
    plot_df["prop"] = plot_df.groupby(["subgroup_type", factor_col])["n"].transform(lambda s: s / s.sum())
    plot_df.to_csv(POST_ANALYSIS_DIR / f"{prefix}_subgroup_plot_data.csv", index=False)

    subgroup_order = [s for s in SUBGROUP_VARS if s in plot_df["subgroup_type"].unique()]
    ncols = 2
    nrows = int(np.ceil(len(subgroup_order) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, max(5, 4 * nrows)), squeeze=False)
    for ax, subgroup_type in zip(axes.ravel(), subgroup_order):
        sub = plot_df.loc[plot_df["subgroup_type"] == subgroup_type]
        pivot = sub.pivot_table(index=factor_col, columns="subgroup_label", values="prop", fill_value=0)
        bottom = np.zeros(len(pivot))
        for col in pivot.columns:
            ax.bar(pivot.index.astype(str), pivot[col].values, bottom=bottom, label=str(col))
            bottom += pivot[col].values
        test_row = results.loc[results["subgroup_type"] == subgroup_type]
        if not test_row.empty:
            title = f"{subgroup_type}\nBH p={test_row['p_adj'].iloc[0]:.3g}; V={test_row['cramers_v'].iloc[0]:.2f}"
        else:
            title = subgroup_type
        ax.set_title(title)
        ax.set_ylabel(f"Proportion within {factor_label}")
        ax.tick_params(axis="x", rotation=45)
        ax.legend(title="Subgroup label", fontsize=8)
    for ax in axes.ravel()[len(subgroup_order):]:
        ax.axis("off")
    fig.suptitle(f"Subgroup label distribution by {factor_label}", y=1.01)
    fig.tight_layout()
    out = POST_ANALYSIS_DIR / f"{prefix}_subgroup_distribution.pdf"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out)
    return results, long

site_subgroup_results, site_subgroup_long = test_subgroup_by_factor(
    Labels_data_disc,
    factor_col="Site",
    prefix="discovery_site",
    factor_label="site",
)

recruitment_subgroup_results, recruitment_subgroup_long = test_subgroup_by_factor(
    Labels_data_disc,
    factor_col="chrrecruit",
    prefix="discovery_recruitment",
    factor_label="recruitment source",
)


## Site Plus Recruitment Checks

The Rmd fits multinomial subgroup models adjusted for site and recruitment source. This notebook keeps that analysis optional: it runs if `statsmodels` is available in the active kernel and otherwise still exports the stratified site-by-subgroup tests within recruitment strata.


In [ ]:
def stratified_site_tests_within_recruitment(df):
    required = {"Site", "chrrecruit"}
    if not required.issubset(df.columns):
        print("Site and chrrecruit columns are required for stratified tests.")
        return pd.DataFrame()
    long = df[["Site", "chrrecruit"] + [c for c in SUBGROUP_VARS if c in df.columns]].copy()
    long = long.melt(id_vars=["Site", "chrrecruit"], value_vars=[c for c in SUBGROUP_VARS if c in df.columns], var_name="subgroup_type", value_name="subgroup_label")
    long = long.dropna(subset=["Site", "chrrecruit", "subgroup_label"])
    rows = []
    for (subgroup_type, recruit), sub in long.groupby(["subgroup_type", "chrrecruit"], sort=False):
        tab = pd.crosstab(sub["Site"].astype(str), sub["subgroup_label"].astype(str))
        stat, p, v = _chi_square_test(tab)
        rows.append({
            "subgroup_type": subgroup_type,
            "chrrecruit": recruit,
            "statistic": stat,
            "p_value": p,
            "cramers_v": v,
            "n": int(tab.to_numpy().sum()),
            "n_sites": tab.shape[0],
            "n_labels": tab.shape[1],
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out["p_adj_within_subgroup"] = out.groupby("subgroup_type")["p_value"].transform(_bh_fdr)
        out.to_csv(POST_ANALYSIS_DIR / "site_by_subgroup_stratified_by_recruitment_tests.csv", index=False)
        display(out.sort_values(["subgroup_type", "p_adj_within_subgroup"], na_position="last"))
    return out


def optional_multinomial_site_recruitment_models(df):
    required = {"Site", "chrrecruit"}
    if not required.issubset(df.columns):
        print("Skipping adjusted multinomial site/recruitment models because Site/chrrecruit columns are unavailable.")
        return pd.DataFrame()
    try:
        import statsmodels.api as sm
    except Exception as exc:
        print("Skipping adjusted multinomial site/recruitment models because statsmodels is unavailable:", exc)
        return pd.DataFrame()

    rows = []
    for subgroup_type in [c for c in SUBGROUP_VARS if c in df.columns]:
        d = df[["Site", "chrrecruit", subgroup_type]].dropna().copy()
        if d[subgroup_type].astype(str).nunique() < 2:
            continue
        y = pd.Categorical(d[subgroup_type].astype(str)).codes
        X_full = pd.get_dummies(d[["Site", "chrrecruit"]].astype(str), drop_first=True, dtype=float)
        X_full = sm.add_constant(X_full, has_constant="add")
        try:
            model = sm.MNLogit(y, X_full).fit(method="newton", disp=False, maxiter=200)
            rows.append({
                "subgroup_scheme": subgroup_type,
                "llf": model.llf,
                "aic": model.aic,
                "bic": model.bic,
                "n": int(d.shape[0]),
                "status": "fit",
            })
        except Exception as exc:
            rows.append({
                "subgroup_scheme": subgroup_type,
                "llf": np.nan,
                "aic": np.nan,
                "bic": np.nan,
                "n": int(d.shape[0]),
                "status": f"failed: {exc}",
            })
    out = pd.DataFrame(rows)
    if not out.empty:
        out.to_csv(POST_ANALYSIS_DIR / "site_recruitment_adjusted_multinomial_models.csv", index=False)
        display(out)
    return out

stratified_site_recruitment_tests = stratified_site_tests_within_recruitment(Labels_data_disc)
adjusted_site_recruitment_models = optional_multinomial_site_recruitment_models(Labels_data_disc)


## Included Versus Excluded CHR Participants


In [ ]:
CHR_data = data_for_post.loc[data_for_post["phenotype"].astype(str).eq("CHR")].copy()
included_ids = set(Labels_data_disc["src_subject_id"].astype(str)) | set(Labels_data_test["src_subject_id"].astype(str))
CHR_data["included"] = np.where(CHR_data["src_subject_id"].astype(str).isin(included_ids), "included", "excluded")

included_vs_excluded_table = demographic_summary_table(
    CHR_data,
    comparison="included",
    output_name="dem_table_included_vs_excluded_chr.csv",
)
display(included_vs_excluded_table.head(30))

included_vs_excluded_differences = summarize_group_differences(
    CHR_data,
    comparisons=["included"],
    output_prefix="included_vs_excluded_chr",
)

print("Included/excluded CHR counts:")
display(CHR_data["included"].value_counts())


# Longitudinal analyses across months 1-5

This replaces the old month 2-only comparisons. It fits baseline-cluster mixed models across all available months and maps follow-up observations back onto baseline cluster centroids for discovery and validation samples.


In [ ]:
import os
import pandas as pd
import importlib
import Utils
Utils = importlib.reload(Utils)
run_longitudinal_multiclust_report = Utils.run_longitudinal_multiclust_report
display_longitudinal_multiclust_results = Utils.display_longitudinal_multiclust_results
load_longitudinal_multiclust_results = Utils.load_longitudinal_multiclust_results

_is_singleclust_result = isinstance(final_metrics.get("data"), pd.DataFrame) if isinstance(final_metrics, dict) else False
_longitudinal_data_dirs = [
    'path/to/simpleclust_data'
    if _is_singleclust_result
    else 'path/to/multiclust_data'
]
_subject_id_column = subject_id_column if "subject_id_column" in globals() else "src_subject_id"

_longitudinal_prescient_ids = (
    prescient_ids
    if "prescient_ids" in globals()
    else (set(prescient["src_subject_id"]) if "prescient" in globals() else None)
)

_validation_baseline_data = None
if "dict_final_test" in globals() and isinstance(dict_final_test, dict) and dict_final_test:
    _validation_baseline_data = dict_final_test
elif "dict_final" in globals() and isinstance(dict_final, dict) and dict_final and "labels_test_final" in globals():
    _validation_baseline_data = dict_final

_validation_subject_ids = None
if isinstance(_validation_baseline_data, dict) and _validation_baseline_data:
    _validation_subject_ids = {
        modality: df[_subject_id_column].astype(str).tolist()
        for modality, df in _validation_baseline_data.items()
        if _subject_id_column in df.columns
    }
elif "subject_id_list_test" in globals() and "modalities" in globals():
    _validation_subject_ids = {
        modality: ids
        for modality, ids in zip(modalities, subject_id_list_test)
    }

_validation_domain_labels = None
if "new_test_labels_by_modality" in globals() and new_test_labels_by_modality:
    _validation_domain_labels = new_test_labels_by_modality
elif "labels_test_by_modality" in globals() and labels_test_by_modality:
    _validation_domain_labels = {k: v for k, v in labels_test_by_modality.items() if v is not None}
elif "labels_test_modalities" in globals() and "modalities" in globals():
    _validation_domain_labels = {
        modality: labels
        for modality, labels in zip(modalities, labels_test_modalities)
        if labels is not None
    }

_validation_final_labels = labels_test_final if "labels_test_final" in globals() else None

longitudinal_results = run_longitudinal_multiclust_report(
    final_metrics=final_metrics,
    meta=meta,
    plots_dir=plots_dir,
    data_dirs=_longitudinal_data_dirs,
    prescient_ids=_longitudinal_prescient_ids,
    vars_to_keep=vars_to_keep if "vars_to_keep" in globals() else None,
    categorical_columns=cat_vars if "cat_vars" in globals() else None,
    validation_domain_labels=_validation_domain_labels,
    validation_final_labels=_validation_final_labels,
    validation_subject_ids=_validation_subject_ids,
    validation_baseline_data=_validation_baseline_data,
    months=(1, 2, 3, 4, 5),
    subject_id_column=_subject_id_column,
    min_features_per_analysis=2,
    min_features_for_cluster_change=1,
    min_followup_timepoints_per_feature=1,
    min_nonmissing_per_timepoint=8,
    min_group_n=4,
    reuse_existing=True,
)

print("Longitudinal outputs saved under:", longitudinal_results["output_dir"])
print("Month files used:")
for month, path in longitudinal_results["month_files"].items():
    print(f"  month {month}: {path}")
print("Analyses completed:", len(longitudinal_results["analyses"]))
if longitudinal_results["analysis_summary"].empty:
    print("No mixed-model or cluster-change analyses were completed. Check longitudinal_results['preprocessing_report'] for preprocessing failures.")
else:
    display(longitudinal_results["analysis_summary"])





# Sankey image embedding can time out on cloud-synced output files; saved files remain available in the output directory.
display_longitudinal_multiclust_results(longitudinal_results, max_analyses=20, show_sankey=False)


In [ ]:
_preprocessing_report = longitudinal_results["preprocessing_report"]
_sort_cols = [
    col for col in ["sample", "month", "modality"]
    if col in _preprocessing_report.columns
]
_preprocessing_report_view = (
    _preprocessing_report.sort_values(_sort_cols)
    if _sort_cols else _preprocessing_report
)
display(_preprocessing_report_view.head(30))

if "status" in _preprocessing_report.columns:
    display(
        _preprocessing_report
        .groupby([c for c in ["sample", "status"] if c in _preprocessing_report.columns])
        .size()
        .reset_index(name="n_rows")
    )


In [ ]:
# Display saved longitudinal results without rerunning the analyses.
import importlib
import Utils
importlib.reload(Utils)
from Utils import load_longitudinal_multiclust_results, display_longitudinal_multiclust_results

_existing_longitudinal_results = load_longitudinal_multiclust_results(
    os.path.join(plots_dir, "longitudinal_all_timepoints")
)
display_longitudinal_multiclust_results(
    _existing_longitudinal_results,
    max_analyses=10,
    show_sankey=False,  # Avoid notebook timeouts while embedding saved Sankey/image files. 
)


# Check conversion with domain maps

## Import and merge conversion data

In [ ]:
conv_prescient = pd.read_csv("path/to/restricted_data/conversion/Prescient-Conversion-11-May-2026.csv")
conv_pronet = pd.read_csv("path/to/restricted_data/conversion/ProNETPsychosisRiskO-ConversoinFloatingFo_DATA_2026-04-27_1629 1.csv")

In [ ]:
# Prescient rows are already a list of converted subjects.
prescient_conversion_subjects = set(conv_prescient["subjectkey"])

# ProNET contains conversion-form rows; use consensus_outcome == 1 as confirmed conversion.
# If you want every ProNET conversion-form row instead, use set(conv_pronet["chric_record_id"]).
pronet_conversion_subjects_all_forms = set(conv_pronet["chric_record_id"])
pronet_conversion_subjects_confirmed = set(
    conv_pronet.loc[
        pd.to_numeric(conv_pronet["chrconv_consensus_outcome"], errors="coerce").eq(1),
        "chric_record_id",
    ]
)

print(f"Raw Prescient conversion rows: {len(prescient_conversion_subjects)}")
print(f"Raw ProNET conversion-form rows: {len(pronet_conversion_subjects_all_forms)}")
print(f"Raw ProNET confirmed conversion rows: {len(pronet_conversion_subjects_confirmed)}")


In [ ]:
def _normalize_subject_id(value):
    if pd.isna(value):
        return None
    value = str(value).strip().lower()
    if value in {"", "nan", "none"}:
        return None
    return value


def make_conversion_labels(subject_ids, conversion_subjects):
    conversion_subjects_normalized = {
        sid for sid in (_normalize_subject_id(s) for s in conversion_subjects)
        if sid is not None
    }
    subject_ids_normalized = pd.Series(subject_ids).map(_normalize_subject_id).reset_index(drop=True)
    return np.where(
        subject_ids_normalized.isin(conversion_subjects_normalized),
        "yes",
        "no",
    ).tolist(), conversion_subjects_normalized, set(subject_ids_normalized.dropna())


def plot_conversion_domain_map(sample_name, labels_by_modality, subject_ids, conversion_subjects, save_file_name):
    conversion_labels, conversion_ids, sample_ids = make_conversion_labels(subject_ids, conversion_subjects)
    n_subjects = len(subject_ids)

    if not all(len(labels_by_modality[stage]) == n_subjects for stage in stage_order):
        label_lengths = {stage: len(labels_by_modality[stage]) for stage in stage_order}
        raise ValueError(
            f"{sample_name}: label lengths do not match subject IDs. "
            f"subject_ids={n_subjects}, label_lengths={label_lengths}"
        )

    conversion_counts = pd.Series(
        conversion_labels,
        name="converted_to_psychosis",
    ).value_counts()
    print(f"Conversion labels mapped to {sample_name} subjects:")
    print(conversion_counts)
    print(
        f"{sample_name}: {len(conversion_ids & sample_ids)} of "
        f"{len(conversion_ids)} raw conversion IDs are present in this clustered sample."
    )

    fig, _ = domain_map(
        new_labels_by_modality=labels_by_modality,
        final_labels=conversion_labels,
        stage_order=stage_order,
        final_name="Conversion",
        top_token="high_severity",
        bottom_token="low_severity",
        final_top_value="yes",
        final_bottom_value="no",
        color_for_top_final="#B64242",
        color_for_bottom_final="#B3D9D5",
        add_gap_in_final=True,
        gap_weight=20,
        title=f"Domain mapping by psychosis conversion ({sample_name})",
        plots_dir=plots_dir,
        save_file_name=save_file_name,
    )
    return fig, conversion_labels


subject_ids_discovery_conversion = final_metrics["data"][stage_order[0]]["src_subject_id"].reset_index(drop=True)
fig_conversion_discovery, conversion_labels_discovery = plot_conversion_domain_map(
    sample_name="discovery",
    labels_by_modality=new_labels_by_modality,
    subject_ids=subject_ids_discovery_conversion,
    conversion_subjects=prescient_conversion_subjects,
    save_file_name="Parcats_by_conversion_discovery.pdf",
)

subject_ids_validation_conversion = subject_ids_test if "subject_ids_test" in globals() else dict_final[stage_order[0]]["src_subject_id"].reset_index(drop=True)
fig_conversion_validation, conversion_labels_validation = plot_conversion_domain_map(
    sample_name="validation",
    labels_by_modality=new_test_labels_by_modality,
    subject_ids=subject_ids_validation_conversion,
    conversion_subjects=pronet_conversion_subjects_confirmed,
    save_file_name="Parcats_by_conversion_validation.pdf",
)


## Predict conversion from clinical profile mapping


In [ ]:
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)


def build_profile_predictor_df(labels_by_modality, subject_ids, conversion_labels):
    df = pd.DataFrame({"src_subject_id": pd.Series(subject_ids).astype(str).reset_index(drop=True)})
    for stage in stage_order:
        df[stage] = pd.Series(labels_by_modality[stage]).astype(str).reset_index(drop=True)
    df["clinical_profile"] = df[stage_order].agg(" | ".join, axis=1)
    df["n_high_severity_domains"] = df[stage_order].eq("high_severity").sum(axis=1)
    df["converted_to_psychosis"] = pd.Series(conversion_labels).astype(str).reset_index(drop=True)
    return df


svm_conversion_discovery_df = build_profile_predictor_df(
    labels_by_modality=new_labels_by_modality,
    subject_ids=subject_ids_discovery_conversion,
    conversion_labels=conversion_labels_discovery,
)
svm_conversion_validation_df = build_profile_predictor_df(
    labels_by_modality=new_test_labels_by_modality,
    subject_ids=subject_ids_validation_conversion,
    conversion_labels=conversion_labels_validation,
)

categorical_predictors = stage_order + ["clinical_profile"]
numeric_predictors = ["n_high_severity_domains"]


def encode_profile_predictors(df, reference_columns=None):
    encoded = pd.get_dummies(
        df[categorical_predictors],
        prefix=categorical_predictors,
        dtype=float,
    )
    encoded = pd.concat(
        [encoded.reset_index(drop=True), df[numeric_predictors].astype(float).reset_index(drop=True)],
        axis=1,
    )
    if reference_columns is not None:
        encoded = encoded.reindex(columns=reference_columns, fill_value=0.0)
    return encoded


X_discovery = encode_profile_predictors(svm_conversion_discovery_df)
y_discovery = svm_conversion_discovery_df["converted_to_psychosis"]
X_validation = encode_profile_predictors(
    svm_conversion_validation_df,
    reference_columns=list(X_discovery.columns),
)
y_validation = svm_conversion_validation_df["converted_to_psychosis"]

print("Discovery conversion class counts:")
print(y_discovery.value_counts())
print("\nValidation conversion class counts:")
print(y_validation.value_counts())

if y_discovery.nunique() < 2:
    raise ValueError("Discovery conversion labels contain fewer than two classes; cannot train conversion SVM.")

svm_conversion_results, conversion_svm = SVM_nested_cv(X_discovery, y_discovery)
svm_conversion_cv_summary = pd.DataFrame({
    "mean": svm_conversion_results["mean_metrics"],
    "std": svm_conversion_results["std_metrics"],
})
print("\nClinical-profile discovery nested-CV summary:")
display(svm_conversion_cv_summary)

validation_pred = conversion_svm.predict(X_validation)
yes_class_index = list(conversion_svm.classes_).index("yes")
validation_score = conversion_svm.predict_proba(X_validation)[:, yes_class_index]

svm_conversion_validation_predictions = svm_conversion_validation_df[["src_subject_id"] + stage_order + ["clinical_profile", "converted_to_psychosis"]].copy()
svm_conversion_validation_predictions["predicted_conversion"] = validation_pred
svm_conversion_validation_predictions["p_conversion"] = validation_score

svm_conversion_validation_metrics = {
    "balanced_accuracy": balanced_accuracy_score(y_validation, validation_pred),
    "sensitivity_yes": recall_score(y_validation, validation_pred, pos_label="yes", zero_division=0),
    "precision_yes": precision_score(y_validation, validation_pred, pos_label="yes", zero_division=0),
    "majority_baseline_balanced_accuracy": balanced_accuracy_score(
        y_validation,
        np.repeat(y_discovery.value_counts().idxmax(), len(y_validation)),
    ),
}
if y_validation.nunique() == 2:
    svm_conversion_validation_metrics["roc_auc"] = roc_auc_score((y_validation == "yes").astype(int), validation_score)

svm_conversion_validation_metrics = pd.Series(svm_conversion_validation_metrics, name="validation_metric")
svm_conversion_validation_report = pd.DataFrame(
    classification_report(y_validation, validation_pred, output_dict=True, zero_division=0)
).T
svm_conversion_validation_confusion = pd.DataFrame(
    confusion_matrix(y_validation, validation_pred, labels=["no", "yes"]),
    index=["true_no", "true_yes"],
    columns=["pred_no", "pred_yes"],
)

print("\nValidation metrics:")
display(svm_conversion_validation_metrics.to_frame())
print("\nValidation confusion matrix:")
display(svm_conversion_validation_confusion)
print("\nValidation classification report:")
display(svm_conversion_validation_report)

svm_conversion_outfile = os.path.join(plots_dir, "SVM_conversion_from_profile_validation_predictions.csv")
svm_conversion_validation_predictions.to_csv(svm_conversion_outfile, index=False)
print("Saved validation predictions to:", svm_conversion_outfile)


## Predict conversion from preprocessed clinical variables


In [ ]:
def build_preprocessed_conversion_feature_df(data_by_modality, subject_ids, conversion_labels, sample_name):
    """Merge the saved preprocessed clustering variables into one wide matrix."""
    merged = None
    for modality in stage_order:
        if modality not in data_by_modality:
            raise KeyError(f"{sample_name}: missing modality data for {modality}")

        df_mod = data_by_modality[modality].copy()
        if "src_subject_id" not in df_mod.columns:
            raise KeyError(f"{sample_name} {modality}: src_subject_id column is missing")

        feature_cols = [col for col in df_mod.columns if col != "src_subject_id"]
        tmp = df_mod[["src_subject_id"] + feature_cols].copy()
        tmp["src_subject_id"] = tmp["src_subject_id"].astype(str)
        tmp = tmp.rename(columns={col: f"{modality}__{col}" for col in feature_cols})

        merged = tmp if merged is None else merged.merge(tmp, on="src_subject_id", how="inner")

    label_df = pd.DataFrame({
        "src_subject_id": pd.Series(subject_ids).astype(str).reset_index(drop=True),
        "converted_to_psychosis": pd.Series(conversion_labels).astype(str).reset_index(drop=True),
    })
    out = merged.merge(label_df, on="src_subject_id", how="inner")

    if len(out) != len(label_df):
        missing = sorted(set(label_df["src_subject_id"]) - set(out["src_subject_id"]))[:10]
        print(
            f"WARNING: {sample_name}: retained {len(out)} of {len(label_df)} subjects "
            f"after merging modalities. Missing examples: {missing}"
        )

    return out


svm_preprocessed_discovery_df = build_preprocessed_conversion_feature_df(
    data_by_modality=final_metrics["data"],
    subject_ids=subject_ids_discovery_conversion,
    conversion_labels=conversion_labels_discovery,
    sample_name="discovery",
)
svm_preprocessed_validation_df = build_preprocessed_conversion_feature_df(
    data_by_modality=dict_final,
    subject_ids=subject_ids_validation_conversion,
    conversion_labels=conversion_labels_validation,
    sample_name="validation",
)

preprocessed_feature_cols = [
    col for col in svm_preprocessed_discovery_df.columns
    if col not in {"src_subject_id", "converted_to_psychosis"}
]
missing_validation_cols = sorted(set(preprocessed_feature_cols) - set(svm_preprocessed_validation_df.columns))
extra_validation_cols = sorted(set(svm_preprocessed_validation_df.columns) - set(preprocessed_feature_cols) - {"src_subject_id", "converted_to_psychosis"})
if missing_validation_cols:
    raise ValueError(f"Validation is missing preprocessed feature columns, examples: {missing_validation_cols[:10]}")
if extra_validation_cols:
    print(f"Validation has {len(extra_validation_cols)} extra preprocessed columns not used by discovery training.")

X_preprocessed_discovery = svm_preprocessed_discovery_df[preprocessed_feature_cols]
y_preprocessed_discovery = svm_preprocessed_discovery_df["converted_to_psychosis"]
X_preprocessed_validation = svm_preprocessed_validation_df[preprocessed_feature_cols]
y_preprocessed_validation = svm_preprocessed_validation_df["converted_to_psychosis"]

print("Preprocessed-variable SVM feature matrix shapes:")
print("discovery:", X_preprocessed_discovery.shape)
print("validation:", X_preprocessed_validation.shape)
print("\nDiscovery conversion class counts:")
print(y_preprocessed_discovery.value_counts())
print("\nValidation conversion class counts:")
print(y_preprocessed_validation.value_counts())

if y_preprocessed_discovery.nunique() < 2:
    raise ValueError("Discovery conversion labels contain fewer than two classes; cannot train preprocessed-variable conversion SVM.")

svm_preprocessed_results, preprocessed_conversion_svm = SVM_nested_cv(
    X_preprocessed_discovery,
    y_preprocessed_discovery,
)
svm_preprocessed_conversion_cv_summary = pd.DataFrame({
    "mean": svm_preprocessed_results["mean_metrics"],
    "std": svm_preprocessed_results["std_metrics"],
})
print("\nPreprocessed-variable discovery nested-CV summary:")
display(svm_preprocessed_conversion_cv_summary)

preprocessed_oof = svm_preprocessed_results["oof_uncertainty"].copy()
svm_preprocessed_discovery_converter_metrics = pd.Series(
    {
        "sensitivity_yes": recall_score(
            preprocessed_oof["y_true"],
            preprocessed_oof["y_pred"],
            pos_label="yes",
            zero_division=0,
        ),
        "precision_yes": precision_score(
            preprocessed_oof["y_true"],
            preprocessed_oof["y_pred"],
            pos_label="yes",
            zero_division=0,
        ),
    },
    name="discovery_oof_metric",
)
print("\nPreprocessed-variable discovery converter metrics from out-of-fold predictions:")
display(svm_preprocessed_discovery_converter_metrics.to_frame())

preprocessed_validation_pred = preprocessed_conversion_svm.predict(X_preprocessed_validation)
yes_class_index = list(preprocessed_conversion_svm.classes_).index("yes")
preprocessed_validation_score = preprocessed_conversion_svm.predict_proba(X_preprocessed_validation)[:, yes_class_index]

svm_preprocessed_conversion_validation_predictions = svm_preprocessed_validation_df[["src_subject_id", "converted_to_psychosis"]].copy()
svm_preprocessed_conversion_validation_predictions["predicted_conversion"] = preprocessed_validation_pred
svm_preprocessed_conversion_validation_predictions["p_conversion"] = preprocessed_validation_score

svm_preprocessed_conversion_validation_metrics = {
    "balanced_accuracy": balanced_accuracy_score(y_preprocessed_validation, preprocessed_validation_pred),
    "sensitivity_yes": recall_score(y_preprocessed_validation, preprocessed_validation_pred, pos_label="yes", zero_division=0),
    "precision_yes": precision_score(y_preprocessed_validation, preprocessed_validation_pred, pos_label="yes", zero_division=0),
    "majority_baseline_balanced_accuracy": balanced_accuracy_score(
        y_preprocessed_validation,
        np.repeat(y_preprocessed_discovery.value_counts().idxmax(), len(y_preprocessed_validation)),
    ),
}
if y_preprocessed_validation.nunique() == 2:
    svm_preprocessed_conversion_validation_metrics["roc_auc"] = roc_auc_score(
        (y_preprocessed_validation == "yes").astype(int),
        preprocessed_validation_score,
    )

svm_preprocessed_conversion_validation_metrics = pd.Series(
    svm_preprocessed_conversion_validation_metrics,
    name="validation_metric",
)
svm_preprocessed_conversion_validation_report = pd.DataFrame(
    classification_report(y_preprocessed_validation, preprocessed_validation_pred, output_dict=True, zero_division=0)
).T
svm_preprocessed_conversion_validation_confusion = pd.DataFrame(
    confusion_matrix(y_preprocessed_validation, preprocessed_validation_pred, labels=["no", "yes"]),
    index=["true_no", "true_yes"],
    columns=["pred_no", "pred_yes"],
)

print("\nPreprocessed-variable validation metrics:")
display(svm_preprocessed_conversion_validation_metrics.to_frame())
print("\nPreprocessed-variable validation confusion matrix:")
display(svm_preprocessed_conversion_validation_confusion)
print("\nPreprocessed-variable validation classification report:")
display(svm_preprocessed_conversion_validation_report)

svm_preprocessed_conversion_outfile = os.path.join(plots_dir, "SVM_conversion_from_preprocessed_clinical_variables_validation_predictions.csv")
svm_preprocessed_conversion_validation_predictions.to_csv(svm_preprocessed_conversion_outfile, index=False)
print("Saved preprocessed-variable validation predictions to:", svm_preprocessed_conversion_outfile)


# Save dataframe with subject IDs and labels

In [ ]:
# Save dataframe with subject ids and all labels discovery

## Create dataframe with subjects ids and all labels for discovery
subject_ids_disc = final_metrics['data'][stage_order[0]]['src_subject_id'].astype(str).reset_index(drop=True)
df_disc_labels = pd.DataFrame({"src_subject_id": subject_ids_disc})
for stage in stage_order:
    df_disc_labels[stage] = pd.Series(new_labels_by_modality[stage]).astype(str).reset_index(drop=True)
df_disc_labels["final"] = pd.Series(final_metrics["final_labels"]).astype(str).reset_index(drop=True) 

## Create dataframe with subjects ids and all labels for test
subject_ids_test = dict_final[stage_order[0]]['src_subject_id'].astype(str).reset_index(drop=True)
df_test_labels = pd.DataFrame({"src_subject_id": subject_ids_test})
for stage in stage_order:
    df_test_labels[stage] = pd.Series(new_test_labels_by_modality[stage]).astype(str).reset_index(drop=True)
df_test_labels["final"] = pd.Series(labels_test_final).astype(str).reset_index(drop=True) 

## Save to CSV files
df_disc_labels.to_csv("path/to/results/study_1/release/Labels/discovery_labels_clin_multiclust.csv", index=False)
df_test_labels.to_csv("path/to/results/study_1/release/Labels/test_labels_clin_multiclust.csv", index=False)
